In [ ]:
# ============================================================
# EfficientNetB0 hybrid experiment - environment check
# ============================================================

import sys
import sklearn

print("Python version: ", sys.version)
print("Scikit-Learn version: ", sklearn.__version__)

print("\nRuntime:")
print("CPU-based repeated hybrid experiment")

print("\nIMPORTANT:")
print("Existing EfficientNetB0 features will be reused.")
print("No EfficientNetB0 feature extraction will be performed.")
print("A GPU is not required for the repeated SVM / RF experiment.")

Python version:  3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Scikit-Learn version:  1.6.1

Runtime:
CPU-based repeated hybrid experiment

IMPORTANT:
Existing EfficientNetB0 features will be reused.
No EfficientNetB0 feature extraction will be performed.
A GPU is not required for the repeated SVM / RF experiment.


In [1]:
# ======================================================================
# CONNECTING TO GOOGLE DRIVE
# ======================================================================

from google.colab import drive
drive.mount("/content/drive")


Mounted at /content/drive


In [2]:
from google.colab import files

uploaded = files.upload()

Saving repeated_stratified_5fold_assignments.csv to repeated_stratified_5fold_assignments.csv


In [3]:
# ======================================================================
# RESTORE THE PROJECT DATASET TO THE COLAB RUNTIME
# ======================================================================

import shutil
import tarfile
from pathlib import Path

# Creating the path to the project archive stored in Google Drive
DRIVE_ARCHIVE = Path("/content/drive/MyDrive/brain_tumour_colab/brain_tumour_colab_bundle.tar")

# Create the local path where the archive will temporarily be copied
LOCAL_ARCHIVE = Path("/content/brain_tumour_colab_bundle.tar")


# Create the path where the project will be restored
PROJECT_ROOT = Path("/content/brain-tumour-mri-classification")


print("=" * 70)
print("RESTORE PROJECT DATA")
print("=" * 70)
print("Drive archive:", DRIVE_ARCHIVE)
print("Drive archive exists:", DRIVE_ARCHIVE.exists())
print("Project root:", PROJECT_ROOT)


# Stop if the project archive cannot be found in Google Drive
if not DRIVE_ARCHIVE.exists():
    raise FileNotFoundError(f"Archive not found: {DRIVE_ARCHIVE}")


# Restore the project only if it is not already present in the runtime
if not PROJECT_ROOT.exists():

    print("\nCopying archive to Colab runtime...")
    shutil.copy2(DRIVE_ARCHIVE, LOCAL_ARCHIVE)


    # Create the project directory
    PROJECT_ROOT.mkdir(parents=True, exist_ok=True)


    print("Extracting archive...")

    with tarfile.open(LOCAL_ARCHIVE, "r") as archive:
        archive.extractall(PROJECT_ROOT, filter="data")

else:
    print("\nProject directory already exists. Skipping archive extraction.")


# ============================================================
# VERIFY THE RESTORED PROJECT
# ============================================================

# Create the path to the fixed five-fold cross-validation file
FOLDS_FILE = (PROJECT_ROOT / "splits" / "five_fold_cross_validation.csv")


# Create the path to the cropped Training and Testing dataset
DATA_DIR = (PROJECT_ROOT / "processed_data_cropped")


# Count every PNG image in the cropped dataset
png_image_count = len(list(DATA_DIR.rglob("*.png")))

print("\n--- Verification ---")
print("Project root exists:", PROJECT_ROOT.exists())
print("Fold file exists:", FOLDS_FILE.exists())
print("Dataset folder exists:", DATA_DIR.exists())
print("PNG images found:", png_image_count)


# Stop if the project was not restored correctly
if not FOLDS_FILE.exists():
    raise FileNotFoundError(f"Five-fold cross-validation file not found: {FOLDS_FILE}")


if not DATA_DIR.exists():
    raise FileNotFoundError(f"Cropped dataset folder not found: {DATA_DIR}")


if png_image_count != 7198:
    raise RuntimeError("Expected 7,198 PNG images, "f"but found {png_image_count}.")

print("\nProject restored successfully.")

RESTORE PROJECT DATA
Drive archive: /content/drive/MyDrive/brain_tumour_colab/brain_tumour_colab_bundle.tar
Drive archive exists: True
Project root: /content/brain-tumour-mri-classification

Copying archive to Colab runtime...
Extracting archive...

--- Verification ---
Project root exists: True
Fold file exists: True
Dataset folder exists: True
PNG images found: 7198

Project restored successfully.


In [4]:
# ============================================================
# VALIDATE THE REPEATED 10 x 5 CROSS-VALIDATION ASSIGNMENTS
# ============================================================

from pathlib import Path
import shutil
import pandas as pd


UPLOADED_SPLIT_FILE = Path(
    "/content/repeated_stratified_5fold_assignments.csv"
)

REPEATED_SPLITS_DIR = (
    PROJECT_ROOT
    / "repeated_cv"
    / "splits"
)

REPEATED_SPLITS_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

REPEATED_FOLDS_FILE = (
    REPEATED_SPLITS_DIR
    / "repeated_stratified_5fold_assignments.csv"
)


# Copy the uploaded file into the restored project structure
shutil.copy2(
    UPLOADED_SPLIT_FILE,
    REPEATED_FOLDS_FILE,
)


# Load the assignments
repeated_folds_df = pd.read_csv(
    REPEATED_FOLDS_FILE
)


# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(repeated_folds_df) == 56000

assert repeated_folds_df["repeat"].nunique() == 10

assert set(
    repeated_folds_df["repeat"]
) == set(range(1, 11))

assert set(
    repeated_folds_df["fold"]
) == {1, 2, 3, 4, 5}

assert (
    repeated_folds_df
    .groupby(
        ["repeat", "fold"]
    )
    .size()
    .eq(1120)
    .all()
)

assert (
    repeated_folds_df
    .groupby(
        ["repeat", "fold", "class"]
    )
    .size()
    .eq(280)
    .all()
)

assert (
    repeated_folds_df[
        "relative_path"
    ]
    .str.startswith("Training/")
    .all()
)


print(
    "Repeated split file:",
    REPEATED_FOLDS_FILE,
)

print(
    "Rows:",
    len(repeated_folds_df),
)

print(
    "Repeats:",
    repeated_folds_df[
        "repeat"
    ].nunique(),
)

print(
    "Folds per repeat:",
    repeated_folds_df[
        "fold"
    ].nunique(),
)

print(
    "\nSplit seeds:"
)

print(
    repeated_folds_df[
        ["repeat", "split_seed"]
    ]
    .drop_duplicates()
    .sort_values("repeat")
    .to_string(index=False)
)

print(
    "\nRepeated 10 x 5 split validation PASSED."
)

Repeated split file: /content/brain-tumour-mri-classification/repeated_cv/splits/repeated_stratified_5fold_assignments.csv
Rows: 56000
Repeats: 10
Folds per repeat: 5

Split seeds:
 repeat  split_seed
      1      202601
      2      202602
      3      202603
      4      202604
      5      202605
      6      202606
      7      202607
      8      202608
      9      202609
     10      202610

Repeated 10 x 5 split validation PASSED.


In [5]:
# ============================================================
# Step 5. REPEATED NESTED-CV EFFICIENTNETB0 FIXED-FEATURE
# HYBRID EXPERIMENT SETUP
#
# IMPORTANT:
#   - Uses the EXISTING EfficientNetB0 feature cache created
#     during the original Hybrid experiment.
#   - EfficientNetB0 features are NOT extracted again.
#   - Old cached five-fold assignments will NOT be used.
#   - The shared repeated 10 × 5 assignments loaded earlier
#     will control all new outer folds.
# ============================================================

import gc
import json
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf


# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------

DATA_DIR = (
    PROJECT_ROOT
    / "processed_data_cropped"
)

assert DATA_DIR.exists(), (
    "Dataset directory was not found:\n"
    f"{DATA_DIR}"
)


# ------------------------------------------------------------
# Persistent Google Drive storage
# ------------------------------------------------------------

DRIVE_ROOT = Path(
    "/content/drive/MyDrive/brain_tumour_colab"
)


assert DRIVE_ROOT.exists(), (
    "Google Drive experiment directory was not found:\n"
    f"{DRIVE_ROOT}"
)


# ------------------------------------------------------------
# EXISTING EfficientNetB0 fixed-feature cache
#
# This file was produced by the original Hybrid 2 experiment.
# It contains the already-extracted fixed ImageNet
# EfficientNetB0 features for all 5,600 Training images.
#
# DO NOT create another feature matrix for the repeated run.
# ------------------------------------------------------------

EFFICIENTNETB0_FEATURE_CACHE_PATH = (
    DRIVE_ROOT
    / "results"
    / "efficientnetb0_fixed_features"
    / "training_efficientnetb0_features.npz"
)


assert EFFICIENTNETB0_FEATURE_CACHE_PATH.exists(), (
    "Existing EfficientNetB0 feature cache was not found:\n"
    f"{EFFICIENTNETB0_FEATURE_CACHE_PATH}\n\n"
    "Do not extract new features until this has been checked."
)


# ------------------------------------------------------------
# Persistent repeated-CV result directories
# ------------------------------------------------------------

REPEATED_EFFICIENTNETB0_SVM_DIR = (
    DRIVE_ROOT
    / "results"
    / "repeated_nested_cv"
    / "efficientnetb0_svm"
)

REPEATED_EFFICIENTNETB0_SVM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


REPEATED_EFFICIENTNETB0_RF_DIR = (
    DRIVE_ROOT
    / "results"
    / "repeated_nested_cv"
    / "efficientnetb0_rf"
)

REPEATED_EFFICIENTNETB0_RF_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ------------------------------------------------------------
# Image / feature / class configuration
# ------------------------------------------------------------

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

FEATURE_DIMENSION = 1280


CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary",
]


CLASS_TO_INDEX = {
    class_name: index
    for index, class_name
    in enumerate(CLASS_NAMES)
}


INDEX_TO_CLASS = {
    index: class_name
    for class_name, index
    in CLASS_TO_INDEX.items()
}


# ------------------------------------------------------------
# Repeated nested-CV configuration
# ------------------------------------------------------------

NUMBER_OF_REPEATS = 10

NUMBER_OF_OUTER_FOLDS = 5

NUMBER_OF_INNER_FOLDS = 3


# ------------------------------------------------------------
# Deterministic inner-CV seed
# ------------------------------------------------------------

BASE_INNER_CV_SEED = 303000


def get_hybrid_inner_cv_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_INNER_CV_SEED
        + repeat_number * 100
        + fold_number
    )


# ------------------------------------------------------------
# Random Forest model/search seeds
# ------------------------------------------------------------

BASE_RF_MODEL_SEED = 404000

BASE_RF_SEARCH_SEED = 414000


def get_hybrid_rf_model_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_RF_MODEL_SEED
        + repeat_number * 100
        + fold_number
    )


def get_hybrid_rf_search_seed(
    repeat_number,
    fold_number,
):
    return (
        BASE_RF_SEARCH_SEED
        + repeat_number * 100
        + fold_number
    )


# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

RANDOM_SEED = 42

os.environ[
    "PYTHONHASHSEED"
] = str(
    RANDOM_SEED
)

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

tf.keras.utils.set_random_seed(
    RANDOM_SEED
)


try:

    tf.config.experimental.enable_op_determinism()

    determinism_status = "enabled"

except Exception as error:

    determinism_status = (
        f"requested but unavailable: {error}"
    )


tf.keras.backend.set_floatx(
    "float32"
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert len(
    repeated_folds_df
) == 56000


assert (
    repeated_folds_df[
        "repeat"
    ].nunique()
    == NUMBER_OF_REPEATS
)


assert (
    repeated_folds_df[
        "fold"
    ].nunique()
    == NUMBER_OF_OUTER_FOLDS
)


# ------------------------------------------------------------
# Display configuration
# ------------------------------------------------------------

print("=" * 70)

print(
    "REPEATED NESTED-CV EFFICIENTNETB0 FIXED-FEATURE SETUP"
)

print("=" * 70)


print(
    "Dataset:",
    DATA_DIR,
)


print(
    "\nExisting EfficientNetB0 feature cache:",
    EFFICIENTNETB0_FEATURE_CACHE_PATH,
)


print(
    "Feature cache exists:",
    EFFICIENTNETB0_FEATURE_CACHE_PATH.exists(),
)


print(
    "\nEfficientNetB0 + SVM results:",
    REPEATED_EFFICIENTNETB0_SVM_DIR,
)


print(
    "EfficientNetB0 + RF results:",
    REPEATED_EFFICIENTNETB0_RF_DIR,
)


print(
    "\nRepeats:",
    NUMBER_OF_REPEATS,
)


print(
    "Outer folds per repeat:",
    NUMBER_OF_OUTER_FOLDS,
)


print(
    "Total outer evaluations per classifier:",
    (
        NUMBER_OF_REPEATS
        * NUMBER_OF_OUTER_FOLDS
    ),
)


print(
    "Inner CV folds:",
    NUMBER_OF_INNER_FOLDS,
)


print(
    "\nExpected feature dimension:",
    FEATURE_DIMENSION,
)


print(
    "Feature extraction required:",
    False,
)


print(
    "\nRepeat 1 / Fold 1 inner-CV seed:",
    get_hybrid_inner_cv_seed(
        1,
        1,
    ),
)


print(
    "Repeat 1 / Fold 1 RF model seed:",
    get_hybrid_rf_model_seed(
        1,
        1,
    ),
)


print(
    "Repeat 1 / Fold 1 RF search seed:",
    get_hybrid_rf_search_seed(
        1,
        1,
    ),
)


print(
    "\nIMPORTANT:"
)

print(
    "The previously extracted EfficientNetB0 feature cache "
    "will be reused."
)

print(
    "No EfficientNetB0 feature extraction will be performed."
)

print(
    "Old cached five-fold assignments will be ignored."
)

print(
    "The shared repeated 10 × 5 assignments will be used."
)

print(
    "The Testing partition is not used."
)


print(
    "\nHybrid 2 Step 5 repeated-CV setup PASSED."
)

REPEATED NESTED-CV EFFICIENTNETB0 FIXED-FEATURE SETUP
Dataset: /content/brain-tumour-mri-classification/processed_data_cropped

Existing EfficientNetB0 feature cache: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/training_efficientnetb0_features.npz
Feature cache exists: True

EfficientNetB0 + SVM results: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_svm
EfficientNetB0 + RF results: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_rf

Repeats: 10
Outer folds per repeat: 5
Total outer evaluations per classifier: 50
Inner CV folds: 3

Expected feature dimension: 1280
Feature extraction required: False

Repeat 1 / Fold 1 inner-CV seed: 303101
Repeat 1 / Fold 1 RF model seed: 404101
Repeat 1 / Fold 1 RF search seed: 414101

IMPORTANT:
The previously extracted EfficientNetB0 feature cache will be reused.
No EfficientNetB0 feature extraction will be performed.
Old cached five-fold assi

In [6]:
# ============================================================
# Step 6. LOAD AND VALIDATE THE EXISTING
# EFFICIENTNETB0 FIXED-FEATURE CACHE
#
# IMPORTANT:
#   - Features are loaded from the original Hybrid 2 experiment.
#   - Features are NOT extracted again.
#   - The old cached five-fold assignments are ignored.
#   - Feature rows are aligned to a canonical 5,600-image
#     Training manifest derived from the shared repeated splits.
# ============================================================


# ------------------------------------------------------------
# 1. Load the existing feature archive
# ------------------------------------------------------------

feature_archive = np.load(
    EFFICIENTNETB0_FEATURE_CACHE_PATH,
    allow_pickle=True,
)


print(
    "Feature archive contents:",
    feature_archive.files,
)


# ------------------------------------------------------------
# 2. Required archive contents
# ------------------------------------------------------------

required_cache_arrays = {
    "features",
    "class_indices",
    "relative_paths",
}


missing_cache_arrays = (
    required_cache_arrays
    - set(
        feature_archive.files
    )
)


assert not missing_cache_arrays, (
    "Existing EfficientNetB0 feature cache is missing "
    f"required arrays: {sorted(missing_cache_arrays)}"
)


# ------------------------------------------------------------
# 3. Load the cached arrays
#
# NOTE:
#   fold_assignments may also exist in the archive because
#   it was created during the original five-fold experiment.
#
#   Those old fold assignments are deliberately NOT loaded
#   for use in the repeated experiment.
# ------------------------------------------------------------

cached_efficientnetb0_features = (
    feature_archive[
        "features"
    ]
)

cached_efficientnetb0_class_indices = (
    feature_archive[
        "class_indices"
    ]
    .astype(
        np.int32
    )
)

cached_efficientnetb0_relative_paths = (
    feature_archive[
        "relative_paths"
    ]
    .astype(str)
)


# ------------------------------------------------------------
# 4. Basic feature-cache validation
# ------------------------------------------------------------

assert cached_efficientnetb0_features.shape == (
    5600,
    FEATURE_DIMENSION,
), (
    "Unexpected EfficientNetB0 feature matrix shape: "
    f"{cached_efficientnetb0_features.shape}"
)


assert cached_efficientnetb0_features.dtype == np.float32, (
    "Expected float32 EfficientNetB0 features, but found "
    f"{cached_efficientnetb0_features.dtype}"
)


assert len(
    cached_efficientnetb0_class_indices
) == 5600


assert len(
    cached_efficientnetb0_relative_paths
) == 5600


assert (
    len(
        np.unique(
            cached_efficientnetb0_relative_paths
        )
    )
    == 5600
), (
    "Duplicate image paths were found "
    "in the cached EfficientNetB0 features."
)


assert np.isfinite(
    cached_efficientnetb0_features
).all(), (
    "Non-finite values were found "
    "in the cached EfficientNetB0 feature matrix."
)


assert set(
    np.unique(
        cached_efficientnetb0_class_indices
    )
) == {
    0,
    1,
    2,
    3,
}, (
    "Unexpected class indices were found "
    "in the feature cache."
)


# ------------------------------------------------------------
# 5. Create the canonical Training manifest
#
# Every repeat contains the same 5,600 Training images.
# Repeat 1 is used only to obtain one canonical copy.
#
# Fold membership is NOT taken from Repeat 1 here.
# ------------------------------------------------------------

canonical_training_df = (
    repeated_folds_df[
        repeated_folds_df[
            "repeat"
        ]
        == 1
    ][
        [
            "relative_path",
            "class",
            "label",
        ]
    ]
    .copy()
    .sort_values(
        "relative_path"
    )
    .reset_index(
        drop=True
    )
)


assert len(
    canonical_training_df
) == 5600


assert (
    canonical_training_df[
        "relative_path"
    ]
    .nunique()
    == 5600
)


# ------------------------------------------------------------
# 6. Verify the cache contains exactly the same
#    5,600 Training images
# ------------------------------------------------------------

cached_path_set = set(
    cached_efficientnetb0_relative_paths.tolist()
)


repeated_training_path_set = set(
    canonical_training_df[
        "relative_path"
    ]
    .astype(str)
    .tolist()
)


assert (
    cached_path_set
    == repeated_training_path_set
), (
    "The existing EfficientNetB0 feature cache and the "
    "shared repeated-CV Training manifest do not "
    "contain exactly the same 5,600 images."
)


# ------------------------------------------------------------
# 7. Align cached feature rows to the canonical
#    Training manifest order
# ------------------------------------------------------------

path_to_cached_index = {
    path: index
    for index, path
    in enumerate(
        cached_efficientnetb0_relative_paths
    )
}


ordered_cache_indices = np.array(
    [
        path_to_cached_index[
            path
        ]
        for path
        in canonical_training_df[
            "relative_path"
        ].astype(str)
    ],
    dtype=np.int64,
)


training_features = (
    cached_efficientnetb0_features[
        ordered_cache_indices
    ]
)


training_class_indices = (
    cached_efficientnetb0_class_indices[
        ordered_cache_indices
    ]
)


training_relative_paths = (
    cached_efficientnetb0_relative_paths[
        ordered_cache_indices
    ]
)


# ------------------------------------------------------------
# 8. Validate labels after path-based alignment
# ------------------------------------------------------------

expected_class_indices = (
    canonical_training_df[
        "label"
    ]
    .to_numpy(
        dtype=np.int32
    )
)


assert np.array_equal(
    training_class_indices,
    expected_class_indices,
), (
    "Cached EfficientNetB0 class indices do not match "
    "the repeated-CV Training manifest after "
    "path-based alignment."
)


assert np.array_equal(
    training_relative_paths,
    canonical_training_df[
        "relative_path"
    ]
    .astype(str)
    .to_numpy(),
), (
    "Cached EfficientNetB0 feature rows are not aligned "
    "with the canonical Training manifest."
)


# ------------------------------------------------------------
# 9. Final feature-matrix validation
# ------------------------------------------------------------

assert training_features.shape == (
    5600,
    1280,
)


assert training_features.dtype == np.float32


assert len(
    training_class_indices
) == 5600


assert len(
    training_relative_paths
) == 5600


assert np.isfinite(
    training_features
).all()


# ------------------------------------------------------------
# 10. Class-distribution validation
# ------------------------------------------------------------

class_counts = (
    pd.Series(
        training_class_indices
    )
    .value_counts()
    .sort_index()
)


assert (
    class_counts
    == 1400
).all(), (
    "Expected exactly 1,400 Training images "
    "from each class."
)


# ------------------------------------------------------------
# 11. Display
# ------------------------------------------------------------

print("=" * 70)

print(
    "EXISTING EFFICIENTNETB0 FEATURE CACHE VALIDATION"
)

print("=" * 70)


print(
    "Loaded from:",
    EFFICIENTNETB0_FEATURE_CACHE_PATH,
)


print(
    "\nFeature matrix:",
    training_features.shape,
)


print(
    "Feature dtype:",
    training_features.dtype,
)


print(
    "Class indices:",
    training_class_indices.shape,
)


print(
    "Unique Training paths:",
    len(
        np.unique(
            training_relative_paths
        )
    ),
)


print(
    "All features finite:",
    np.isfinite(
        training_features
    ).all(),
)


print(
    "\nClass counts:"
)

for class_index, count in (
    class_counts.items()
):

    print(
        f"{INDEX_TO_CLASS[class_index]:12s}:",
        count,
    )


print(
    "\nIMPORTANT:"
)

print(
    "Existing EfficientNetB0 features were LOADED."
)

print(
    "No EfficientNetB0 feature extraction was performed."
)

print(
    "Old cached five-fold assignments were NOT used."
)

print(
    "Feature rows were aligned using image paths."
)

print(
    "Shared repeated 10 × 5 assignments remain authoritative."
)

print(
    "Testing images were not used."
)


print(
    "\nHybrid 2 Step 6 feature-cache validation PASSED."
)

Feature archive contents: ['features', 'class_indices', 'fold_assignments', 'relative_paths']
EXISTING EFFICIENTNETB0 FEATURE CACHE VALIDATION
Loaded from: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/training_efficientnetb0_features.npz

Feature matrix: (5600, 1280)
Feature dtype: float32
Class indices: (5600,)
Unique Training paths: 5600
All features finite: True

Class counts:
glioma      : 1400
meningioma  : 1400
notumor     : 1400
pituitary   : 1400

IMPORTANT:
Existing EfficientNetB0 features were LOADED.
No EfficientNetB0 feature extraction was performed.
Old cached five-fold assignments were NOT used.
Feature rows were aligned using image paths.
Shared repeated 10 × 5 assignments remain authoritative.
Testing images were not used.

Hybrid 2 Step 6 feature-cache validation PASSED.


In [7]:
# ============================================================
# Step 7. BUILD AND VALIDATE THE REPEATED OUTER PARTITIONS
# FOR EFFICIENTNETB0 FIXED FEATURES
#
# IMPORTANT:
#   - Uses the shared repeated 10 × 5 assignment file.
#   - Uses the already-loaded 5,600 × 1,280 feature matrix.
#   - Does NOT use the old cached five-fold assignments.
#   - Does NOT use the Testing partition.
# ============================================================


# ------------------------------------------------------------
# 1. Map every Training image path to its canonical
#    feature-matrix row
# ------------------------------------------------------------

path_to_feature_index = {
    path: index
    for index, path
    in enumerate(
        training_relative_paths
    )
}


assert len(
    path_to_feature_index
) == 5600


# ------------------------------------------------------------
# 2. Create repeated Hybrid 2 assignment table
# ------------------------------------------------------------

hybrid_repeated_assignments = (
    repeated_folds_df
    .copy()
)


hybrid_repeated_assignments[
    "feature_index"
] = (
    hybrid_repeated_assignments[
        "relative_path"
    ]
    .astype(str)
    .map(
        path_to_feature_index
    )
)


# ------------------------------------------------------------
# 3. Validate feature-row mapping
# ------------------------------------------------------------

assert (
    hybrid_repeated_assignments[
        "feature_index"
    ]
    .notna()
    .all()
), (
    "At least one repeated-CV image could not "
    "be matched to the cached EfficientNetB0 features."
)


hybrid_repeated_assignments[
    "feature_index"
] = (
    hybrid_repeated_assignments[
        "feature_index"
    ]
    .astype(
        np.int64
    )
)


assert len(
    hybrid_repeated_assignments
) == 56000


# ------------------------------------------------------------
# 4. Define outer-partition function
# ------------------------------------------------------------

def make_hybrid_outer_partition(
    repeat_number,
    fold_number,
):
    """
    Return one repeated outer-training /
    outer-validation partition using the fixed
    EfficientNetB0 feature matrix.
    """

    assert (
        1
        <= repeat_number
        <= NUMBER_OF_REPEATS
    )

    assert (
        1
        <= fold_number
        <= NUMBER_OF_OUTER_FOLDS
    )


    # --------------------------------------------------------
    # Select one repeat
    # --------------------------------------------------------

    repeat_df = (
        hybrid_repeated_assignments[
            hybrid_repeated_assignments[
                "repeat"
            ]
            == repeat_number
        ]
        .copy()
    )


    assert len(
        repeat_df
    ) == 5600


    # --------------------------------------------------------
    # Identify outer training / validation rows
    # --------------------------------------------------------

    outer_training_df = (
        repeat_df[
            repeat_df[
                "fold"
            ]
            != fold_number
        ]
        .copy()
    )


    outer_validation_df = (
        repeat_df[
            repeat_df[
                "fold"
            ]
            == fold_number
        ]
        .copy()
    )


    # --------------------------------------------------------
    # Feature-matrix indices
    # --------------------------------------------------------

    outer_training_indices = (
        outer_training_df[
            "feature_index"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    outer_validation_indices = (
        outer_validation_df[
            "feature_index"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    # --------------------------------------------------------
    # Features
    # --------------------------------------------------------

    outer_training_features = (
        training_features[
            outer_training_indices
        ]
    )


    outer_validation_features = (
        training_features[
            outer_validation_indices
        ]
    )


    # --------------------------------------------------------
    # Labels
    # --------------------------------------------------------

    outer_training_labels = (
        training_class_indices[
            outer_training_indices
        ]
    )


    outer_validation_labels = (
        training_class_indices[
            outer_validation_indices
        ]
    )


    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------

    outer_training_paths = (
        training_relative_paths[
            outer_training_indices
        ]
    )


    outer_validation_paths = (
        training_relative_paths[
            outer_validation_indices
        ]
    )


    # --------------------------------------------------------
    # Validate sizes
    # --------------------------------------------------------

    assert outer_training_features.shape == (
        4480,
        FEATURE_DIMENSION,
    )


    assert outer_validation_features.shape == (
        1120,
        FEATURE_DIMENSION,
    )


    assert len(
        outer_training_labels
    ) == 4480


    assert len(
        outer_validation_labels
    ) == 1120


    # --------------------------------------------------------
    # Validate no overlap
    # --------------------------------------------------------

    assert set(
        outer_training_paths
    ).isdisjoint(
        set(
            outer_validation_paths
        )
    )


    assert (
        len(
            set(
                outer_training_paths
            )
        )
        == 4480
    )


    assert (
        len(
            set(
                outer_validation_paths
            )
        )
        == 1120
    )


    # --------------------------------------------------------
    # Validate class balance
    # --------------------------------------------------------

    outer_training_class_counts = (
        pd.Series(
            outer_training_labels
        )
        .value_counts()
        .sort_index()
    )


    outer_validation_class_counts = (
        pd.Series(
            outer_validation_labels
        )
        .value_counts()
        .sort_index()
    )


    assert (
        outer_training_class_counts
        == 1120
    ).all()


    assert (
        outer_validation_class_counts
        == 280
    ).all()


    # --------------------------------------------------------
    # Return partition
    # --------------------------------------------------------

    return {
        "repeat":
            repeat_number,

        "fold":
            fold_number,

        "split_seed":
            int(
                outer_validation_df[
                    "split_seed"
                ].iloc[0]
            ),

        "outer_training_features":
            outer_training_features,

        "outer_training_labels":
            outer_training_labels,

        "outer_training_paths":
            outer_training_paths,

        "outer_validation_features":
            outer_validation_features,

        "outer_validation_labels":
            outer_validation_labels,

        "outer_validation_paths":
            outer_validation_paths,
    }


# ------------------------------------------------------------
# 5. Test Repeat 1 / Fold 1
# ------------------------------------------------------------

hybrid_partition_test = (
    make_hybrid_outer_partition(
        repeat_number=1,
        fold_number=1,
    )
)


# ------------------------------------------------------------
# 6. Display validation
# ------------------------------------------------------------

print("=" * 70)

print(
    "EFFICIENTNETB0 FIXED-FEATURE REPEATED OUTER PARTITION"
)

print("=" * 70)


print(
    "Repeat:",
    hybrid_partition_test[
        "repeat"
    ],
)


print(
    "Fold:",
    hybrid_partition_test[
        "fold"
    ],
)


print(
    "Split seed:",
    hybrid_partition_test[
        "split_seed"
    ],
)


print(
    "\nOuter Training features:",
    hybrid_partition_test[
        "outer_training_features"
    ].shape,
)


print(
    "Outer validation features:",
    hybrid_partition_test[
        "outer_validation_features"
    ].shape,
)


print(
    "\nOuter Training class counts:"
)

print(
    pd.Series(
        hybrid_partition_test[
            "outer_training_labels"
        ]
    )
    .value_counts()
    .sort_index()
)


print(
    "\nOuter validation class counts:"
)

print(
    pd.Series(
        hybrid_partition_test[
            "outer_validation_labels"
        ]
    )
    .value_counts()
    .sort_index()
)


print(
    "\nIMPORTANT:"
)

print(
    "Old cached five-fold assignments were NOT used."
)

print(
    "Shared repeated 10 × 5 assignments were used."
)

print(
    "Testing images were not used."
)


print(
    "\nHybrid 2 Step 7 repeated outer-partition "
    "validation PASSED."
)

EFFICIENTNETB0 FIXED-FEATURE REPEATED OUTER PARTITION
Repeat: 1
Fold: 1
Split seed: 202601

Outer Training features: (4480, 1280)
Outer validation features: (1120, 1280)

Outer Training class counts:
0    1120
1    1120
2    1120
3    1120
Name: count, dtype: int64

Outer validation class counts:
0    280
1    280
2    280
3    280
Name: count, dtype: int64

IMPORTANT:
Old cached five-fold assignments were NOT used.
Shared repeated 10 × 5 assignments were used.
Testing images were not used.

Hybrid 2 Step 7 repeated outer-partition validation PASSED.


In [8]:
# ============================================================
# Step 8. DEFINE THE REPEATED NESTED-CV
# EFFICIENTNETB0 FIXED-FEATURE + SVM SEARCH
#
# IMPORTANT:
#   - StandardScaler remains INSIDE the sklearn pipeline.
#   - Hyperparameter selection uses only the 4,480
#     outer-training feature vectors.
#   - Every outer fold receives its own deterministic
#     shuffled 3-fold inner CV.
#   - Outer-validation data is not used for selection.
# ============================================================

from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    StandardScaler,
)

from sklearn.svm import SVC


# ------------------------------------------------------------
# 1. SVM hyperparameter grid
#
# Same grid used for:
#   - Classical HOG-GLCM + SVM
#   - ResNet50 fixed features + SVM
#   - EfficientNetB0 fixed features + SVM
#
# Total = 25 candidate configurations
# ------------------------------------------------------------

SVM_PARAMETER_GRID = {
    "classifier__C": [
        0.01,
        0.1,
        1,
        10,
        100,
    ],

    "classifier__gamma": [
        "scale",
        1e-5,
        1e-4,
        1e-3,
        1e-2,
    ],
}


# ------------------------------------------------------------
# 2. Base EfficientNetB0 + SVM pipeline
#
# StandardScaler MUST remain inside the pipeline so that
# during inner CV it is fitted only on each inner-training
# subset.
# ------------------------------------------------------------

def build_hybrid_svm_pipeline():

    return Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(),
            ),

            (
                "classifier",
                SVC(
                    kernel="rbf",
                ),
            ),
        ]
    )


# ------------------------------------------------------------
# 3. Build one fold-specific inner search
# ------------------------------------------------------------

def build_hybrid_svm_search(
    repeat_number,
    fold_number,
):

    inner_cv_seed = (
        get_hybrid_inner_cv_seed(
            repeat_number,
            fold_number,
        )
    )


    inner_cross_validation = (
        StratifiedKFold(
            n_splits=NUMBER_OF_INNER_FOLDS,
            shuffle=True,
            random_state=inner_cv_seed,
        )
    )


    svm_search = GridSearchCV(
        estimator=build_hybrid_svm_pipeline(),

        param_grid=SVM_PARAMETER_GRID,

        scoring="f1_macro",

        cv=inner_cross_validation,

        n_jobs=-1,

        refit=True,

        return_train_score=False,
    )


    return svm_search


# ------------------------------------------------------------
# 4. Validate the search definition
# ------------------------------------------------------------

number_of_svm_configurations = (
    len(
        SVM_PARAMETER_GRID[
            "classifier__C"
        ]
    )
    *
    len(
        SVM_PARAMETER_GRID[
            "classifier__gamma"
        ]
    )
)


assert (
    number_of_svm_configurations
    == 25
)


test_svm_search = (
    build_hybrid_svm_search(
        repeat_number=1,
        fold_number=1,
    )
)


assert (
    test_svm_search.scoring
    == "f1_macro"
)


assert (
    test_svm_search.cv.n_splits
    == 3
)


assert (
    test_svm_search.cv.random_state
    == 303101
)


assert (
    test_svm_search.refit
    is True
)


# ------------------------------------------------------------
# 5. Display
# ------------------------------------------------------------

print("=" * 70)

print(
    "EFFICIENTNETB0 FIXED-FEATURE + SVM SEARCH"
)

print("=" * 70)


print(
    "SVM kernel:",
    "RBF",
)


print(
    "Feature standardisation:",
    "StandardScaler inside Pipeline",
)


print(
    "C values:",
    SVM_PARAMETER_GRID[
        "classifier__C"
    ],
)


print(
    "Gamma values:",
    SVM_PARAMETER_GRID[
        "classifier__gamma"
    ],
)


print(
    "Candidate configurations:",
    number_of_svm_configurations,
)


print(
    "Inner CV folds:",
    NUMBER_OF_INNER_FOLDS,
)


print(
    "Selection metric:",
    test_svm_search.scoring,
)


print(
    "\nRepeat 1 / Fold 1 inner-CV seed:",
    test_svm_search.cv.random_state,
)


print(
    "\nIMPORTANT:"
)

print(
    "A fresh SVM search will be created "
    "for every outer fold."
)

print(
    "Only outer-training features will be used "
    "for hyperparameter selection."
)

print(
    "Outer-validation features remain untouched "
    "until final fold evaluation."
)

print(
    "Testing images are not used."
)


print(
    "\nHybrid 2 Step 8 SVM search definition PASSED."
)

EFFICIENTNETB0 FIXED-FEATURE + SVM SEARCH
SVM kernel: RBF
Feature standardisation: StandardScaler inside Pipeline
C values: [0.01, 0.1, 1, 10, 100]
Gamma values: ['scale', 1e-05, 0.0001, 0.001, 0.01]
Candidate configurations: 25
Inner CV folds: 3
Selection metric: f1_macro

Repeat 1 / Fold 1 inner-CV seed: 303101

IMPORTANT:
A fresh SVM search will be created for every outer fold.
Only outer-training features will be used for hyperparameter selection.
Outer-validation features remain untouched until final fold evaluation.
Testing images are not used.

Hybrid 2 Step 8 SVM search definition PASSED.


In [9]:
# ============================================================
# Step 9. EFFICIENTNETB0 FIXED-FEATURE + SVM
# PERSISTENCE AND RESUME UTILITIES
#
# IMPORTANT:
#   - Every completed outer fold will be saved to Google Drive.
#   - Completed folds can be detected after a disconnect.
#   - Existing completed folds will later be skipped.
#   - Files are written atomically to reduce the risk of
#     partially written results.
# ============================================================

import json
import os
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Convert NumPy objects to ordinary Python objects
#    before writing JSON
# ------------------------------------------------------------

def make_json_serializable(value):

    if isinstance(
        value,
        np.integer,
    ):
        return int(
            value
        )

    if isinstance(
        value,
        np.floating,
    ):
        return float(
            value
        )

    if isinstance(
        value,
        np.ndarray,
    ):
        return value.tolist()

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    raise TypeError(
        "Object of type "
        f"{type(value).__name__} "
        "is not JSON serializable."
    )


# ------------------------------------------------------------
# 2. Atomic JSON writer
# ------------------------------------------------------------

def save_json_atomic(
    data,
    output_path,
):

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = (
        output_path.with_suffix(
            output_path.suffix
            + ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            data,
            file,
            indent=2,
            default=make_json_serializable,
        )


    os.replace(
        temporary_path,
        output_path,
    )


# ------------------------------------------------------------
# 3. Atomic CSV writer
# ------------------------------------------------------------

def save_dataframe_atomic(
    dataframe,
    output_path,
):

    output_path = Path(
        output_path
    )

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = (
        output_path.with_suffix(
            output_path.suffix
            + ".tmp"
        )
    )


    dataframe.to_csv(
        temporary_path,
        index=False,
    )


    os.replace(
        temporary_path,
        output_path,
    )


# ------------------------------------------------------------
# 4. Create the directory for one repeat / fold
# ------------------------------------------------------------

def get_efficientnetb0_svm_fold_directory(
    repeat_number,
    fold_number,
):

    fold_directory = (
        REPEATED_EFFICIENTNETB0_SVM_DIR
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
    )


    return fold_directory


# ------------------------------------------------------------
# 5. Define the files required for a completed fold
# ------------------------------------------------------------

def get_efficientnetb0_svm_fold_paths(
    repeat_number,
    fold_number,
):

    fold_directory = (
        get_efficientnetb0_svm_fold_directory(
            repeat_number,
            fold_number,
        )
    )


    return {

        "directory":
            fold_directory,

        "search_results":
            (
                fold_directory
                / "inner_search_results.csv"
            ),

        "selected_parameters":
            (
                fold_directory
                / "selected_parameters.json"
            ),

        "outer_predictions":
            (
                fold_directory
                / "outer_predictions.csv"
            ),

        "outer_metrics":
            (
                fold_directory
                / "outer_metrics.json"
            ),

        "completion_marker":
            (
                fold_directory
                / "COMPLETED.json"
            ),
    }


# ------------------------------------------------------------
# 6. Determine whether one fold is already complete
#
# The completion marker is written LAST.
# ------------------------------------------------------------

def is_efficientnetb0_svm_fold_complete(
    repeat_number,
    fold_number,
):

    fold_paths = (
        get_efficientnetb0_svm_fold_paths(
            repeat_number,
            fold_number,
        )
    )


    required_files = [

        fold_paths[
            "search_results"
        ],

        fold_paths[
            "selected_parameters"
        ],

        fold_paths[
            "outer_predictions"
        ],

        fold_paths[
            "outer_metrics"
        ],

        fold_paths[
            "completion_marker"
        ],
    ]


    return all(
        file_path.exists()
        for file_path
        in required_files
    )


# ------------------------------------------------------------
# 7. Validate directory structure using Repeat 1 / Fold 1
#
# This does NOT train anything.
# ------------------------------------------------------------

test_svm_fold_paths = (
    get_efficientnetb0_svm_fold_paths(
        repeat_number=1,
        fold_number=1,
    )
)


assert (
    test_svm_fold_paths[
        "directory"
    ]
    ==
    (
        REPEATED_EFFICIENTNETB0_SVM_DIR
        / "repeat_01"
        / "fold_01"
    )
)


# ------------------------------------------------------------
# 8. Display
# ------------------------------------------------------------

print("=" * 70)

print(
    "EFFICIENTNETB0 FIXED-FEATURE + SVM "
    "PERSISTENCE / RESUME SETUP"
)

print("=" * 70)


print(
    "Results root:",
    REPEATED_EFFICIENTNETB0_SVM_DIR,
)


print(
    "\nExample fold directory:"
)

print(
    test_svm_fold_paths[
        "directory"
    ]
)


print(
    "\nFiles saved for every completed fold:"
)

print(
    " - inner_search_results.csv"
)

print(
    " - selected_parameters.json"
)

print(
    " - outer_predictions.csv"
)

print(
    " - outer_metrics.json"
)

print(
    " - COMPLETED.json"
)


print(
    "\nRepeat 1 / Fold 1 currently complete:",
    is_efficientnetb0_svm_fold_complete(
        1,
        1,
    ),
)


print(
    "\nIMPORTANT:"
)

print(
    "The completion marker will be written only "
    "after all fold outputs are saved."
)

print(
    "After a Colab disconnect, completed folds "
    "can therefore be skipped safely."
)


print(
    "\nHybrid 2 Step 9 SVM persistence / resume "
    "setup PASSED."
)

EFFICIENTNETB0 FIXED-FEATURE + SVM PERSISTENCE / RESUME SETUP
Results root: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_svm

Example fold directory:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_svm/repeat_01/fold_01

Files saved for every completed fold:
 - inner_search_results.csv
 - selected_parameters.json
 - outer_predictions.csv
 - outer_metrics.json
 - COMPLETED.json

Repeat 1 / Fold 1 currently complete: True

IMPORTANT:
The completion marker will be written only after all fold outputs are saved.
After a Colab disconnect, completed folds can therefore be skipped safely.

Hybrid 2 Step 9 SVM persistence / resume setup PASSED.


In [ ]:
# ============================================================
# HYBRID2 STEP 9-11. RUN ALL 10 × 5 REPEATED NESTED-CV
# EFFICIENTNETB0 FIXED-FEATURE + SVM FOLDS
#
# Total:
#   10 repeats × 5 outer folds = 50 outer evaluations
#
# Resume behaviour:
#   - completed + valid fold -> SKIP
#   - incomplete / corrupt fold -> DELETE and RERUN
#   - save immediately after every completed fold
#
# IMPORTANT:
#   - EfficientNetB0 features are NOT extracted again.
#   - The existing 5,600 × 1,280 feature cache is reused.
#   - OLD cached fold_assignments are ignored.
#   - Hyperparameter selection occurs only inside the
#     4,480-sample outer-training partition.
#   - The 1,120 outer-validation samples are evaluated once.
#   - Testing data is NEVER used.
#
# Expected SVM search:
#   StandardScaler -> RBF SVC
#   C = [0.01, 0.1, 1, 10, 100]
#   gamma = ["scale", 1e-5, 1e-4, 1e-3, 1e-2]
#   25 configurations
#   3-fold stratified inner CV
#   scoring = "f1_macro"
#
# Persistence:
#   Each outer fold saves:
#       inner_search_results.csv
#       selected_parameters.json
#       outer_predictions.csv
#       outer_metrics.json
#       COMPLETED.json
#
#   COMPLETED.json IS WRITTEN LAST.
# ============================================================

import gc
import json
import os
import shutil
import tempfile
import time

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


# ============================================================
# 1. EXPLICIT EXPERIMENT CONSTANTS
# ============================================================

NUMBER_OF_REPEATS = 10

NUMBER_OF_OUTER_FOLDS = 5

NUMBER_OF_INNER_FOLDS = 3

FEATURE_DIMENSION = 1280

EXPECTED_OUTER_TRAINING_SAMPLES = 4480

EXPECTED_OUTER_VALIDATION_SAMPLES = 1120

EXPECTED_SVM_CANDIDATES = 25


CLASS_NAMES = [
    "glioma",
    "meningioma",
    "notumor",
    "pituitary",
]


INDEX_TO_CLASS = {
    0: "glioma",
    1: "meningioma",
    2: "notumor",
    3: "pituitary",
}


REPEATED_EFFICIENTNETB0_SVM_DIR = Path(
    "/content/drive/MyDrive/brain_tumour_colab/"
    "results/repeated_nested_cv/efficientnetb0_svm"
)


REPEATED_EFFICIENTNETB0_SVM_DIR.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. REQUIRED PREVIOUS-STEP VALIDATION
#
# This cell deliberately uses the already-created:
#
#   make_hybrid_outer_partition()
#   get_hybrid_inner_cv_seed()
#   build_hybrid_svm_search()
#
# from Hybrid2 Steps 7 and 8.
#
# If Colab reconnects, rerun the setup/cache/partition/search-
# definition cells first, then rerun THIS SAME CELL.
# ============================================================

assert (
    "make_hybrid_outer_partition"
    in globals()
), (
    "make_hybrid_outer_partition() is not defined. "
    "Rerun HYBRID2 Step 7 first."
)


assert (
    "get_hybrid_inner_cv_seed"
    in globals()
), (
    "get_hybrid_inner_cv_seed() is not defined. "
    "Rerun the Hybrid2 setup/search-definition cells first."
)


assert (
    "build_hybrid_svm_search"
    in globals()
), (
    "build_hybrid_svm_search() is not defined. "
    "Rerun HYBRID2 Step 8 first."
)


assert (
    get_hybrid_inner_cv_seed(
        1,
        1,
    )
    == 303101
), (
    "Unexpected R1/F1 inner-CV seed. "
    "Expected 303101."
)


print(
    "Required Hybrid2 functions: PASSED"
)

print(
    "R1/F1 expected inner-CV seed:",
    get_hybrid_inner_cv_seed(
        1,
        1,
    ),
)


# ============================================================
# 3. PERSISTENT FOLD PATHS
# ============================================================

def get_efficientnetb0_svm_fold_paths(
    repeat_number,
    fold_number,
):

    repeat_directory = (
        REPEATED_EFFICIENTNETB0_SVM_DIR
        / f"repeat_{repeat_number:02d}"
    )


    fold_directory = (
        repeat_directory
        / f"fold_{fold_number:02d}"
    )


    return {

        "directory":
            fold_directory,

        "search_results":
            fold_directory
            / "inner_search_results.csv",

        "selected_parameters":
            fold_directory
            / "selected_parameters.json",

        "outer_predictions":
            fold_directory
            / "outer_predictions.csv",

        "outer_metrics":
            fold_directory
            / "outer_metrics.json",

        "completion_marker":
            fold_directory
            / "COMPLETED.json",
    }


# ============================================================
# 4. ATOMIC DATAFRAME SAVE
#
# Write to a temporary file in the same directory first.
# The final filename appears only after the write succeeds.
# ============================================================

def save_dataframe_atomic(
    dataframe,
    destination_path,
):

    destination_path = Path(
        destination_path
    )


    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = (
        destination_path.parent
        / (
            destination_path.name
            + ".tmp"
        )
    )


    dataframe.to_csv(
        temporary_path,
        index=False,
    )


    os.replace(
        temporary_path,
        destination_path,
    )


    assert (
        destination_path.exists()
    ), (
        "Atomic dataframe save failed: "
        f"{destination_path}"
    )


# ============================================================
# 5. ATOMIC JSON SAVE
# ============================================================

def save_json_atomic(
    dictionary,
    destination_path,
):

    destination_path = Path(
        destination_path
    )


    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = (
        destination_path.parent
        / (
            destination_path.name
            + ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            dictionary,
            file,
            indent=4,
            ensure_ascii=False,
        )


        file.flush()

        os.fsync(
            file.fileno()
        )


    os.replace(
        temporary_path,
        destination_path,
    )


    assert (
        destination_path.exists()
    ), (
        "Atomic JSON save failed: "
        f"{destination_path}"
    )


# ============================================================
# 6. VALIDATE AN ALREADY-COMPLETED FOLD
#
# A fold is skipped only if:
#   - every required file exists
#   - completion marker says completed
#   - repeat/fold identifiers are correct
#   - 25 SVM search configurations were saved
#   - 1,120 outer predictions were saved
#   - metrics contain the correct sample counts
#   - prediction labels are valid
# ============================================================

def validate_completed_efficientnetb0_svm_fold(
    repeat_number,
    fold_number,
):

    fold_paths = (
        get_efficientnetb0_svm_fold_paths(
            repeat_number,
            fold_number,
        )
    )


    required_paths = [
        fold_paths[
            "search_results"
        ],
        fold_paths[
            "selected_parameters"
        ],
        fold_paths[
            "outer_predictions"
        ],
        fold_paths[
            "outer_metrics"
        ],
        fold_paths[
            "completion_marker"
        ],
    ]


    if not all(
        path.exists()
        for path
        in required_paths
    ):

        return False


    try:

        # ----------------------------------------------------
        # Completion marker
        # ----------------------------------------------------

        with open(
            fold_paths[
                "completion_marker"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            completion = (
                json.load(
                    file
                )
            )


        if (
            completion.get(
                "status"
            )
            != "completed"
        ):

            return False


        if (
            completion.get(
                "model"
            )
            != "EfficientNetB0_fixed_features_SVM"
        ):

            return False


        if (
            int(
                completion.get(
                    "repeat"
                )
            )
            != repeat_number
        ):

            return False


        if (
            int(
                completion.get(
                    "fold"
                )
            )
            != fold_number
        ):

            return False


        # ----------------------------------------------------
        # Outer metrics
        # ----------------------------------------------------

        with open(
            fold_paths[
                "outer_metrics"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            metrics = (
                json.load(
                    file
                )
            )


        if (
            metrics.get(
                "model"
            )
            != "EfficientNetB0_fixed_features_SVM"
        ):

            return False


        if (
            int(
                metrics[
                    "repeat"
                ]
            )
            != repeat_number
        ):

            return False


        if (
            int(
                metrics[
                    "fold"
                ]
            )
            != fold_number
        ):

            return False


        if (
            int(
                metrics[
                    "outer_training_samples"
                ]
            )
            != EXPECTED_OUTER_TRAINING_SAMPLES
        ):

            return False


        if (
            int(
                metrics[
                    "outer_validation_samples"
                ]
            )
            != EXPECTED_OUTER_VALIDATION_SAMPLES
        ):

            return False


        if not np.isfinite(
            float(
                metrics[
                    "macro_f1"
                ]
            )
        ):

            return False


        # ----------------------------------------------------
        # Saved predictions
        # ----------------------------------------------------

        saved_predictions = (
            pd.read_csv(
                fold_paths[
                    "outer_predictions"
                ]
            )
        )


        if (
            len(
                saved_predictions
            )
            != EXPECTED_OUTER_VALIDATION_SAMPLES
        ):

            return False


        required_prediction_columns = {
            "relative_path",
            "true_label",
            "predicted_label",
            "true_class",
            "predicted_class",
        }


        if not (
            required_prediction_columns
            .issubset(
                saved_predictions.columns
            )
        ):

            return False


        if (
            saved_predictions[
                "relative_path"
            ]
            .nunique()
            != EXPECTED_OUTER_VALIDATION_SAMPLES
        ):

            return False


        if not set(
            saved_predictions[
                "true_label"
            ]
            .astype(
                int
            )
            .unique()
        ).issubset(
            {
                0,
                1,
                2,
                3,
            }
        ):

            return False


        if not set(
            saved_predictions[
                "predicted_label"
            ]
            .astype(
                int
            )
            .unique()
        ).issubset(
            {
                0,
                1,
                2,
                3,
            }
        ):

            return False


        # ----------------------------------------------------
        # Saved inner search
        # ----------------------------------------------------

        saved_search_results = (
            pd.read_csv(
                fold_paths[
                    "search_results"
                ]
            )
        )


        if (
            len(
                saved_search_results
            )
            != EXPECTED_SVM_CANDIDATES
        ):

            return False


        # ----------------------------------------------------
        # Selected parameters
        # ----------------------------------------------------

        with open(
            fold_paths[
                "selected_parameters"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            selected_parameters = (
                json.load(
                    file
                )
            )


        if (
            selected_parameters.get(
                "model"
            )
            != "EfficientNetB0_fixed_features_SVM"
        ):

            return False


        if (
            int(
                selected_parameters[
                    "repeat"
                ]
            )
            != repeat_number
        ):

            return False


        if (
            int(
                selected_parameters[
                    "fold"
                ]
            )
            != fold_number
        ):

            return False


        if (
            int(
                selected_parameters[
                    "number_of_candidates"
                ]
            )
            != EXPECTED_SVM_CANDIDATES
        ):

            return False


        return True


    except Exception:

        return False


# ============================================================
# 7. PRE-RUN DIAGNOSTIC
#
# Show anything already saved on Drive BEFORE starting.
# ============================================================

existing_valid_folds = []


for repeat_number in range(
    1,
    NUMBER_OF_REPEATS + 1,
):

    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    ):

        if (
            validate_completed_efficientnetb0_svm_fold(
                repeat_number,
                fold_number,
            )
        ):

            existing_valid_folds.append(
                (
                    repeat_number,
                    fold_number,
                )
            )


print(
    "\n"
    + "=" * 72
)

print(
    "EFFICIENTNETB0 FIXED-FEATURE + SVM"
)

print(
    "PERSISTENT RESUME CHECK"
)

print(
    "=" * 72
)


print(
    "Results root:"
)

print(
    REPEATED_EFFICIENTNETB0_SVM_DIR
)


print(
    "\nValid folds already saved:",
    len(
        existing_valid_folds
    ),
    "/ 50",
)


if (
    len(
        existing_valid_folds
    )
    > 0
):

    print(
        "Already-completed folds:"
    )

    print(
        existing_valid_folds
    )


print(
    "\nPersistence/resume setup: PASSED"
)


# ============================================================
# 8. RUN ALL 10 REPEATS × 5 OUTER FOLDS
# ============================================================

completed_folds = 0

skipped_folds = 0

rerun_folds = 0


full_run_start_time = (
    time.perf_counter()
)


for repeat_number in range(
    1,
    NUMBER_OF_REPEATS + 1,
):

    print(
        "\n"
        + "=" * 72
    )

    print(
        f"REPEAT {repeat_number} / "
        f"{NUMBER_OF_REPEATS}"
    )

    print(
        "=" * 72
    )


    for fold_number in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    ):

        print(
            "\n"
            + "-" * 72
        )

        print(
            f"Repeat {repeat_number} / "
            f"Fold {fold_number}"
        )

        print(
            "-" * 72
        )


        # ----------------------------------------------------
        # Persistent fold paths
        # ----------------------------------------------------

        fold_paths = (
            get_efficientnetb0_svm_fold_paths(
                repeat_number,
                fold_number,
            )
        )


        fold_directory = (
            fold_paths[
                "directory"
            ]
        )


        # ----------------------------------------------------
        # Skip already-completed valid folds
        # ----------------------------------------------------

        if (
            validate_completed_efficientnetb0_svm_fold(
                repeat_number,
                fold_number,
            )
        ):

            print(
                "Status: already completed "
                "and validated."
            )

            print(
                "Action: SKIPPING."
            )


            skipped_folds += 1

            continue


        # ----------------------------------------------------
        # Delete an incomplete/corrupt previous attempt
        #
        # IMPORTANT:
        # A runtime interruption BEFORE COMPLETED.json is
        # written leaves an incomplete fold directory.
        #
        # On rerun, only that incomplete fold is deleted.
        # All previously VERIFIED folds remain untouched.
        # ----------------------------------------------------

        if fold_directory.exists():

            print(
                "Existing fold directory is incomplete "
                "or invalid."
            )

            print(
                "Deleting incomplete fold before rerun..."
            )


            shutil.rmtree(
                fold_directory
            )


            rerun_folds += 1


        fold_directory.mkdir(
            parents=True,
            exist_ok=True,
        )


        # ----------------------------------------------------
        # Obtain correct SHARED repeated outer partition
        #
        # This comes from:
        #
        # repeated_stratified_5fold_assignments.csv
        #
        # NOT from the obsolete fold_assignments stored
        # inside the feature archive.
        # ----------------------------------------------------

        partition = (
            make_hybrid_outer_partition(
                repeat_number=
                    repeat_number,

                fold_number=
                    fold_number,
            )
        )


        X_outer_train = (
            partition[
                "outer_training_features"
            ]
        )


        y_outer_train = (
            partition[
                "outer_training_labels"
            ]
        )


        X_outer_validation = (
            partition[
                "outer_validation_features"
            ]
        )


        y_outer_validation = (
            partition[
                "outer_validation_labels"
            ]
        )


        outer_validation_paths = (
            partition[
                "outer_validation_paths"
            ]
        )


        # ----------------------------------------------------
        # Outer partition assertions
        # ----------------------------------------------------

        assert X_outer_train.shape == (
            EXPECTED_OUTER_TRAINING_SAMPLES,
            FEATURE_DIMENSION,
        )


        assert X_outer_validation.shape == (
            EXPECTED_OUTER_VALIDATION_SAMPLES,
            FEATURE_DIMENSION,
        )


        assert (
            len(
                y_outer_train
            )
            == EXPECTED_OUTER_TRAINING_SAMPLES
        )


        assert (
            len(
                y_outer_validation
            )
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        assert (
            len(
                outer_validation_paths
            )
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        assert (
            len(
                np.unique(
                    outer_validation_paths
                )
            )
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        train_class_counts = (
            np.bincount(
                np.asarray(
                    y_outer_train,
                    dtype=int,
                ),
                minlength=4,
            )
        )


        validation_class_counts = (
            np.bincount(
                np.asarray(
                    y_outer_validation,
                    dtype=int,
                ),
                minlength=4,
            )
        )


        assert np.array_equal(
            train_class_counts,
            np.array(
                [
                    1120,
                    1120,
                    1120,
                    1120,
                ]
            ),
        )


        assert np.array_equal(
            validation_class_counts,
            np.array(
                [
                    280,
                    280,
                    280,
                    280,
                ]
            ),
        )


        # ----------------------------------------------------
        # Fold-specific deterministic inner-CV seed
        # ----------------------------------------------------

        inner_cv_seed = (
            get_hybrid_inner_cv_seed(
                repeat_number,
                fold_number,
            )
        )


        expected_inner_cv_seed = (
            303000
            + repeat_number * 100
            + fold_number
        )


        assert (
            inner_cv_seed
            == expected_inner_cv_seed
        ), (
            "Unexpected inner-CV seed. "
            f"Expected {expected_inner_cv_seed}, "
            f"received {inner_cv_seed}."
        )


        # ----------------------------------------------------
        # Fresh SVM hyperparameter search
        # ----------------------------------------------------

        svm_search = (
            build_hybrid_svm_search(
                repeat_number,
                fold_number,
            )
        )


        # ----------------------------------------------------
        # Diagnostics BEFORE expensive search
        # ----------------------------------------------------

        print(
            "Split seed:",
            partition[
                "split_seed"
            ],
        )


        print(
            "Inner-CV seed:",
            inner_cv_seed,
        )


        print(
            "Outer Training:",
            X_outer_train.shape,
        )


        print(
            "Outer validation:",
            X_outer_validation.shape,
        )


        print(
            "Training class counts:",
            train_class_counts.tolist(),
        )


        print(
            "Validation class counts:",
            validation_class_counts.tolist(),
        )


        print(
            "SVM candidates:",
            EXPECTED_SVM_CANDIDATES,
        )


        print(
            "\nStarting inner 3-fold "
            "hyperparameter search..."
        )


        fold_start_time = (
            time.perf_counter()
        )


        # ----------------------------------------------------
        # NESTED MODEL SELECTION
        #
        # ONLY outer-training data enters GridSearchCV.
        #
        # StandardScaler is INSIDE the Pipeline, so scaling
        # is fitted independently within each inner-CV split.
        #
        # GridSearchCV(refit=True) then automatically refits
        # the winning StandardScaler + RBF SVM pipeline using
        # all 4,480 outer-training samples.
        #
        # Outer-validation data has NOT been supplied here.
        # ----------------------------------------------------

        svm_search.fit(
            X_outer_train,
            y_outer_train,
        )


        search_elapsed_seconds = (
            time.perf_counter()
            - fold_start_time
        )


        # ----------------------------------------------------
        # Validate completed inner search
        # ----------------------------------------------------

        assert hasattr(
            svm_search,
            "best_estimator_",
        )


        assert hasattr(
            svm_search,
            "best_params_",
        )


        assert hasattr(
            svm_search,
            "best_score_",
        )


        assert (
            len(
                svm_search.cv_results_[
                    "params"
                ]
            )
            == EXPECTED_SVM_CANDIDATES
        )


        best_inner_macro_f1 = float(
            svm_search.best_score_
        )


        assert np.isfinite(
            best_inner_macro_f1
        )


        # ----------------------------------------------------
        # OUTER EVALUATION
        #
        # The held-out 1,120-image OUTER validation fold
        # is evaluated ONCE after model selection.
        #
        # Testing is not involved.
        # ----------------------------------------------------

        outer_predictions = (
            svm_search.predict(
                X_outer_validation
            )
        )


        outer_predictions = np.asarray(
            outer_predictions,
            dtype=int,
        )


        assert (
            len(
                outer_predictions
            )
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        assert set(
            np.unique(
                outer_predictions
            )
        ).issubset(
            {
                0,
                1,
                2,
                3,
            }
        )


        # ----------------------------------------------------
        # Metrics
        # ----------------------------------------------------

        outer_accuracy = float(
            accuracy_score(
                y_outer_validation,
                outer_predictions,
            )
        )


        outer_balanced_accuracy = float(
            balanced_accuracy_score(
                y_outer_validation,
                outer_predictions,
            )
        )


        outer_macro_f1 = float(
            f1_score(
                y_outer_validation,
                outer_predictions,
                average="macro",
            )
        )


        outer_macro_precision = float(
            precision_score(
                y_outer_validation,
                outer_predictions,
                average="macro",
                zero_division=0,
            )
        )


        outer_macro_recall = float(
            recall_score(
                y_outer_validation,
                outer_predictions,
                average="macro",
                zero_division=0,
            )
        )


        outer_confusion_matrix = (
            confusion_matrix(
                y_outer_validation,
                outer_predictions,
                labels=[
                    0,
                    1,
                    2,
                    3,
                ],
            )
        )


        outer_classification_report = (
            classification_report(
                y_outer_validation,
                outer_predictions,
                labels=[
                    0,
                    1,
                    2,
                    3,
                ],
                target_names=
                    CLASS_NAMES,
                output_dict=True,
                zero_division=0,
            )
        )


        # ----------------------------------------------------
        # Save complete inner-search results
        # ----------------------------------------------------

        search_results_df = (
            pd.DataFrame(
                svm_search.cv_results_
            )
        )


        assert (
            len(
                search_results_df
            )
            == EXPECTED_SVM_CANDIDATES
        )


        # ----------------------------------------------------
        # Outer-validation prediction table
        # ----------------------------------------------------

        predictions_df = pd.DataFrame(
            {
                "relative_path":
                    outer_validation_paths,

                "true_label":
                    np.asarray(
                        y_outer_validation,
                        dtype=int,
                    ),

                "predicted_label":
                    outer_predictions,
            }
        )


        predictions_df[
            "true_class"
        ] = (
            predictions_df[
                "true_label"
            ]
            .map(
                INDEX_TO_CLASS
            )
        )


        predictions_df[
            "predicted_class"
        ] = (
            predictions_df[
                "predicted_label"
            ]
            .map(
                INDEX_TO_CLASS
            )
        )


        assert (
            len(
                predictions_df
            )
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        assert (
            predictions_df[
                "relative_path"
            ]
            .nunique()
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        assert (
            predictions_df[
                "true_class"
            ]
            .notna()
            .all()
        )


        assert (
            predictions_df[
                "predicted_class"
            ]
            .notna()
            .all()
        )


        # ----------------------------------------------------
        # Selected hyperparameter record
        # ----------------------------------------------------

        selected_parameters = {

            "model":
                "EfficientNetB0_fixed_features_SVM",

            "feature_dimension":
                int(
                    FEATURE_DIMENSION
                ),

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "outer_split_seed":
                int(
                    partition[
                        "split_seed"
                    ]
                ),

            "inner_cv_seed":
                int(
                    inner_cv_seed
                ),

            "inner_cv_folds":
                int(
                    NUMBER_OF_INNER_FOLDS
                ),

            "selection_metric":
                "f1_macro",

            "number_of_candidates":
                int(
                    EXPECTED_SVM_CANDIDATES
                ),

            "best_inner_macro_f1":
                float(
                    best_inner_macro_f1
                ),

            "best_parameters":
                svm_search.best_params_,
        }


        # ----------------------------------------------------
        # Outer-fold metric record
        # ----------------------------------------------------

        outer_metrics = {

            "model":
                "EfficientNetB0_fixed_features_SVM",

            "feature_dimension":
                int(
                    FEATURE_DIMENSION
                ),

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "outer_split_seed":
                int(
                    partition[
                        "split_seed"
                    ]
                ),

            "inner_cv_seed":
                int(
                    inner_cv_seed
                ),

            "outer_training_samples":
                int(
                    EXPECTED_OUTER_TRAINING_SAMPLES
                ),

            "outer_validation_samples":
                int(
                    EXPECTED_OUTER_VALIDATION_SAMPLES
                ),

            "best_inner_macro_f1":
                float(
                    best_inner_macro_f1
                ),

            "accuracy":
                float(
                    outer_accuracy
                ),

            "balanced_accuracy":
                float(
                    outer_balanced_accuracy
                ),

            "macro_f1":
                float(
                    outer_macro_f1
                ),

            "macro_precision":
                float(
                    outer_macro_precision
                ),

            "macro_recall":
                float(
                    outer_macro_recall
                ),

            "confusion_matrix":
                outer_confusion_matrix.tolist(),

            "classification_report":
                outer_classification_report,

            "elapsed_seconds":
                float(
                    search_elapsed_seconds
                ),
        }


        # ----------------------------------------------------
        # SAVE SUBSTANTIVE FILES FIRST
        #
        # If the runtime dies during any save below,
        # COMPLETED.json will not exist and this fold will be
        # automatically rerun next time.
        # ----------------------------------------------------

        save_dataframe_atomic(
            search_results_df,
            fold_paths[
                "search_results"
            ],
        )


        print(
            "Saved:",
            fold_paths[
                "search_results"
            ],
        )


        save_json_atomic(
            selected_parameters,
            fold_paths[
                "selected_parameters"
            ],
        )


        print(
            "Saved:",
            fold_paths[
                "selected_parameters"
            ],
        )


        save_dataframe_atomic(
            predictions_df,
            fold_paths[
                "outer_predictions"
            ],
        )


        print(
            "Saved:",
            fold_paths[
                "outer_predictions"
            ],
        )


        save_json_atomic(
            outer_metrics,
            fold_paths[
                "outer_metrics"
            ],
        )


        print(
            "Saved:",
            fold_paths[
                "outer_metrics"
            ],
        )


        # ----------------------------------------------------
        # READ SAVED OUTPUTS BACK BEFORE COMPLETION
        # ----------------------------------------------------

        saved_search_results = (
            pd.read_csv(
                fold_paths[
                    "search_results"
                ]
            )
        )


        saved_predictions = (
            pd.read_csv(
                fold_paths[
                    "outer_predictions"
                ]
            )
        )


        with open(
            fold_paths[
                "selected_parameters"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            saved_selected_parameters = (
                json.load(
                    file
                )
            )


        with open(
            fold_paths[
                "outer_metrics"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            saved_metrics = (
                json.load(
                    file
                )
            )


        assert (
            len(
                saved_search_results
            )
            == EXPECTED_SVM_CANDIDATES
        )


        assert (
            len(
                saved_predictions
            )
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        assert (
            saved_predictions[
                "relative_path"
            ]
            .nunique()
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        assert (
            int(
                saved_selected_parameters[
                    "repeat"
                ]
            )
            == repeat_number
        )


        assert (
            int(
                saved_selected_parameters[
                    "fold"
                ]
            )
            == fold_number
        )


        assert (
            int(
                saved_metrics[
                    "repeat"
                ]
            )
            == repeat_number
        )


        assert (
            int(
                saved_metrics[
                    "fold"
                ]
            )
            == fold_number
        )


        assert (
            int(
                saved_metrics[
                    "outer_training_samples"
                ]
            )
            == EXPECTED_OUTER_TRAINING_SAMPLES
        )


        assert (
            int(
                saved_metrics[
                    "outer_validation_samples"
                ]
            )
            == EXPECTED_OUTER_VALIDATION_SAMPLES
        )


        assert np.isclose(
            float(
                saved_metrics[
                    "macro_f1"
                ]
            ),
            outer_macro_f1,
            rtol=0.0,
            atol=1e-12,
        )


        # ----------------------------------------------------
        # COMPLETION MARKER MUST BE WRITTEN LAST
        # ----------------------------------------------------

        completion_information = {

            "status":
                "completed",

            "model":
                "EfficientNetB0_fixed_features_SVM",

            "repeat":
                int(
                    repeat_number
                ),

            "fold":
                int(
                    fold_number
                ),

            "outer_split_seed":
                int(
                    partition[
                        "split_seed"
                    ]
                ),

            "inner_cv_seed":
                int(
                    inner_cv_seed
                ),

            "required_outputs_verified":
                True,
        }


        save_json_atomic(
            completion_information,
            fold_paths[
                "completion_marker"
            ],
        )


        # ----------------------------------------------------
        # FINAL SAVED-FOLD VALIDATION
        # ----------------------------------------------------

        assert (
            validate_completed_efficientnetb0_svm_fold(
                repeat_number,
                fold_number,
            )
        ), (
            "Saved fold failed final "
            "completion validation."
        )


        completed_folds += 1


        # ----------------------------------------------------
        # Fold status
        # ----------------------------------------------------

        print(
            "\nBest parameters:",
            svm_search.best_params_,
        )


        print(
            "Best inner macro F1:",
            round(
                best_inner_macro_f1,
                6,
            ),
        )


        print(
            "Outer accuracy:",
            round(
                outer_accuracy,
                6,
            ),
        )


        print(
            "Outer balanced accuracy:",
            round(
                outer_balanced_accuracy,
                6,
            ),
        )


        print(
            "Outer macro F1:",
            round(
                outer_macro_f1,
                6,
            ),
        )


        print(
            "Elapsed seconds:",
            round(
                search_elapsed_seconds,
                2,
            ),
        )


        print(
            "Status: SAVED AND VERIFIED."
        )


        print(
            "Completion marker:",
            fold_paths[
                "completion_marker"
            ],
        )


        # ----------------------------------------------------
        # Release fold-specific objects
        # ----------------------------------------------------

        del svm_search
        del partition

        del X_outer_train
        del y_outer_train

        del X_outer_validation
        del y_outer_validation

        del outer_predictions

        del search_results_df
        del predictions_df

        gc.collect()


    # --------------------------------------------------------
    # Repeat completion status
    # --------------------------------------------------------

    completed_in_repeat = sum(

        validate_completed_efficientnetb0_svm_fold(
            repeat_number,
            fold_number,
        )

        for fold_number
        in range(
            1,
            NUMBER_OF_OUTER_FOLDS + 1,
        )
    )


    print(
        "\n"
        f"Repeat {repeat_number} status: "
        f"{completed_in_repeat}/5 folds complete."
    )


# ============================================================
# 9. FINAL EXPERIMENT VALIDATION
# ============================================================

total_valid_completed_folds = sum(

    validate_completed_efficientnetb0_svm_fold(
        repeat_number,
        fold_number,
    )

    for repeat_number
    in range(
        1,
        NUMBER_OF_REPEATS + 1,
    )

    for fold_number
    in range(
        1,
        NUMBER_OF_OUTER_FOLDS + 1,
    )
)


full_run_elapsed_seconds = (
    time.perf_counter()
    - full_run_start_time
)


# ============================================================
# 10. FINAL STATUS
# ============================================================

print(
    "\n"
    + "=" * 72
)

print(
    "EFFICIENTNETB0 FIXED-FEATURE + SVM "
    "REPEATED NESTED-CV RUN FINISHED"
)

print(
    "=" * 72
)


print(
    "Valid completed folds:",
    total_valid_completed_folds,
    "/ 50",
)


print(
    "New folds completed this run:",
    completed_folds,
)


print(
    "Previously completed folds skipped:",
    skipped_folds,
)


print(
    "Incomplete/corrupt folds rerun:",
    rerun_folds,
)


print(
    "Total runtime this session (seconds):",
    round(
        full_run_elapsed_seconds,
        2,
    ),
)


print(
    "\nResults root:"
)

print(
    REPEATED_EFFICIENTNETB0_SVM_DIR
)


if (
    total_valid_completed_folds
    == 50
):

    print(
        "\nALL 50 EFFICIENTNETB0 + SVM "
        "OUTER FOLDS COMPLETED AND VERIFIED."
    )

else:

    print(
        "\nExperiment is not yet complete."
    )

    print(
        "Rerun this SAME CELL after reconnecting."
    )

    print(
        "Valid completed folds will be skipped."
    )


print(
    "\nTesting partition was not used."
)


print(
    "\n"
    + "=" * 72
)

print(
    "HYBRID2 EFFICIENTNETB0 + SVM "
    "REPEATED RUNNER: PASSED"
)

print(
    "=" * 72
)

Required Hybrid2 functions: PASSED
R1/F1 expected inner-CV seed: 303101

EFFICIENTNETB0 FIXED-FEATURE + SVM
PERSISTENT RESUME CHECK
Results root:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_svm

Valid folds already saved: 50 / 50
Already-completed folds:
[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (7, 1), (7, 2), (7, 3), (7, 4), (7, 5), (8, 1), (8, 2), (8, 3), (8, 4), (8, 5), (9, 1), (9, 2), (9, 3), (9, 4), (9, 5), (10, 1), (10, 2), (10, 3), (10, 4), (10, 5)]

Persistence/resume setup: PASSED

REPEAT 1 / 10

------------------------------------------------------------------------
Repeat 1 / Fold 1
------------------------------------------------------------------------
Status: already completed and validated.
Action: SKIPPING.

-------------------------

In [ ]:
# ============================================================
# ACCESS AND AGGREGATE
# EFFICIENTNETB0 FIXED-FEATURE + SVM RESULTS
#
# Reads the exact files produced by the repeated Hybrid 2
# SVM experiment.
#
# Expected structure:
#
# repeated_nested_cv/efficientnetb0_svm/
#     repeat_01/
#         fold_01/
#             inner_search_results.csv
#             selected_parameters.json
#             outer_predictions.csv
#             outer_metrics.json
#             COMPLETED.json
#
# Produces:
#     all_fold_results.csv
#     repeat_level_summary.csv
#
# IMPORTANT:
#   - No model fitting occurs here.
#   - No feature extraction occurs here.
#   - Testing data is NOT accessed.
#   - Wilcoxon later uses the 10 repeat-level mean macro-F1
#     values, NOT the 50 folds as independent observations.
# ============================================================

import json
from pathlib import Path

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Results root
# ------------------------------------------------------------

RESULTS_ROOT = Path(
    "/content/drive/MyDrive/brain_tumour_colab/"
    "results/repeated_nested_cv/efficientnetb0_svm"
)


assert RESULTS_ROOT.exists(), (
    "Results directory does not exist:\n"
    f"{RESULTS_ROOT}"
)


# ------------------------------------------------------------
# 2. Required files for every completed fold
# ------------------------------------------------------------

REQUIRED_FILES = [
    "inner_search_results.csv",
    "selected_parameters.json",
    "outer_predictions.csv",
    "outer_metrics.json",
    "COMPLETED.json",
]


# ------------------------------------------------------------
# 3. Scan all 50 fold directories
# ------------------------------------------------------------

fold_records = []

problem_folds = []


for repeat_number in range(
    1,
    11,
):

    for fold_number in range(
        1,
        6,
    ):

        fold_directory = (
            RESULTS_ROOT
            / f"repeat_{repeat_number:02d}"
            / f"fold_{fold_number:02d}"
        )


        # ----------------------------------------------------
        # Check folder
        # ----------------------------------------------------

        if not fold_directory.exists():

            problem_folds.append(
                {
                    "repeat":
                        repeat_number,

                    "fold":
                        fold_number,

                    "problem":
                        "folder missing",

                    "path":
                        str(
                            fold_directory
                        ),
                }
            )

            continue


        # ----------------------------------------------------
        # Check required files
        # ----------------------------------------------------

        missing_files = [
            filename

            for filename
            in REQUIRED_FILES

            if not (
                fold_directory
                / filename
            ).exists()
        ]


        if missing_files:

            problem_folds.append(
                {
                    "repeat":
                        repeat_number,

                    "fold":
                        fold_number,

                    "problem":
                        "missing required files: "
                        + ", ".join(
                            missing_files
                        ),

                    "path":
                        str(
                            fold_directory
                        ),
                }
            )

            continue


        # ----------------------------------------------------
        # Load completion marker
        # ----------------------------------------------------

        with open(
            fold_directory
            / "COMPLETED.json",
            "r",
            encoding="utf-8",
        ) as file:

            completion = json.load(
                file
            )


        if completion.get(
            "status"
        ) != "completed":

            problem_folds.append(
                {
                    "repeat":
                        repeat_number,

                    "fold":
                        fold_number,

                    "problem":
                        "completion marker does not "
                        "say completed",

                    "path":
                        str(
                            fold_directory
                        ),
                }
            )

            continue


        # ----------------------------------------------------
        # Validate completion identifiers
        # ----------------------------------------------------

        if (
            int(
                completion.get(
                    "repeat"
                )
            )
            != repeat_number
        ):

            problem_folds.append(
                {
                    "repeat":
                        repeat_number,

                    "fold":
                        fold_number,

                    "problem":
                        "COMPLETED.json repeat mismatch",

                    "path":
                        str(
                            fold_directory
                        ),
                }
            )

            continue


        if (
            int(
                completion.get(
                    "fold"
                )
            )
            != fold_number
        ):

            problem_folds.append(
                {
                    "repeat":
                        repeat_number,

                    "fold":
                        fold_number,

                    "problem":
                        "COMPLETED.json fold mismatch",

                    "path":
                        str(
                            fold_directory
                        ),
                }
            )

            continue


        # ----------------------------------------------------
        # Load outer metrics
        # ----------------------------------------------------

        with open(
            fold_directory
            / "outer_metrics.json",
            "r",
            encoding="utf-8",
        ) as file:

            metrics = json.load(
                file
            )


        # ----------------------------------------------------
        # Load selected parameters
        # ----------------------------------------------------

        with open(
            fold_directory
            / "selected_parameters.json",
            "r",
            encoding="utf-8",
        ) as file:

            selected = json.load(
                file
            )


        best_parameters = (
            selected.get(
                "best_parameters",
                {},
            )
        )


        # ----------------------------------------------------
        # Validate saved prediction count
        # ----------------------------------------------------

        saved_predictions = (
            pd.read_csv(
                fold_directory
                / "outer_predictions.csv"
            )
        )


        if (
            len(
                saved_predictions
            )
            != 1120
        ):

            problem_folds.append(
                {
                    "repeat":
                        repeat_number,

                    "fold":
                        fold_number,

                    "problem":
                        "outer_predictions.csv does not "
                        "contain 1120 rows",

                    "path":
                        str(
                            fold_directory
                        ),
                }
            )

            continue


        # ----------------------------------------------------
        # Validate saved inner-search count
        # ----------------------------------------------------

        saved_search_results = (
            pd.read_csv(
                fold_directory
                / "inner_search_results.csv"
            )
        )


        if (
            len(
                saved_search_results
            )
            != 25
        ):

            problem_folds.append(
                {
                    "repeat":
                        repeat_number,

                    "fold":
                        fold_number,

                    "problem":
                        "inner_search_results.csv does not "
                        "contain 25 SVM configurations",

                    "path":
                        str(
                            fold_directory
                        ),
                }
            )

            continue


        # ----------------------------------------------------
        # Build one fold-level record
        #
        # IMPORTANT:
        # Optional metadata uses .get() because the first
        # completed fold may use the earlier persistence schema.
        #
        # This does NOT affect its experimental result.
        # ----------------------------------------------------

        fold_records.append(
            {
                "model":
                    metrics.get(
                        "model"
                    ),

                "feature_dimension":
                    metrics.get(
                        "feature_dimension",
                        selected.get(
                            "feature_dimension"
                        ),
                    ),

                "repeat":
                    int(
                        metrics[
                            "repeat"
                        ]
                    ),

                "fold":
                    int(
                        metrics[
                            "fold"
                        ]
                    ),

                "outer_split_seed":
                    metrics.get(
                        "outer_split_seed"
                    ),

                "inner_cv_seed":
                    selected.get(
                        "inner_cv_seed"
                    ),

                "outer_training_samples":
                    metrics.get(
                        "outer_training_samples"
                    ),

                "outer_validation_samples":
                    metrics.get(
                        "outer_validation_samples"
                    ),

                "best_C":
                    best_parameters.get(
                        "classifier__C"
                    ),

                "best_gamma":
                    best_parameters.get(
                        "classifier__gamma"
                    ),

                "best_inner_macro_f1":
                    metrics.get(
                        "best_inner_macro_f1"
                    ),

                "accuracy":
                    metrics.get(
                        "accuracy"
                    ),

                "balanced_accuracy":
                    metrics.get(
                        "balanced_accuracy"
                    ),

                "macro_precision":
                    metrics.get(
                        "macro_precision"
                    ),

                "macro_recall":
                    metrics.get(
                        "macro_recall"
                    ),

                "macro_f1":
                    metrics.get(
                        "macro_f1"
                    ),

                "elapsed_seconds":
                    metrics.get(
                        "elapsed_seconds"
                    ),

                "fold_directory":
                    str(
                        fold_directory
                    ),
            }
        )


# ------------------------------------------------------------
# 4. Create fold-level table
# ------------------------------------------------------------

fold_results_df = pd.DataFrame(
    fold_records
)


if not fold_results_df.empty:

    fold_results_df = (
        fold_results_df
        .sort_values(
            [
                "repeat",
                "fold",
            ]
        )
        .reset_index(
            drop=True
        )
    )


# ------------------------------------------------------------
# 5. Display scan result
# ------------------------------------------------------------

print("=" * 72)

print(
    "EFFICIENTNETB0 + SVM SAVED RESULT CHECK"
)

print("=" * 72)


print(
    "Results root:",
    RESULTS_ROOT,
)


print(
    "\nValid fold results found:",
    len(
        fold_results_df
    ),
    "/ 50",
)


print(
    "Problem folds:",
    len(
        problem_folds
    ),
)


if problem_folds:

    problem_folds_df = (
        pd.DataFrame(
            problem_folds
        )
    )


    print(
        "\nPROBLEM FOLDS:"
    )


    display(
        problem_folds_df
    )


else:

    print(
        "\nAll 50 fold directories contain "
        "the required saved files."
    )


# ------------------------------------------------------------
# 6. Stop aggregation if fewer than 50 valid folds exist
# ------------------------------------------------------------

if len(
    fold_results_df
) != 50:

    raise RuntimeError(
        "\nOnly "
        f"{len(fold_results_df)} / 50 "
        "valid fold result sets were found.\n"
        "See the problem-fold table above."
    )


# ------------------------------------------------------------
# 7. Validate repeated structure
# ------------------------------------------------------------

assert (
    fold_results_df[
        "repeat"
    ]
    .nunique()
    == 10
)


assert (
    fold_results_df
    .groupby(
        "repeat"
    )
    .size()
    .eq(
        5
    )
    .all()
)


assert (
    fold_results_df[
        [
            "repeat",
            "fold",
        ]
    ]
    .drop_duplicates()
    .shape[
        0
    ]
    == 50
)


assert np.isfinite(
    fold_results_df[
        "macro_f1"
    ]
).all()


# ------------------------------------------------------------
# 8. Build the 10 repeat-level observations
#
# These mean macro-F1 values are the values later used
# for paired Wilcoxon comparison.
# ------------------------------------------------------------

repeat_level_summary_df = (
    fold_results_df
    .groupby(
        "repeat",
        as_index=False,
    )
    .agg(
        mean_accuracy=(
            "accuracy",
            "mean",
        ),

        mean_balanced_accuracy=(
            "balanced_accuracy",
            "mean",
        ),

        mean_macro_precision=(
            "macro_precision",
            "mean",
        ),

        mean_macro_recall=(
            "macro_recall",
            "mean",
        ),

        mean_macro_f1=(
            "macro_f1",
            "mean",
        ),

        sd_macro_f1=(
            "macro_f1",
            "std",
        ),
    )
)


repeat_level_summary_df[
    "wilcoxon_value"
] = (
    repeat_level_summary_df[
        "mean_macro_f1"
    ]
)


assert (
    len(
        repeat_level_summary_df
    )
    == 10
)


assert (
    repeat_level_summary_df[
        "repeat"
    ]
    .tolist()
    == list(
        range(
            1,
            11,
        )
    )
)


# ------------------------------------------------------------
# 9. Save consolidated tables
# ------------------------------------------------------------

ALL_FOLD_RESULTS_PATH = (
    RESULTS_ROOT
    / "all_fold_results.csv"
)


REPEAT_LEVEL_SUMMARY_PATH = (
    RESULTS_ROOT
    / "repeat_level_summary.csv"
)


fold_results_df.to_csv(
    ALL_FOLD_RESULTS_PATH,
    index=False,
)


repeat_level_summary_df.to_csv(
    REPEAT_LEVEL_SUMMARY_PATH,
    index=False,
)


assert ALL_FOLD_RESULTS_PATH.exists()

assert REPEAT_LEVEL_SUMMARY_PATH.exists()


# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print(
    "\n"
    + "=" * 72
)

print(
    "50 OUTER-FOLD RESULTS"
)

print(
    "=" * 72
)


display(
    fold_results_df
)


print(
    "\n"
    + "=" * 72
)

print(
    "10 REPEAT-LEVEL RESULTS"
)

print(
    "=" * 72
)


display(
    repeat_level_summary_df
)


# ------------------------------------------------------------
# 11. Print the 10 values needed later for Wilcoxon
# ------------------------------------------------------------

print(
    "\nRepeat-level macro-F1 / Wilcoxon values:"
)


for row in (
    repeat_level_summary_df
    .itertuples(
        index=False
    )
):

    print(
        f"R{int(row.repeat)} "
        f"{row.mean_macro_f1:.6f}"
    )


# ------------------------------------------------------------
# 12. Overall descriptive values
# ------------------------------------------------------------

print(
    "\nOverall mean outer-fold macro F1:",
    round(
        fold_results_df[
            "macro_f1"
        ].mean(),
        6,
    ),
)


print(
    "Mean of 10 repeat-level macro F1 values:",
    round(
        repeat_level_summary_df[
            "mean_macro_f1"
        ].mean(),
        6,
    ),
)


print(
    "\nSaved fold-level table:"
)

print(
    ALL_FOLD_RESULTS_PATH
)


print(
    "\nSaved repeat-level table:"
)

print(
    REPEAT_LEVEL_SUMMARY_PATH
)


print(
    "\nWilcoxon statistical unit:"
)

print(
    "10 repeat-level mean macro-F1 values."
)


print(
    "\nTesting partition was not accessed."
)


print(
    "\nEFFICIENTNETB0 + SVM RESULT ACCESS PASSED."
)

EFFICIENTNETB0 + SVM SAVED RESULT CHECK
Results root: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_svm

Valid fold results found: 50 / 50
Problem folds: 0

All 50 fold directories contain the required saved files.

50 OUTER-FOLD RESULTS


,model,feature_dimension,repeat,fold,outer_split_seed,inner_cv_seed,outer_training_samples,outer_validation_samples,best_C,best_gamma,best_inner_macro_f1,accuracy,balanced_accuracy,macro_precision,macro_recall,macro_f1,elapsed_seconds,fold_directory
0,EfficientNetB0_fixed_features_SVM,NaN,1,1,202601,303101,4480,1120,10,scale,0.946264,0.969643,0.969643,0.970210,0.969643,0.969703,898.195966,/content/drive/MyDrive/brain_tumour_colab/resu...
1,EfficientNetB0_fixed_features_SVM,1280.0,1,2,202601,303102,4480,1120,10,scale,0.950320,0.958036,0.958036,0.958403,0.958036,0.957941,826.587332,/content/drive/MyDrive/brain_tumour_colab/resu...
2,EfficientNetB0_fixed_features_SVM,1280.0,1,3,202601,303103,4480,1120,100,scale,0.948763,0.966964,0.966964,0.968148,0.966964,0.967131,816.467225,/content/drive/MyDrive/brain_tumour_colab/resu...
3,EfficientNetB0_fixed_features_SVM,1280.0,1,4,202601,303104,4480,1120,10,scale,0.945740,0.960714,0.960714,0.960964,0.960714,0.960686,817.693064,/content/drive/MyDrive/brain_tumour_colab/resu...
4,EfficientNetB0_fixed_features_SVM,1280.0,1,5,202601,303105,4480,1120,10,scale,0.945536,0.956250,0.956250,0.960480,0.956250,0.956796,808.223281,/content/drive/MyDrive/brain_tumour_colab/resu...
5,EfficientNetB0_fixed_features_SVM,1280.0,2,1,202602,303201,4480,1120,100,scale,0.946600,0.967857,0.967857,0.968862,0.967857,0.967890,799.361159,/content/drive/MyDrive/brain_tumour_colab/resu...
6,EfficientNetB0_fixed_features_SVM,1280.0,2,2,202602,303202,4480,1120,10,scale,0.950336,0.952679,0.952679,0.956879,0.952679,0.953078,793.012600,/content/drive/MyDrive/brain_tumour_colab/resu...
7,EfficientNetB0_fixed_features_SVM,1280.0,2,3,202602,303203,4480,1120,100,scale,0.943681,0.969643,0.969643,0.970050,0.969643,0.969717,834.893827,/content/drive/MyDrive/brain_tumour_colab/resu...
8,EfficientNetB0_fixed_features_SVM,1280.0,2,4,202602,303204,4480,1120,10,scale,0.943832,0.970536,0.970536,0.970928,0.970536,0.970637,853.617028,/content/drive/MyDrive/brain_tumour_colab/resu...
9,EfficientNetB0_fixed_features_SVM,1280.0,2,5,202602,303205,4480,1120,10,scale,0.947323,0.962500,0.962500,0.962812,0.962500,0.962543,791.829854,/content/drive/MyDrive/brain_tumour_colab/resu...



10 REPEAT-LEVEL RESULTS


,repeat,mean_accuracy,mean_balanced_accuracy,mean_macro_precision,mean_macro_recall,mean_macro_f1,sd_macro_f1,wilcoxon_value
0,1,0.962321,0.962321,0.963641,0.962321,0.962451,0.005699,0.962451
1,2,0.964643,0.964643,0.965906,0.964643,0.964773,0.007251,0.964773
2,3,0.963750,0.963750,0.964502,0.963750,0.963843,0.004205,0.963843
3,4,0.962857,0.962857,0.963530,0.962857,0.962954,0.006726,0.962954
4,5,0.965536,0.965536,0.966228,0.965536,0.965606,0.006935,0.965606
5,6,0.963393,0.963393,0.963963,0.963393,0.963463,0.006667,0.963463
6,7,0.964107,0.964107,0.964855,0.964107,0.964205,0.003145,0.964205
7,8,0.961786,0.961786,0.963018,0.961786,0.961940,0.007264,0.961940
8,9,0.964643,0.964643,0.965260,0.964643,0.964722,0.004471,0.964722
9,10,0.961786,0.961786,0.962618,0.961786,0.961902,0.004790,0.961902



Repeat-level macro-F1 / Wilcoxon values:
R1 0.962451
R2 0.964773
R3 0.963843
R4 0.962954
R5 0.965606
R6 0.963463
R7 0.964205
R8 0.961940
R9 0.964722
R10 0.961902

Overall mean outer-fold macro F1: 0.963586
Mean of 10 repeat-level macro F1 values: 0.963586

Saved fold-level table:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_svm/all_fold_results.csv

Saved repeat-level table:
/content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_svm/repeat_level_summary.csv

Wilcoxon statistical unit:
10 repeat-level mean macro-F1 values.

Testing partition was not accessed.

EFFICIENTNETB0 + SVM RESULT ACCESS PASSED.


In [10]:
# ============================================================
# EFFICIENTNETB0 + RF — CHECKPOINTED STEP 1
# RESTART / RECOVERY CONFIGURATION
#
# CRITICAL RESUME BEHAVIOUR:
#
#   - candidate manifest saved once per outer fold
#   - EVERY completed candidate × inner-fold fit will be saved
#   - completed stages will be saved
#   - completed inner searches will be saved
#   - completed outer folds will be skipped
#   - interrupted outer folds will resume INSIDE the search
#
# IMPORTANT:
#   - Existing EfficientNetB0 1,280-D features are reused.
#   - Existing shared 10 × 5 assignments are reused.
#   - OLD cached fold_assignments remain ignored.
#   - No feature extraction occurs.
#   - No fold assignments are regenerated.
#   - Existing completed RF folds are preserved.
#   - Nothing is deleted in this cell.
#   - Testing data is NEVER used.
# ============================================================

import json
import math
import os

from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestClassifier,
)

from sklearn.model_selection import (
    ParameterGrid,
    ParameterSampler,
    StratifiedKFold,
)


# ============================================================
# 1. REQUIRED HYBRID2 STATE
#
# The EfficientNetB0 cache / shared partition setup must
# already have been rerun after the Colab restart.
# ============================================================

required_efficientnetb0_rf_globals = [

    "make_hybrid_outer_partition",

    "get_hybrid_inner_cv_seed",

    "FEATURE_DIMENSION",

    "CLASS_NAMES",

    "INDEX_TO_CLASS",
]


missing_efficientnetb0_rf_globals = [

    variable_name

    for variable_name
    in required_efficientnetb0_rf_globals

    if variable_name
    not in globals()
]


assert not missing_efficientnetb0_rf_globals, (
    "Required Hybrid2 setup is missing:\n"
    f"{missing_efficientnetb0_rf_globals}\n\n"
    "Rerun the existing EfficientNetB0 cache / repeated-"
    "partition setup cells first."
)


assert (
    FEATURE_DIMENSION
    == 1280
)


# ============================================================
# 2. VALIDATE THE EXISTING EFFICIENTNETB0 FEATURE/PARTITION
#    STATE WITHOUT TRAINING ANY MODEL
#
# This confirms the existing cache and shared repeated split
# are loaded correctly.
# ============================================================

checkpoint_test_partition = (
    make_hybrid_outer_partition(
        repeat_number=1,
        fold_number=1,
    )
)


checkpoint_test_X_train = (
    checkpoint_test_partition[
        "outer_training_features"
    ]
)


checkpoint_test_y_train = (
    np.asarray(
        checkpoint_test_partition[
            "outer_training_labels"
        ],
        dtype=int,
    )
)


checkpoint_test_X_validation = (
    checkpoint_test_partition[
        "outer_validation_features"
    ]
)


checkpoint_test_y_validation = (
    np.asarray(
        checkpoint_test_partition[
            "outer_validation_labels"
        ],
        dtype=int,
    )
)


checkpoint_test_validation_paths = (
    np.asarray(
        checkpoint_test_partition[
            "outer_validation_paths"
        ],
        dtype=str,
    )
)


assert checkpoint_test_X_train.shape == (
    4480,
    1280,
)


assert checkpoint_test_X_validation.shape == (
    1120,
    1280,
)


assert len(
    checkpoint_test_y_train
) == 4480


assert len(
    checkpoint_test_y_validation
) == 1120


assert len(
    checkpoint_test_validation_paths
) == 1120


assert (
    len(
        np.unique(
            checkpoint_test_validation_paths
        )
    )
    == 1120
)


assert np.isfinite(
    checkpoint_test_X_train
).all()


assert np.isfinite(
    checkpoint_test_X_validation
).all()


assert np.array_equal(
    np.bincount(
        checkpoint_test_y_train,
        minlength=4,
    ),
    np.array(
        [
            1120,
            1120,
            1120,
            1120,
        ]
    ),
)


assert np.array_equal(
    np.bincount(
        checkpoint_test_y_validation,
        minlength=4,
    ),
    np.array(
        [
            280,
            280,
            280,
            280,
        ]
    ),
)


# ============================================================
# 3. RESULTS ROOT
#
# SAME location used by the previous EfficientNetB0 + RF
# runner.
# ============================================================

EFFICIENTNETB0_RF_RESULTS_ROOT = Path(
    "/content/drive/MyDrive/brain_tumour_colab/"
    "results/repeated_nested_cv/efficientnetb0_rf"
)


EFFICIENTNETB0_RF_RESULTS_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# Keep the familiar notebook variable available as well.
REPEATED_EFFICIENTNETB0_RF_DIR = (
    EFFICIENTNETB0_RF_RESULTS_ROOT
)


# ============================================================
# 4. DETERMINISTIC RF SEED SCHEDULES
#
# SAME schedules used for:
#
#   Classical RF
#   ResNet50 fixed-feature + RF
#   previous EfficientNetB0 + RF implementation
# ============================================================

BASE_EFFICIENTNETB0_RF_MODEL_SEED = 404000

BASE_EFFICIENTNETB0_RF_SEARCH_SEED = 414000


def get_efficientnetb0_rf_model_seed(
    repeat_number,
    fold_number,
):

    return (
        BASE_EFFICIENTNETB0_RF_MODEL_SEED
        + repeat_number * 100
        + fold_number
    )


def get_efficientnetb0_rf_search_seed(
    repeat_number,
    fold_number,
):

    return (
        BASE_EFFICIENTNETB0_RF_SEARCH_SEED
        + repeat_number * 100
        + fold_number
    )


assert (
    get_hybrid_inner_cv_seed(
        1,
        1,
    )
    == 303101
)


assert (
    get_efficientnetb0_rf_model_seed(
        1,
        1,
    )
    == 404101
)


assert (
    get_efficientnetb0_rf_search_seed(
        1,
        1,
    )
    == 414101
)


# ============================================================
# 5. EXACT RANDOM FOREST SEARCH SPACE
#
# SAME RF parameter space as the classical RF and ResNet50 RF.
# ============================================================

EFFICIENTNETB0_RF_PARAMETER_DISTRIBUTIONS = {

    "criterion": [
        "gini",
        "entropy",
    ],

    "max_depth": [
        None,
        20,
        40,
        60,
        80,
    ],

    "min_samples_split": [
        2,
        5,
        10,
        20,
    ],

    "min_samples_leaf": [
        1,
        2,
        4,
        8,
    ],

    "max_features": [
        "sqrt",
        "log2",
        0.02,
        0.05,
        0.10,
        0.20,
    ],

    "bootstrap": [
        True,
        False,
    ],
}


efficientnetb0_rf_total_possible_combinations = len(
    list(
        ParameterGrid(
            EFFICIENTNETB0_RF_PARAMETER_DISTRIBUTIONS
        )
    )
)


assert (
    efficientnetb0_rf_total_possible_combinations
    == 1920
)


# ============================================================
# 6. CHECKPOINTED SUCCESSIVE-HALVING CONFIGURATION
#
# Same search budget:
#
#   Stage 1:
#       100 candidates × 25 trees
#
#   Stage 2:
#        34 candidates × 75 trees
#
#   Stage 3:
#        12 candidates × 225 trees
#
#   Stage 4:
#         4 candidates × 675 trees
#
#   Final:
#         best Stage-4 candidate is refitted on all 4,480
#         outer-training samples.
# ============================================================

EFFICIENTNETB0_RF_INITIAL_CANDIDATES = 100

EFFICIENTNETB0_RF_HALVING_FACTOR = 3


EFFICIENTNETB0_RF_STAGE_RESOURCES = [
    25,
    75,
    225,
    675,
]


EFFICIENTNETB0_RF_EXPECTED_STAGE_CANDIDATES = [
    100,
    34,
    12,
    4,
]


assert (
    EFFICIENTNETB0_RF_INITIAL_CANDIDATES
    == 100
)


assert (
    EFFICIENTNETB0_RF_HALVING_FACTOR
    == 3
)


assert (
    EFFICIENTNETB0_RF_STAGE_RESOURCES
    == [
        25,
        75,
        225,
        675,
    ]
)


assert (
    EFFICIENTNETB0_RF_EXPECTED_STAGE_CANDIDATES
    == [
        100,
        34,
        12,
        4,
    ]
)


# ============================================================
# 7. ATOMIC DATAFRAME SAVE
#
# Each checkpoint is first written to a temporary file and
# then atomically moved to its final filename.
# ============================================================

def efficientnetb0_rf_save_dataframe_atomic(
    dataframe,
    destination_path,
):

    destination_path = Path(
        destination_path
    )


    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = (
        destination_path.parent
        / (
            destination_path.name
            + ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
        newline="",
    ) as file:

        dataframe.to_csv(
            file,
            index=False,
        )


        file.flush()

        os.fsync(
            file.fileno()
        )


    os.replace(
        temporary_path,
        destination_path,
    )


    assert destination_path.exists()


# ============================================================
# 8. JSON SERIALISATION
# ============================================================

def efficientnetb0_rf_json_default(
    value,
):

    if isinstance(
        value,
        np.integer,
    ):

        return int(
            value
        )


    if isinstance(
        value,
        np.floating,
    ):

        return float(
            value
        )


    if isinstance(
        value,
        np.bool_,
    ):

        return bool(
            value
        )


    if isinstance(
        value,
        np.ndarray,
    ):

        return value.tolist()


    if isinstance(
        value,
        Path,
    ):

        return str(
            value
        )


    raise TypeError(
        "Object of type "
        f"{type(value).__name__} "
        "is not JSON serializable."
    )


# ============================================================
# 9. ATOMIC JSON SAVE
# ============================================================

def efficientnetb0_rf_save_json_atomic(
    dictionary,
    destination_path,
):

    destination_path = Path(
        destination_path
    )


    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )


    temporary_path = (
        destination_path.parent
        / (
            destination_path.name
            + ".tmp"
        )
    )


    with open(
        temporary_path,
        "w",
        encoding="utf-8",
    ) as file:

        json.dump(
            dictionary,
            file,
            indent=4,
            ensure_ascii=False,
            default=
                efficientnetb0_rf_json_default,
        )


        file.flush()

        os.fsync(
            file.fileno()
        )


    os.replace(
        temporary_path,
        destination_path,
    )


    assert destination_path.exists()


# ============================================================
# 10. CHECKPOINTED FOLD PATHS
#
# Final fold filenames stay identical to the previous Hybrid2
# RF implementation.
#
# NEW resumable files live in:
#
#       search_progress/
# ============================================================

def get_checkpointed_efficientnetb0_rf_fold_paths(
    repeat_number,
    fold_number,
):

    fold_directory = (
        EFFICIENTNETB0_RF_RESULTS_ROOT
        / f"repeat_{repeat_number:02d}"
        / f"fold_{fold_number:02d}"
    )


    search_progress_directory = (
        fold_directory
        / "search_progress"
    )


    return {

        # ----------------------------------------------------
        # Directories
        # ----------------------------------------------------

        "directory":
            fold_directory,

        "search_progress_directory":
            search_progress_directory,


        # ----------------------------------------------------
        # New resumable search state
        # ----------------------------------------------------

        "candidate_manifest":
            (
                search_progress_directory
                / "candidate_manifest.csv"
            ),

        "inner_fit_progress":
            (
                search_progress_directory
                / "inner_fit_progress.csv"
            ),

        "search_complete":
            (
                search_progress_directory
                / "search_complete.json"
            ),


        # ----------------------------------------------------
        # Final RF fold outputs
        # ----------------------------------------------------

        "search_results":
            (
                fold_directory
                / "inner_search_results.csv"
            ),

        "selected_parameters":
            (
                fold_directory
                / "selected_parameters.json"
            ),

        "outer_predictions":
            (
                fold_directory
                / "outer_predictions.csv"
            ),

        "outer_metrics":
            (
                fold_directory
                / "outer_metrics.json"
            ),

        "completion_marker":
            (
                fold_directory
                / "COMPLETED.json"
            ),
    }


# ============================================================
# 11. FULL OUTER-FOLD COMPLETION CHECK
#
# Existing fully completed folds from the previous RF runner
# remain valid and can be skipped.
#
# search_progress/ is NOT required for a previously completed
# fold.
# ============================================================

def is_checkpointed_efficientnetb0_rf_fold_complete(
    repeat_number,
    fold_number,
):

    paths = (
        get_checkpointed_efficientnetb0_rf_fold_paths(
            repeat_number,
            fold_number,
        )
    )


    required_final_files = [

        paths[
            "search_results"
        ],

        paths[
            "selected_parameters"
        ],

        paths[
            "outer_predictions"
        ],

        paths[
            "outer_metrics"
        ],

        paths[
            "completion_marker"
        ],
    ]


    if not all(
        path.exists()
        for path
        in required_final_files
    ):

        return False


    try:

        # ----------------------------------------------------
        # Completion marker
        # ----------------------------------------------------

        with open(
            paths[
                "completion_marker"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            completion = json.load(
                file
            )


        if (
            completion.get(
                "status"
            )
            != "completed"
        ):

            return False


        if (
            completion.get(
                "model"
            )
            != "EfficientNetB0_fixed_features_RF"
        ):

            return False


        if (
            int(
                completion[
                    "repeat"
                ]
            )
            != repeat_number
        ):

            return False


        if (
            int(
                completion[
                    "fold"
                ]
            )
            != fold_number
        ):

            return False


        # ----------------------------------------------------
        # Outer metrics
        # ----------------------------------------------------

        with open(
            paths[
                "outer_metrics"
            ],
            "r",
            encoding="utf-8",
        ) as file:

            metrics = json.load(
                file
            )


        if (
            metrics.get(
                "model"
            )
            != "EfficientNetB0_fixed_features_RF"
        ):

            return False


        if (
            int(
                metrics[
                    "repeat"
                ]
            )
            != repeat_number
        ):

            return False


        if (
            int(
                metrics[
                    "fold"
                ]
            )
            != fold_number
        ):

            return False


        if (
            int(
                metrics[
                    "outer_training_samples"
                ]
            )
            != 4480
        ):

            return False


        if (
            int(
                metrics[
                    "outer_validation_samples"
                ]
            )
            != 1120
        ):

            return False


        if not np.isfinite(
            float(
                metrics[
                    "macro_f1"
                ]
            )
        ):

            return False


        # ----------------------------------------------------
        # Predictions
        # ----------------------------------------------------

        predictions_df = pd.read_csv(
            paths[
                "outer_predictions"
            ]
        )


        if (
            len(
                predictions_df
            )
            != 1120
        ):

            return False


        if (
            predictions_df[
                "relative_path"
            ]
            .nunique()
            != 1120
        ):

            return False


        return True


    except Exception:

        return False


# ============================================================
# 12. INNER CV CONSTRUCTOR
#
# Same deterministic 3 inner folds are reused for every
# candidate within the given outer fold.
# ============================================================

def make_checkpointed_efficientnetb0_rf_inner_cv(
    repeat_number,
    fold_number,
):

    return StratifiedKFold(
        n_splits=3,
        shuffle=True,
        random_state=
            get_hybrid_inner_cv_seed(
                repeat_number,
                fold_number,
            ),
    )


# ============================================================
# 13. PARAMETER RECONSTRUCTION HELPERS
#
# Required because candidate manifests are stored as CSV and
# loaded again after Colab disconnects.
# ============================================================

def efficientnetb0_rf_optional_integer(
    value,
):

    if pd.isna(
        value
    ):

        return None


    return int(
        value
    )


def efficientnetb0_rf_normalise_max_features(
    value,
):

    if isinstance(
        value,
        str,
    ):

        cleaned_value = (
            value.strip()
        )


        if cleaned_value in (
            "sqrt",
            "log2",
        ):

            return cleaned_value


        return float(
            cleaned_value
        )


    return float(
        value
    )


def efficientnetb0_rf_candidate_parameters_from_row(
    row,
):

    bootstrap_value = (
        row[
            "bootstrap"
        ]
    )


    if isinstance(
        bootstrap_value,
        str,
    ):

        bootstrap_value = (
            bootstrap_value
            .strip()
            .lower()
            == "true"
        )

    else:

        bootstrap_value = bool(
            bootstrap_value
        )


    return {

        "criterion":
            str(
                row[
                    "criterion"
                ]
            ),

        "max_depth":
            efficientnetb0_rf_optional_integer(
                row[
                    "max_depth"
                ]
            ),

        "min_samples_split":
            int(
                row[
                    "min_samples_split"
                ]
            ),

        "min_samples_leaf":
            int(
                row[
                    "min_samples_leaf"
                ]
            ),

        "max_features":
            efficientnetb0_rf_normalise_max_features(
                row[
                    "max_features"
                ]
            ),

        "bootstrap":
            bootstrap_value,
    }


# ============================================================
# 14. AUDIT CURRENT EFFICIENTNETB0 + RF STATE
#
# NOTHING IS DELETED.
# ============================================================

fully_completed_efficientnetb0_rf_folds = []

checkpointed_partial_efficientnetb0_rf_folds = []

other_incomplete_efficientnetb0_rf_directories = []


for repeat_number in range(
    1,
    11,
):

    for fold_number in range(
        1,
        6,
    ):

        paths = (
            get_checkpointed_efficientnetb0_rf_fold_paths(
                repeat_number,
                fold_number,
            )
        )


        if (
            is_checkpointed_efficientnetb0_rf_fold_complete(
                repeat_number,
                fold_number,
            )
        ):

            fully_completed_efficientnetb0_rf_folds.append(
                (
                    repeat_number,
                    fold_number,
                )
            )


            continue


        if (
            paths[
                "inner_fit_progress"
            ]
            .exists()
        ):

            try:

                progress_df = pd.read_csv(
                    paths[
                        "inner_fit_progress"
                    ]
                )


                completed_inner_fits = len(
                    progress_df
                )


            except Exception:

                completed_inner_fits = -1


            checkpointed_partial_efficientnetb0_rf_folds.append(
                (
                    repeat_number,
                    fold_number,
                    completed_inner_fits,
                )
            )


            continue


        if (
            paths[
                "directory"
            ]
            .exists()
        ):

            other_incomplete_efficientnetb0_rf_directories.append(
                (
                    repeat_number,
                    fold_number,
                )
            )


# ============================================================
# 15. DISPLAY
# ============================================================

print("=" * 80)

print(
    "EFFICIENTNETB0 + RF CHECKPOINTED RECOVERY SETUP"
)

print("=" * 80)


print(
    "Results root:",
    EFFICIENTNETB0_RF_RESULTS_ROOT,
)


print(
    "\nValidated outer-training shape:",
    checkpoint_test_X_train.shape,
)


print(
    "Validated outer-validation shape:",
    checkpoint_test_X_validation.shape,
)


print(
    "Feature dimension:",
    FEATURE_DIMENSION,
)


print(
    "\nRF parameter combinations:",
    efficientnetb0_rf_total_possible_combinations,
)


print(
    "Initial candidates:",
    EFFICIENTNETB0_RF_INITIAL_CANDIDATES,
)


print(
    "Candidate progression:",
    "100 -> 34 -> 12 -> 4 -> winner",
)


print(
    "Tree resources:",
    EFFICIENTNETB0_RF_STAGE_RESOURCES,
)


print(
    "\nR1/F1 inner-CV seed:",
    get_hybrid_inner_cv_seed(
        1,
        1,
    ),
)


print(
    "R1/F1 RF model seed:",
    get_efficientnetb0_rf_model_seed(
        1,
        1,
    ),
)


print(
    "R1/F1 RF search seed:",
    get_efficientnetb0_rf_search_seed(
        1,
        1,
    ),
)


print(
    "\nFully completed outer folds:",
    len(
        fully_completed_efficientnetb0_rf_folds
    ),
    "/ 50",
)


if fully_completed_efficientnetb0_rf_folds:

    print(
        fully_completed_efficientnetb0_rf_folds
    )


print(
    "\nFolds with NEW checkpoint progress:",
    len(
        checkpointed_partial_efficientnetb0_rf_folds
    ),
)


if checkpointed_partial_efficientnetb0_rf_folds:

    print(
        checkpointed_partial_efficientnetb0_rf_folds
    )


print(
    "\nOther incomplete fold directories:",
    len(
        other_incomplete_efficientnetb0_rf_directories
    ),
)


if other_incomplete_efficientnetb0_rf_directories:

    print(
        other_incomplete_efficientnetb0_rf_directories
    )


print(
    "\nIMPORTANT:"
)


print(
    "Nothing was deleted."
)


print(
    "No Random Forest model was fitted."
)


print(
    "No EfficientNetB0 features were extracted."
)


print(
    "The existing shared 10 × 5 assignments were reused."
)


print(
    "Testing was not accessed."
)


print(
    "\nEFFICIENTNETB0 + RF CHECKPOINTED STEP 1: PASSED"
)


# ============================================================
# 16. CLEAN UP TEMPORARY PARTITION VALIDATION OBJECTS
# ============================================================

del checkpoint_test_partition

del checkpoint_test_X_train
del checkpoint_test_y_train

del checkpoint_test_X_validation
del checkpoint_test_y_validation

del checkpoint_test_validation_paths

EFFICIENTNETB0 + RF CHECKPOINTED RECOVERY SETUP
Results root: /content/drive/MyDrive/brain_tumour_colab/results/repeated_nested_cv/efficientnetb0_rf

Validated outer-training shape: (4480, 1280)
Validated outer-validation shape: (1120, 1280)
Feature dimension: 1280

RF parameter combinations: 1920
Initial candidates: 100
Candidate progression: 100 -> 34 -> 12 -> 4 -> winner
Tree resources: [25, 75, 225, 675]

R1/F1 inner-CV seed: 303101
R1/F1 RF model seed: 404101
R1/F1 RF search seed: 414101

Fully completed outer folds: 43 / 50
[(1, 1), (1, 2), (1, 3), (1, 4), (1, 5), (2, 1), (2, 2), (2, 3), (2, 4), (2, 5), (3, 1), (3, 2), (3, 3), (3, 4), (3, 5), (4, 1), (4, 2), (4, 3), (4, 4), (4, 5), (5, 1), (5, 2), (5, 3), (5, 4), (5, 5), (6, 1), (6, 2), (6, 3), (6, 4), (6, 5), (7, 1), (7, 2), (7, 3), (7, 4), (7, 5), (8, 1), (8, 2), (8, 3), (8, 4), (8, 5), (9, 1), (9, 2), (9, 3)]

Folds with NEW checkpoint progress: 1
[(9, 4, 402)]

Other incomplete fold directories: 0

IMPORTANT:
Nothing was dele

In [ ]:
# ============================================================
# EFFICIENTNETB0 + RF — CHECKPOINTED STEP 2
# MASTER 10 × 5 REPEATED NESTED-CV RUNNER
#
# TOTAL:
#   10 repeats × 5 outer folds = 50 outer evaluations
#
# RESUME BEHAVIOUR:
#
#   COMPLETED OUTER FOLD
#       -> SKIPPED
#
#   COMPLETED INNER SEARCH
#       -> LOADED
#       -> only final refit / outer evaluation is repeated
#
#   PARTIALLY COMPLETED INNER SEARCH
#       -> candidate manifest loaded
#       -> completed inner fits loaded
#       -> completed fits SKIPPED
#       -> resumes from first missing candidate × inner fold
#
# CHECKPOINT FREQUENCY:
#   EVERY candidate × inner-fold fit is saved immediately.
#
# SEARCH:
#   100 initial RF candidates
#   3-fold stratified inner CV
#   scoring = macro-F1
#
#   Stage 1:
#       100 candidates × 25 trees
#
#   Stage 2:
#        34 candidates × 75 trees
#
#   Stage 3:
#        12 candidates × 225 trees
#
#   Stage 4:
#         4 candidates × 675 trees
#
#   Winner:
#       best Stage-4 candidate
#
# IMPORTANT:
#   - Existing EfficientNetB0 1,280-D features are reused.
#   - Shared 10 × 5 assignments are reused.
#   - Outer validation is NEVER used in model selection.
#   - Testing is NEVER accessed.
# ============================================================

import gc
import json
import math
import time

import numpy as np
import pandas as pd

from sklearn.ensemble import (
    RandomForestClassifier,
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)

from sklearn.model_selection import (
    ParameterSampler,
)


# ============================================================
# 1. MASTER VALIDATION
# ============================================================

assert FEATURE_DIMENSION == 1280

assert (
    EFFICIENTNETB0_RF_INITIAL_CANDIDATES
    == 100
)

assert (
    EFFICIENTNETB0_RF_HALVING_FACTOR
    == 3
)

assert (
    EFFICIENTNETB0_RF_STAGE_RESOURCES
    == [
        25,
        75,
        225,
        675,
    ]
)

assert (
    EFFICIENTNETB0_RF_EXPECTED_STAGE_CANDIDATES
    == [
        100,
        34,
        12,
        4,
    ]
)


# ============================================================
# 2. CREATE / LOAD DETERMINISTIC CANDIDATE MANIFEST
# ============================================================

def get_or_create_efficientnetb0_rf_candidate_manifest(
    repeat_number,
    fold_number,
    candidate_manifest_path,
):

    if candidate_manifest_path.exists():

        candidate_manifest_df = (
            pd.read_csv(
                candidate_manifest_path
            )
        )


        print(
            "Loaded existing candidate manifest."
        )


    else:

        sampled_candidates = list(
            ParameterSampler(

                EFFICIENTNETB0_RF_PARAMETER_DISTRIBUTIONS,

                n_iter=
                    EFFICIENTNETB0_RF_INITIAL_CANDIDATES,

                random_state=
                    get_efficientnetb0_rf_search_seed(
                        repeat_number,
                        fold_number,
                    ),
            )
        )


        assert len(
            sampled_candidates
        ) == 100


        candidate_rows = []


        for (
            candidate_id,
            parameters,
        ) in enumerate(
            sampled_candidates,
            start=1,
        ):

            candidate_rows.append(
                {
                    "candidate_id":
                        int(
                            candidate_id
                        ),

                    "criterion":
                        parameters[
                            "criterion"
                        ],

                    "max_depth":
                        parameters[
                            "max_depth"
                        ],

                    "min_samples_split":
                        parameters[
                            "min_samples_split"
                        ],

                    "min_samples_leaf":
                        parameters[
                            "min_samples_leaf"
                        ],

                    "max_features":
                        parameters[
                            "max_features"
                        ],

                    "bootstrap":
                        parameters[
                            "bootstrap"
                        ],
                }
            )


        candidate_manifest_df = pd.DataFrame(
            candidate_rows
        )


        assert len(
            candidate_manifest_df
        ) == 100


        assert (
            candidate_manifest_df[
                "candidate_id"
            ]
            .nunique()
            == 100
        )


        efficientnetb0_rf_save_dataframe_atomic(
            candidate_manifest_df,
            candidate_manifest_path,
        )


        print(
            "Created and saved new candidate manifest."
        )


    candidate_manifest_df[
        "candidate_id"
    ] = (
        candidate_manifest_df[
            "candidate_id"
        ]
        .astype(int)
    )


    assert (
        candidate_manifest_df[
            "candidate_id"
        ]
        .tolist()
        == list(
            range(
                1,
                101,
            )
        )
    )


    return candidate_manifest_df


# ============================================================
# 3. LOAD INNER-FIT CHECKPOINTS
# ============================================================

def load_efficientnetb0_rf_inner_progress(
    progress_path,
):

    progress_columns = [
        "repeat",
        "outer_fold",
        "stage",
        "n_estimators",
        "candidate_id",
        "inner_fold",
        "macro_f1",
        "fit_seconds",
    ]


    if progress_path.exists():

        progress_df = pd.read_csv(
            progress_path
        )


        print(
            "Loaded existing checkpoint progress:",
            len(
                progress_df
            ),
            "completed inner fits",
        )


    else:

        progress_df = pd.DataFrame(
            columns=
                progress_columns
        )


        print(
            "No previous inner-fit checkpoint found."
        )


    if len(
        progress_df
    ) > 0:

        assert set(
            progress_columns
        ).issubset(
            progress_df.columns
        )


        integer_columns = [
            "repeat",
            "outer_fold",
            "stage",
            "n_estimators",
            "candidate_id",
            "inner_fold",
        ]


        for column_name in (
            integer_columns
        ):

            progress_df[
                column_name
            ] = (
                pd.to_numeric(
                    progress_df[
                        column_name
                    ]
                )
                .astype(int)
            )


        progress_df[
            "macro_f1"
        ] = pd.to_numeric(
            progress_df[
                "macro_f1"
            ]
        )


        progress_df[
            "fit_seconds"
        ] = pd.to_numeric(
            progress_df[
                "fit_seconds"
            ]
        )


        assert not (
            progress_df
            .duplicated(
                subset=[
                    "stage",
                    "candidate_id",
                    "inner_fold",
                ]
            )
            .any()
        )


    return progress_df


# ============================================================
# 4. SAVE ONE INDIVIDUAL INNER FIT
# ============================================================

def checkpoint_efficientnetb0_rf_inner_fit(
    progress_df,
    new_record,
    progress_path,
):

    new_row_df = pd.DataFrame(
        [
            new_record
        ]
    )


    updated_progress_df = pd.concat(
        [
            progress_df,
            new_row_df,
        ],
        ignore_index=True,
    )


    assert not (
        updated_progress_df
        .duplicated(
            subset=[
                "stage",
                "candidate_id",
                "inner_fold",
            ]
        )
        .any()
    )


    efficientnetb0_rf_save_dataframe_atomic(
        updated_progress_df,
        progress_path,
    )


    return updated_progress_df


# ============================================================
# 5. CHECK WHETHER AN INNER FIT IS ALREADY COMPLETE
# ============================================================

def efficientnetb0_rf_inner_fit_exists(
    progress_df,
    stage_number,
    candidate_id,
    inner_fold_number,
):

    if len(
        progress_df
    ) == 0:

        return False


    return bool(
        (
            (
                progress_df[
                    "stage"
                ]
                == int(
                    stage_number
                )
            )
            &
            (
                progress_df[
                    "candidate_id"
                ]
                == int(
                    candidate_id
                )
            )
            &
            (
                progress_df[
                    "inner_fold"
                ]
                == int(
                    inner_fold_number
                )
            )
        )
        .any()
    )


# ============================================================
# 6. BUILD ONE COMPLETED STAGE RESULT
# ============================================================

def build_efficientnetb0_rf_stage_results(
    progress_df,
    candidate_manifest_df,
    active_candidate_ids,
    stage_number,
    number_of_trees,
):

    stage_progress_df = (
        progress_df[
            (
                progress_df[
                    "stage"
                ]
                == stage_number
            )
            &
            (
                progress_df[
                    "candidate_id"
                ]
                .astype(int)
                .isin(
                    active_candidate_ids
                )
            )
        ]
        .copy()
    )


    expected_stage_fits = (
        len(
            active_candidate_ids
        )
        * 3
    )


    assert (
        len(
            stage_progress_df
        )
        == expected_stage_fits
    ), (
        f"Stage {stage_number} incomplete. "
        f"Expected {expected_stage_fits} fits, "
        f"found {len(stage_progress_df)}."
    )


    candidate_fold_counts = (
        stage_progress_df
        .groupby(
            "candidate_id"
        )
        .size()
    )


    assert (
        candidate_fold_counts
        .eq(3)
        .all()
    )


    stage_results_df = (
        stage_progress_df
        .groupby(
            "candidate_id",
            as_index=False,
        )
        .agg(

            mean_test_score=(
                "macro_f1",
                "mean",
            ),

            std_test_score=(
                "macro_f1",
                "std",
            ),

            total_fit_seconds=(
                "fit_seconds",
                "sum",
            ),

            inner_folds_completed=(
                "inner_fold",
                "count",
            ),
        )
    )


    stage_results_df = (
        stage_results_df
        .merge(
            candidate_manifest_df,
            on="candidate_id",
            how="left",
            validate="one_to_one",
        )
    )


    stage_results_df = (
        stage_results_df
        .sort_values(
            by=[
                "mean_test_score",
                "candidate_id",
            ],
            ascending=[
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


    stage_results_df.insert(
        0,
        "rank_test_score",
        np.arange(
            1,
            len(
                stage_results_df
            )
            + 1,
        ),
    )


    stage_results_df.insert(
        0,
        "n_resources",
        int(
            number_of_trees
        ),
    )


    stage_results_df.insert(
        0,
        "iter",
        int(
            stage_number
            - 1
        ),
    )


    return stage_results_df


# ============================================================
# 7. RUN / RESUME ONE OUTER FOLD
# ============================================================

def run_checkpointed_efficientnetb0_rf_outer_fold(
    repeat_number,
    fold_number,
):

    paths = (
        get_checkpointed_efficientnetb0_rf_fold_paths(
            repeat_number,
            fold_number,
        )
    )


    fold_directory = (
        paths[
            "directory"
        ]
    )


    search_progress_directory = (
        paths[
            "search_progress_directory"
        ]
    )


    fold_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    search_progress_directory.mkdir(
        parents=True,
        exist_ok=True,
    )


    # ========================================================
    # 7A. COMPLETED OUTER FOLD
    # ========================================================

    if (
        is_checkpointed_efficientnetb0_rf_fold_complete(
            repeat_number,
            fold_number,
        )
    ):

        print(
            "\n"
            f"Repeat {repeat_number:02d} / "
            f"Fold {fold_number:02d} "
            "already COMPLETE -> SKIPPED"
        )


        return "skipped"


    # ========================================================
    # 7B. SHARED OUTER PARTITION
    # ========================================================

    partition = (
        make_hybrid_outer_partition(
            repeat_number=
                repeat_number,

            fold_number=
                fold_number,
        )
    )


    X_outer_train = (
        partition[
            "outer_training_features"
        ]
    )


    y_outer_train = np.asarray(
        partition[
            "outer_training_labels"
        ],
        dtype=int,
    )


    X_outer_validation = (
        partition[
            "outer_validation_features"
        ]
    )


    y_outer_validation = np.asarray(
        partition[
            "outer_validation_labels"
        ],
        dtype=int,
    )


    outer_validation_paths = np.asarray(
        partition[
            "outer_validation_paths"
        ],
        dtype=str,
    )


    assert X_outer_train.shape == (
        4480,
        1280,
    )


    assert X_outer_validation.shape == (
        1120,
        1280,
    )


    assert len(
        y_outer_train
    ) == 4480


    assert len(
        y_outer_validation
    ) == 1120


    assert len(
        outer_validation_paths
    ) == 1120


    assert (
        len(
            np.unique(
                outer_validation_paths
            )
        )
        == 1120
    )


    train_counts = np.bincount(
        y_outer_train,
        minlength=4,
    )


    validation_counts = np.bincount(
        y_outer_validation,
        minlength=4,
    )


    assert np.array_equal(
        train_counts,
        np.array(
            [
                1120,
                1120,
                1120,
                1120,
            ]
        ),
    )


    assert np.array_equal(
        validation_counts,
        np.array(
            [
                280,
                280,
                280,
                280,
            ]
        ),
    )


    print(
        "\n"
        + "=" * 88
    )


    print(
        "EFFICIENTNETB0 FIXED FEATURES + RF"
    )


    print(
        f"REPEAT {repeat_number:02d} / "
        f"FOLD {fold_number:02d}"
    )


    print(
        "=" * 88
    )


    print(
        "Outer training:",
        X_outer_train.shape,
    )


    print(
        "Outer validation:",
        X_outer_validation.shape,
    )


    print(
        "Split seed:",
        partition[
            "split_seed"
        ],
    )


    print(
        "Inner-CV seed:",
        get_hybrid_inner_cv_seed(
            repeat_number,
            fold_number,
        ),
    )


    print(
        "RF model seed:",
        get_efficientnetb0_rf_model_seed(
            repeat_number,
            fold_number,
        ),
    )


    print(
        "RF search seed:",
        get_efficientnetb0_rf_search_seed(
            repeat_number,
            fold_number,
        ),
    )


    # ========================================================
    # 7C. INNER SEARCH
    # ========================================================

    candidate_manifest_path = (
        paths[
            "candidate_manifest"
        ]
    )


    progress_path = (
        paths[
            "inner_fit_progress"
        ]
    )


    search_complete_path = (
        paths[
            "search_complete"
        ]
    )


    # --------------------------------------------------------
    # Search already complete
    # --------------------------------------------------------

    if search_complete_path.exists():

        with open(
            search_complete_path,
            "r",
            encoding="utf-8",
        ) as file:

            search_summary = json.load(
                file
            )


        assert (
            search_summary[
                "status"
            ]
            == "completed"
        )


        assert (
            int(
                search_summary[
                    "repeat"
                ]
            )
            == repeat_number
        )


        assert (
            int(
                search_summary[
                    "outer_fold"
                ]
            )
            == fold_number
        )


        print(
            "\nInner RF search already COMPLETE "
            "-> loaded from disk."
        )


    # --------------------------------------------------------
    # Search new / partially complete
    # --------------------------------------------------------

    else:

        candidate_manifest_df = (
            get_or_create_efficientnetb0_rf_candidate_manifest(
                repeat_number,
                fold_number,
                candidate_manifest_path,
            )
        )


        progress_df = (
            load_efficientnetb0_rf_inner_progress(
                progress_path
            )
        )


        if len(
            progress_df
        ) > 0:

            assert (
                progress_df[
                    "repeat"
                ]
                .eq(
                    repeat_number
                )
                .all()
            )


            assert (
                progress_df[
                    "outer_fold"
                ]
                .eq(
                    fold_number
                )
                .all()
            )


        inner_cv = (
            make_checkpointed_efficientnetb0_rf_inner_cv(
                repeat_number,
                fold_number,
            )
        )


        inner_splits = list(
            inner_cv.split(
                X_outer_train,
                y_outer_train,
            )
        )


        assert len(
            inner_splits
        ) == 3


        active_candidate_ids = (
            candidate_manifest_df[
                "candidate_id"
            ]
            .astype(int)
            .tolist()
        )


        model_seed = (
            get_efficientnetb0_rf_model_seed(
                repeat_number,
                fold_number,
            )
        )


        all_stage_results = []


        # ====================================================
        # SUCCESSIVE-HALVING STAGES
        # ====================================================

        for (
            stage_number,
            number_of_trees,
        ) in enumerate(
            EFFICIENTNETB0_RF_STAGE_RESOURCES,
            start=1,
        ):

            expected_candidates = (
                EFFICIENTNETB0_RF_EXPECTED_STAGE_CANDIDATES[
                    stage_number - 1
                ]
            )


            assert (
                len(
                    active_candidate_ids
                )
                == expected_candidates
            )


            print(
                "\n"
                + "-" * 88
            )


            print(
                f"STAGE {stage_number} / 4"
            )


            print(
                "-" * 88
            )


            print(
                "Trees:",
                number_of_trees,
            )


            print(
                "Candidates entering:",
                len(
                    active_candidate_ids
                ),
            )


            # =================================================
            # CANDIDATES
            # =================================================

            for (
                candidate_position,
                candidate_id,
            ) in enumerate(
                active_candidate_ids,
                start=1,
            ):

                candidate_row = (
                    candidate_manifest_df[
                        candidate_manifest_df[
                            "candidate_id"
                        ]
                        == int(
                            candidate_id
                        )
                    ]
                    .iloc[0]
                )


                candidate_parameters = (
                    efficientnetb0_rf_candidate_parameters_from_row(
                        candidate_row
                    )
                )


                # =============================================
                # THREE INNER FOLDS
                # =============================================

                for (
                    inner_fold_number,
                    (
                        inner_training_indices,
                        inner_validation_indices,
                    ),
                ) in enumerate(
                    inner_splits,
                    start=1,
                ):

                    if (
                        efficientnetb0_rf_inner_fit_exists(
                            progress_df,
                            stage_number,
                            candidate_id,
                            inner_fold_number,
                        )
                    ):

                        print(
                            f"Stage {stage_number} | "
                            f"Candidate "
                            f"{candidate_position}/"
                            f"{len(active_candidate_ids)} | "
                            f"ID {candidate_id:03d} | "
                            f"Inner fold "
                            f"{inner_fold_number}/3 "
                            "-> SKIPPED"
                        )


                        continue


                    model = (
                        RandomForestClassifier(

                            **candidate_parameters,

                            n_estimators=
                                int(
                                    number_of_trees
                                ),

                            random_state=
                                model_seed,

                            n_jobs=-1,
                        )
                    )


                    fit_start_time = (
                        time.perf_counter()
                    )


                    model.fit(

                        X_outer_train[
                            inner_training_indices
                        ],

                        y_outer_train[
                            inner_training_indices
                        ],
                    )


                    inner_predictions = (
                        model.predict(

                            X_outer_train[
                                inner_validation_indices
                            ]
                        )
                    )


                    fit_seconds = (
                        time.perf_counter()
                        - fit_start_time
                    )


                    inner_macro_f1 = float(
                        f1_score(

                            y_outer_train[
                                inner_validation_indices
                            ],

                            inner_predictions,

                            average="macro",
                        )
                    )


                    assert np.isfinite(
                        inner_macro_f1
                    )


                    # =========================================
                    # IMMEDIATE CHECKPOINT
                    # =========================================

                    progress_df = (
                        checkpoint_efficientnetb0_rf_inner_fit(

                            progress_df,

                            {
                                "repeat":
                                    int(
                                        repeat_number
                                    ),

                                "outer_fold":
                                    int(
                                        fold_number
                                    ),

                                "stage":
                                    int(
                                        stage_number
                                    ),

                                "n_estimators":
                                    int(
                                        number_of_trees
                                    ),

                                "candidate_id":
                                    int(
                                        candidate_id
                                    ),

                                "inner_fold":
                                    int(
                                        inner_fold_number
                                    ),

                                "macro_f1":
                                    float(
                                        inner_macro_f1
                                    ),

                                "fit_seconds":
                                    float(
                                        fit_seconds
                                    ),
                            },

                            progress_path,
                        )
                    )


                    print(
                        f"Stage {stage_number} | "
                        f"Candidate "
                        f"{candidate_position}/"
                        f"{len(active_candidate_ids)} | "
                        f"ID {candidate_id:03d} | "
                        f"Inner fold "
                        f"{inner_fold_number}/3 | "
                        f"F1={inner_macro_f1:.6f} | "
                        f"{fit_seconds / 60:.2f} min | "
                        "CHECKPOINT SAVED"
                    )


                    del model
                    del inner_predictions

                    gc.collect()


            # =================================================
            # COMPLETE STAGE
            # =================================================

            stage_results_df = (
                build_efficientnetb0_rf_stage_results(

                    progress_df,
                    candidate_manifest_df,
                    active_candidate_ids,

                    stage_number,
                    number_of_trees,
                )
            )


            stage_results_path = (
                search_progress_directory
                / (
                    f"stage_{stage_number:02d}"
                    "_results.csv"
                )
            )


            efficientnetb0_rf_save_dataframe_atomic(
                stage_results_df,
                stage_results_path,
            )


            all_stage_results.append(
                stage_results_df.copy()
            )


            if stage_number < 4:

                number_to_keep = (
                    math.ceil(
                        len(
                            active_candidate_ids
                        )
                        /
                        EFFICIENTNETB0_RF_HALVING_FACTOR
                    )
                )


            else:

                number_to_keep = 1


            selected_candidate_ids = (
                stage_results_df[
                    "candidate_id"
                ]
                .head(
                    number_to_keep
                )
                .astype(int)
                .tolist()
            )


            selected_candidates_df = (
                stage_results_df[
                    stage_results_df[
                        "candidate_id"
                    ]
                    .isin(
                        selected_candidate_ids
                    )
                ]
                .copy()
            )


            selected_path = (
                search_progress_directory
                / (
                    f"stage_{stage_number:02d}"
                    "_selected_candidates.csv"
                )
            )


            efficientnetb0_rf_save_dataframe_atomic(
                selected_candidates_df,
                selected_path,
            )


            print(
                "\nStage completed and saved."
            )


            print(
                "Best stage macro-F1:",
                round(
                    float(
                        stage_results_df[
                            "mean_test_score"
                        ]
                        .iloc[0]
                    ),
                    6,
                ),
            )


            print(
                "Candidates advancing:",
                number_to_keep,
            )


            print(
                "Advancing candidate IDs:",
                selected_candidate_ids,
            )


            active_candidate_ids = (
                selected_candidate_ids
            )


        # ====================================================
        # SAVE INNER SEARCH WINNER
        # ====================================================

        assert len(
            all_stage_results
        ) == 4


        final_stage_results_df = (
            all_stage_results[
                -1
            ]
            .copy()
        )


        assert len(
            final_stage_results_df
        ) == 4


        winner_row = (
            final_stage_results_df
            .iloc[0]
        )


        winning_candidate_id = int(
            winner_row[
                "candidate_id"
            ]
        )


        winning_candidate_row = (
            candidate_manifest_df[
                candidate_manifest_df[
                    "candidate_id"
                ]
                == winning_candidate_id
            ]
            .iloc[0]
        )


        winning_parameters = (
            efficientnetb0_rf_candidate_parameters_from_row(
                winning_candidate_row
            )
        )


        best_inner_macro_f1 = float(
            winner_row[
                "mean_test_score"
            ]
        )


        selected_n_estimators = 675


        complete_search_results_df = pd.concat(
            all_stage_results,
            ignore_index=True,
        )


        assert len(
            complete_search_results_df
        ) == 150


        efficientnetb0_rf_save_dataframe_atomic(

            complete_search_results_df,

            paths[
                "search_results"
            ],
        )


        search_summary = {

            "status":
                "completed",

            "model":
                "EfficientNetB0_fixed_features_RF",

            "repeat":
                int(
                    repeat_number
                ),

            "outer_fold":
                int(
                    fold_number
                ),

            "winning_candidate_id":
                int(
                    winning_candidate_id
                ),

            "winning_parameters":
                winning_parameters,

            "selected_n_estimators":
                int(
                    selected_n_estimators
                ),

            "best_inner_macro_f1":
                float(
                    best_inner_macro_f1
                ),

            "inner_cv_seed":
                int(
                    get_hybrid_inner_cv_seed(
                        repeat_number,
                        fold_number,
                    )
                ),

            "model_seed":
                int(
                    get_efficientnetb0_rf_model_seed(
                        repeat_number,
                        fold_number,
                    )
                ),

            "search_seed":
                int(
                    get_efficientnetb0_rf_search_seed(
                        repeat_number,
                        fold_number,
                    )
                ),

            "required_inner_fits":
                int(
                    150
                    * 3
                ),
        }


        efficientnetb0_rf_save_json_atomic(
            search_summary,
            search_complete_path,
        )


        print(
            "\n"
            + "=" * 88
        )


        print(
            "INNER SEARCH COMPLETE AND CHECKPOINTED"
        )


        print(
            "=" * 88
        )


        print(
            "Winning candidate:",
            winning_candidate_id,
        )


        print(
            "Winning parameters:",
            winning_parameters,
        )


        print(
            "Selected trees:",
            selected_n_estimators,
        )


        print(
            "Best inner macro-F1:",
            round(
                best_inner_macro_f1,
                6,
            ),
        )


    # ========================================================
    # 7D. LOAD WINNER
    # ========================================================

    winning_candidate_id = int(
        search_summary[
            "winning_candidate_id"
        ]
    )


    winning_parameters = (
        search_summary[
            "winning_parameters"
        ]
    )


    selected_n_estimators = int(
        search_summary[
            "selected_n_estimators"
        ]
    )


    best_inner_macro_f1 = float(
        search_summary[
            "best_inner_macro_f1"
        ]
    )


    assert (
        selected_n_estimators
        == 675
    )


    # ========================================================
    # 7E. FINAL REFIT ON ALL 4,480 OUTER-TRAINING SAMPLES
    # ========================================================

    print(
        "\nRefitting winning RF on all "
        "4,480 outer-training samples..."
    )


    final_refit_start = (
        time.perf_counter()
    )


    final_model = (
        RandomForestClassifier(

            **winning_parameters,

            n_estimators=
                selected_n_estimators,

            random_state=
                get_efficientnetb0_rf_model_seed(
                    repeat_number,
                    fold_number,
                ),

            n_jobs=-1,
        )
    )


    final_model.fit(
        X_outer_train,
        y_outer_train,
    )


    final_refit_seconds = (
        time.perf_counter()
        - final_refit_start
    )


    # ========================================================
    # 7F. OUTER VALIDATION — ONCE
    # ========================================================

    prediction_start = (
        time.perf_counter()
    )


    outer_predictions = (
        final_model.predict(
            X_outer_validation
        )
    )


    prediction_seconds = (
        time.perf_counter()
        - prediction_start
    )


    outer_predictions = np.asarray(
        outer_predictions,
        dtype=int,
    )


    assert len(
        outer_predictions
    ) == 1120


    assert set(
        np.unique(
            outer_predictions
        )
    ).issubset(
        {
            0,
            1,
            2,
            3,
        }
    )


    # ========================================================
    # 7G. OUTER METRICS
    # ========================================================

    outer_accuracy = float(
        accuracy_score(
            y_outer_validation,
            outer_predictions,
        )
    )


    outer_balanced_accuracy = float(
        balanced_accuracy_score(
            y_outer_validation,
            outer_predictions,
        )
    )


    outer_macro_precision = float(
        precision_score(
            y_outer_validation,
            outer_predictions,
            average="macro",
            zero_division=0,
        )
    )


    outer_macro_recall = float(
        recall_score(
            y_outer_validation,
            outer_predictions,
            average="macro",
            zero_division=0,
        )
    )


    outer_macro_f1 = float(
        f1_score(
            y_outer_validation,
            outer_predictions,
            average="macro",
        )
    )


    outer_confusion_matrix = confusion_matrix(
        y_outer_validation,
        outer_predictions,
        labels=[
            0,
            1,
            2,
            3,
        ],
    )


    outer_classification_report = (
        classification_report(
            y_outer_validation,
            outer_predictions,
            labels=[
                0,
                1,
                2,
                3,
            ],
            target_names=
                CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        )
    )


    # ========================================================
    # 7H. PREDICTION TABLE
    # ========================================================

    predictions_df = pd.DataFrame(
        {
            "relative_path":
                outer_validation_paths,

            "true_label":
                y_outer_validation,

            "predicted_label":
                outer_predictions,
        }
    )


    predictions_df[
        "true_class"
    ] = (
        predictions_df[
            "true_label"
        ]
        .map(
            INDEX_TO_CLASS
        )
    )


    predictions_df[
        "predicted_class"
    ] = (
        predictions_df[
            "predicted_label"
        ]
        .map(
            INDEX_TO_CLASS
        )
    )


    assert len(
        predictions_df
    ) == 1120


    assert (
        predictions_df[
            "relative_path"
        ]
        .nunique()
        == 1120
    )


    # ========================================================
    # 7I. TOTAL SAVED INNER FIT TIME
    # ========================================================

    saved_progress_df = pd.read_csv(
        progress_path
    )


    cumulative_inner_fit_seconds = float(
        saved_progress_df[
            "fit_seconds"
        ]
        .sum()
    )


    # ========================================================
    # 7J. SELECTED PARAMETERS
    # ========================================================

    selected_parameters = {

        "model":
            "EfficientNetB0_fixed_features_RF",

        "feature_dimension":
            1280,

        "repeat":
            int(
                repeat_number
            ),

        "fold":
            int(
                fold_number
            ),

        "outer_split_seed":
            int(
                partition[
                    "split_seed"
                ]
            ),

        "inner_cv_seed":
            int(
                get_hybrid_inner_cv_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "rf_model_seed":
            int(
                get_efficientnetb0_rf_model_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "rf_search_seed":
            int(
                get_efficientnetb0_rf_search_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "inner_cv_folds":
            3,

        "selection_metric":
            "f1_macro",

        "search_method":
            "manually_checkpointed_successive_halving",

        "initial_candidates":
            100,

        "halving_factor":
            3,

        "resource":
            "n_estimators",

        "tree_resources":
            [
                25,
                75,
                225,
                675,
            ],

        "candidate_counts_by_iteration":
            {
                "0": 100,
                "1": 34,
                "2": 12,
                "3": 4,
            },

        "winning_candidate_id":
            int(
                winning_candidate_id
            ),

        "selected_n_estimators":
            int(
                selected_n_estimators
            ),

        "best_inner_macro_f1":
            float(
                best_inner_macro_f1
            ),

        "best_parameters":
            winning_parameters,
    }


    # ========================================================
    # 7K. OUTER METRICS
    # ========================================================

    outer_metrics = {

        "model":
            "EfficientNetB0_fixed_features_RF",

        "feature_dimension":
            1280,

        "repeat":
            int(
                repeat_number
            ),

        "fold":
            int(
                fold_number
            ),

        "outer_split_seed":
            int(
                partition[
                    "split_seed"
                ]
            ),

        "inner_cv_seed":
            int(
                get_hybrid_inner_cv_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "rf_model_seed":
            int(
                get_efficientnetb0_rf_model_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "rf_search_seed":
            int(
                get_efficientnetb0_rf_search_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "outer_training_samples":
            4480,

        "outer_validation_samples":
            1120,

        "winning_candidate_id":
            int(
                winning_candidate_id
            ),

        "selected_n_estimators":
            int(
                selected_n_estimators
            ),

        "best_inner_macro_f1":
            float(
                best_inner_macro_f1
            ),

        "accuracy":
            float(
                outer_accuracy
            ),

        "balanced_accuracy":
            float(
                outer_balanced_accuracy
            ),

        "macro_precision":
            float(
                outer_macro_precision
            ),

        "macro_recall":
            float(
                outer_macro_recall
            ),

        "macro_f1":
            float(
                outer_macro_f1
            ),

        "confusion_matrix":
            outer_confusion_matrix.tolist(),

        "classification_report":
            outer_classification_report,

        "cumulative_inner_fit_seconds":
            float(
                cumulative_inner_fit_seconds
            ),

        "final_refit_seconds":
            float(
                final_refit_seconds
            ),

        "prediction_seconds":
            float(
                prediction_seconds
            ),
    }


    # ========================================================
    # 7L. SAVE FINAL SUBSTANTIVE OUTPUTS
    # ========================================================

    assert (
        paths[
            "search_results"
        ]
        .exists()
    )


    efficientnetb0_rf_save_json_atomic(
        selected_parameters,
        paths[
            "selected_parameters"
        ],
    )


    efficientnetb0_rf_save_dataframe_atomic(
        predictions_df,
        paths[
            "outer_predictions"
        ],
    )


    efficientnetb0_rf_save_json_atomic(
        outer_metrics,
        paths[
            "outer_metrics"
        ],
    )


    # ========================================================
    # 7M. READ EVERYTHING BACK
    # ========================================================

    saved_search_results = pd.read_csv(
        paths[
            "search_results"
        ]
    )


    saved_predictions = pd.read_csv(
        paths[
            "outer_predictions"
        ]
    )


    with open(
        paths[
            "selected_parameters"
        ],
        "r",
        encoding="utf-8",
    ) as file:

        saved_selected_parameters = json.load(
            file
        )


    with open(
        paths[
            "outer_metrics"
        ],
        "r",
        encoding="utf-8",
    ) as file:

        saved_metrics = json.load(
            file
        )


    assert len(
        saved_search_results
    ) == 150


    assert (
        (
            saved_search_results[
                "iter"
            ]
            .astype(int)
            == 0
        )
        .sum()
        == 100
    )


    assert (
        (
            saved_search_results[
                "iter"
            ]
            .astype(int)
            == 1
        )
        .sum()
        == 34
    )


    assert (
        (
            saved_search_results[
                "iter"
            ]
            .astype(int)
            == 2
        )
        .sum()
        == 12
    )


    assert (
        (
            saved_search_results[
                "iter"
            ]
            .astype(int)
            == 3
        )
        .sum()
        == 4
    )


    assert (
        sorted(
            saved_search_results[
                "n_resources"
            ]
            .astype(int)
            .unique()
            .tolist()
        )
        == [
            25,
            75,
            225,
            675,
        ]
    )


    assert len(
        saved_predictions
    ) == 1120


    assert (
        saved_predictions[
            "relative_path"
        ]
        .nunique()
        == 1120
    )


    assert (
        int(
            saved_selected_parameters[
                "repeat"
            ]
        )
        == repeat_number
    )


    assert (
        int(
            saved_selected_parameters[
                "fold"
            ]
        )
        == fold_number
    )


    assert (
        int(
            saved_metrics[
                "repeat"
            ]
        )
        == repeat_number
    )


    assert (
        int(
            saved_metrics[
                "fold"
            ]
        )
        == fold_number
    )


    assert (
        int(
            saved_metrics[
                "outer_training_samples"
            ]
        )
        == 4480
    )


    assert (
        int(
            saved_metrics[
                "outer_validation_samples"
            ]
        )
        == 1120
    )


    assert np.isclose(
        float(
            saved_metrics[
                "macro_f1"
            ]
        ),
        outer_macro_f1,
        rtol=0.0,
        atol=1e-12,
    )


    # ========================================================
    # 7N. COMPLETED.JSON MUST BE WRITTEN LAST
    # ========================================================

    completion_information = {

        "status":
            "completed",

        "model":
            "EfficientNetB0_fixed_features_RF",

        "repeat":
            int(
                repeat_number
            ),

        "fold":
            int(
                fold_number
            ),

        "outer_split_seed":
            int(
                partition[
                    "split_seed"
                ]
            ),

        "inner_cv_seed":
            int(
                get_hybrid_inner_cv_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "rf_model_seed":
            int(
                get_efficientnetb0_rf_model_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "rf_search_seed":
            int(
                get_efficientnetb0_rf_search_seed(
                    repeat_number,
                    fold_number,
                )
            ),

        "inner_search_completed":
            True,

        "outer_evaluation_completed":
            True,

        "required_outputs_verified":
            True,
    }


    efficientnetb0_rf_save_json_atomic(
        completion_information,
        paths[
            "completion_marker"
        ],
    )


    # ========================================================
    # 7O. FINAL VALIDATION
    # ========================================================

    assert (
        is_checkpointed_efficientnetb0_rf_fold_complete(
            repeat_number,
            fold_number,
        )
    )


    print(
        "\n"
        + "=" * 88
    )


    print(
        f"EFFICIENTNETB0 + RF — REPEAT "
        f"{repeat_number:02d} / "
        f"FOLD {fold_number:02d} COMPLETE"
    )


    print(
        "=" * 88
    )


    print(
        "Winning candidate:",
        winning_candidate_id,
    )


    print(
        "Best inner macro-F1:",
        round(
            best_inner_macro_f1,
            6,
        ),
    )


    print(
        "Outer accuracy:",
        round(
            outer_accuracy,
            6,
        ),
    )


    print(
        "Outer balanced accuracy:",
        round(
            outer_balanced_accuracy,
            6,
        ),
    )


    print(
        "Outer macro precision:",
        round(
            outer_macro_precision,
            6,
        ),
    )


    print(
        "Outer macro recall:",
        round(
            outer_macro_recall,
            6,
        ),
    )


    print(
        "Outer macro F1:",
        round(
            outer_macro_f1,
            6,
        ),
    )


    print(
        "Accumulated inner-fit time:",
        round(
            cumulative_inner_fit_seconds
            / 60,
            2,
        ),
        "minutes",
    )


    print(
        "Final refit time:",
        round(
            final_refit_seconds
            / 60,
            2,
        ),
        "minutes",
    )


    print(
        "COMPLETED.json written LAST."
    )


    del final_model
    del outer_predictions

    del X_outer_train
    del y_outer_train

    del X_outer_validation
    del y_outer_validation

    del partition

    gc.collect()


    return "completed"


# ============================================================
# 8. REFRESH CONSOLIDATED RESULTS
# ============================================================

def refresh_checkpointed_efficientnetb0_rf_summaries():

    fold_records = []


    for repeat_number in range(
        1,
        11,
    ):

        for fold_number in range(
            1,
            6,
        ):

            if not (
                is_checkpointed_efficientnetb0_rf_fold_complete(
                    repeat_number,
                    fold_number,
                )
            ):

                continue


            paths = (
                get_checkpointed_efficientnetb0_rf_fold_paths(
                    repeat_number,
                    fold_number,
                )
            )


            with open(
                paths[
                    "outer_metrics"
                ],
                "r",
                encoding="utf-8",
            ) as file:

                metrics = json.load(
                    file
                )


            fold_records.append(
                {
                    "repeat":
                        int(
                            repeat_number
                        ),

                    "fold":
                        int(
                            fold_number
                        ),

                    "accuracy":
                        float(
                            metrics[
                                "accuracy"
                            ]
                        ),

                    "balanced_accuracy":
                        float(
                            metrics[
                                "balanced_accuracy"
                            ]
                        ),

                    "macro_precision":
                        float(
                            metrics[
                                "macro_precision"
                            ]
                        ),

                    "macro_recall":
                        float(
                            metrics[
                                "macro_recall"
                            ]
                        ),

                    "macro_f1":
                        float(
                            metrics[
                                "macro_f1"
                            ]
                        ),

                    "best_inner_macro_f1":
                        float(
                            metrics[
                                "best_inner_macro_f1"
                            ]
                        ),
                }
            )


    all_fold_results_df = pd.DataFrame(
        fold_records
    )


    if len(
        all_fold_results_df
    ) > 0:

        all_fold_results_df = (
            all_fold_results_df
            .sort_values(
                [
                    "repeat",
                    "fold",
                ]
            )
            .reset_index(
                drop=True
            )
        )


        efficientnetb0_rf_save_dataframe_atomic(

            all_fold_results_df,

            EFFICIENTNETB0_RF_RESULTS_ROOT
            / "all_fold_results.csv",
        )


    repeat_records = []


    if len(
        all_fold_results_df
    ) > 0:

        for repeat_number in range(
            1,
            11,
        ):

            repeat_df = (
                all_fold_results_df[
                    all_fold_results_df[
                        "repeat"
                    ]
                    == repeat_number
                ]
                .copy()
            )


            if len(
                repeat_df
            ) != 5:

                continue


            repeat_records.append(
                {
                    "repeat":
                        int(
                            repeat_number
                        ),

                    "mean_accuracy":
                        float(
                            repeat_df[
                                "accuracy"
                            ]
                            .mean()
                        ),

                    "mean_balanced_accuracy":
                        float(
                            repeat_df[
                                "balanced_accuracy"
                            ]
                            .mean()
                        ),

                    "mean_macro_precision":
                        float(
                            repeat_df[
                                "macro_precision"
                            ]
                            .mean()
                        ),

                    "mean_macro_recall":
                        float(
                            repeat_df[
                                "macro_recall"
                            ]
                            .mean()
                        ),

                    "mean_macro_f1":
                        float(
                            repeat_df[
                                "macro_f1"
                            ]
                            .mean()
                        ),

                    "sd_macro_f1":
                        float(
                            repeat_df[
                                "macro_f1"
                            ]
                            .std(
                                ddof=1
                            )
                        ),

                    "wilcoxon_value":
                        float(
                            repeat_df[
                                "macro_f1"
                            ]
                            .mean()
                        ),
                }
            )


    repeat_summary_df = pd.DataFrame(
        repeat_records
    )


    if len(
        repeat_summary_df
    ) > 0:

        repeat_summary_df = (
            repeat_summary_df
            .sort_values(
                "repeat"
            )
            .reset_index(
                drop=True
            )
        )


        efficientnetb0_rf_save_dataframe_atomic(

            repeat_summary_df,

            EFFICIENTNETB0_RF_RESULTS_ROOT
            / "repeat_level_summary.csv",
        )


    return (
        all_fold_results_df,
        repeat_summary_df,
    )


# ============================================================
# 9. RUN ALL 50 OUTER POSITIONS
# ============================================================

newly_completed_folds = 0

previously_completed_folds = 0


master_start_time = (
    time.perf_counter()
)


for repeat_number in range(
    1,
    11,
):

    print(
        "\n"
        + "#" * 88
    )


    print(
        f"STARTING REPEAT "
        f"{repeat_number} / 10"
    )


    print(
        "#" * 88
    )


    for fold_number in range(
        1,
        6,
    ):

        fold_status = (
            run_checkpointed_efficientnetb0_rf_outer_fold(
                repeat_number,
                fold_number,
            )
        )


        if (
            fold_status
            == "completed"
        ):

            newly_completed_folds += 1


        elif (
            fold_status
            == "skipped"
        ):

            previously_completed_folds += 1


        (
            current_fold_results,
            current_repeat_summary,
        ) = (
            refresh_checkpointed_efficientnetb0_rf_summaries()
        )


        print(
            "\nCURRENT EFFICIENTNETB0 + RF PROGRESS:",
            len(
                current_fold_results
            ),
            "/ 50 outer folds complete",
        )


        print(
            "Completed full repeats:",
            len(
                current_repeat_summary
            ),
            "/ 10",
        )


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

(
    final_rf_fold_results,
    final_rf_repeat_summary,
) = (
    refresh_checkpointed_efficientnetb0_rf_summaries()
)


master_elapsed_seconds = (
    time.perf_counter()
    - master_start_time
)


print(
    "\n"
    + "=" * 88
)


print(
    "EFFICIENTNETB0 FIXED-FEATURE + RF MASTER RUN FINISHED"
)


print(
    "=" * 88
)


print(
    "Completed outer folds:",
    len(
        final_rf_fold_results
    ),
    "/ 50",
)


print(
    "Completed full repeats:",
    len(
        final_rf_repeat_summary
    ),
    "/ 10",
)


print(
    "New outer folds completed this session:",
    newly_completed_folds,
)


print(
    "Previously completed outer folds skipped:",
    previously_completed_folds,
)


print(
    "Current-session elapsed time:",
    round(
        master_elapsed_seconds
        / 60,
        2,
    ),
    "minutes",
)


print(
    "\nResults root:"
)


print(
    EFFICIENTNETB0_RF_RESULTS_ROOT
)


if (
    len(
        final_rf_fold_results
    )
    == 50
):

    assert (
        len(
            final_rf_repeat_summary
        )
        == 10
    )


    print(
        "\nALL 50 EFFICIENTNETB0 + RF "
        "OUTER FOLDS ARE COMPLETE."
    )


    print(
        "\nFold-level summary:"
    )


    print(
        EFFICIENTNETB0_RF_RESULTS_ROOT
        / "all_fold_results.csv"
    )


    print(
        "\nRepeat-level summary:"
    )


    print(
        EFFICIENTNETB0_RF_RESULTS_ROOT
        / "repeat_level_summary.csv"
    )


else:

    print(
        "\nThe experiment stopped before all "
        "50 outer folds completed."
    )


    print(
        "Rerun THIS SAME CELL after reconnecting."
    )


    print(
        "Completed outer folds will be skipped."
    )


    print(
        "Completed candidate × inner-fold fits "
        "inside the current unfinished fold "
        "will also be skipped."
    )


print(
    "\nTesting partition was NEVER accessed."
)


print(
    "\nEFFICIENTNETB0 + RF CHECKPOINTED "
    "MASTER RUNNER: PASSED"
)


########################################################################################
STARTING REPEAT 1 / 10
########################################################################################

Repeat 01 / Fold 01 already COMPLETE -> SKIPPED

CURRENT EFFICIENTNETB0 + RF PROGRESS: 7 / 50 outer folds complete
Completed full repeats: 1 / 10

Repeat 01 / Fold 02 already COMPLETE -> SKIPPED

CURRENT EFFICIENTNETB0 + RF PROGRESS: 7 / 50 outer folds complete
Completed full repeats: 1 / 10

Repeat 01 / Fold 03 already COMPLETE -> SKIPPED

CURRENT EFFICIENTNETB0 + RF PROGRESS: 7 / 50 outer folds complete
Completed full repeats: 1 / 10

Repeat 01 / Fold 04 already COMPLETE -> SKIPPED

CURRENT EFFICIENTNETB0 + RF PROGRESS: 7 / 50 outer folds complete
Completed full repeats: 1 / 10

Repeat 01 / Fold 05 already COMPLETE -> SKIPPED

CURRENT EFFICIENTNETB0 + RF PROGRESS: 7 / 50 outer folds complete
Completed full repeats: 1 / 10

##############################################################

/tmp/ipykernel_960/361614969.py:410: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  updated_progress_df = pd.concat(


Stage 1 | Candidate 1/100 | ID 001 | Inner fold 1/3 | F1=0.872865 | 0.14 min | CHECKPOINT SAVED
Stage 1 | Candidate 1/100 | ID 001 | Inner fold 2/3 | F1=0.856731 | 0.18 min | CHECKPOINT SAVED
Stage 1 | Candidate 1/100 | ID 001 | Inner fold 3/3 | F1=0.854625 | 0.16 min | CHECKPOINT SAVED
Stage 1 | Candidate 2/100 | ID 002 | Inner fold 1/3 | F1=0.873102 | 0.07 min | CHECKPOINT SAVED
Stage 1 | Candidate 2/100 | ID 002 | Inner fold 2/3 | F1=0.841081 | 0.06 min | CHECKPOINT SAVED
Stage 1 | Candidate 2/100 | ID 002 | Inner fold 3/3 | F1=0.845677 | 0.05 min | CHECKPOINT SAVED
Stage 1 | Candidate 3/100 | ID 003 | Inner fold 1/3 | F1=0.870423 | 0.16 min | CHECKPOINT SAVED
Stage 1 | Candidate 3/100 | ID 003 | Inner fold 2/3 | F1=0.868642 | 0.14 min | CHECKPOINT SAVED
Stage 1 | Candidate 3/100 | ID 003 | Inner fold 3/3 | F1=0.876802 | 0.14 min | CHECKPOINT SAVED
Stage 1 | Candidate 4/100 | ID 004 | Inner fold 1/3 | F1=0.853056 | 0.04 min | CHECKPOINT SAVED
Stage 1 | Candidate 4/100 | ID 004 | Inn

In [ ]:
# ============================================================
# HYBRID EXPERIMENT SETUP AND REPRODUCIBILITY
# ============================================================

import os
import random
import numpy as np
import pandas as pd
import tensorflow as tf
import time


# ============================================================
# Reproducibility
# ============================================================

# Set the seed for random number
RANDOM_SEED = 42

# Making experiment runs reproducible
os.environ["PYTHONHASHSEED"] = str(RANDOM_SEED)
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Making determenistic TesnsorFlow operations where supported
try:
  tf.config.experimental.enable_op_determinism()
  determinism_status = "enabled"

except Exception as error:
  determinism_status = ("requested but TensorFlow returned:" f" {error}")

# Keep TensorFlow/Keras calculations in float 32
tf.keras.backend.set_floatx("float32")



# ============================================================
# 2. Fixed experiment configuration
# ============================================================

IMAGE_HEIGHT = 224
IMAGE_WIDTH = 224
IMAGE_CHANNELS = 3

NUMBER_OF_CLASSES = 4

CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

CLASS_TO_INDEX = {
    class_name: index
    for index, class_name in enumerate(CLASS_NAMES)
}

INDEX_TO_CLASS = {
    index: class_name
    for index, class_name in enumerate(CLASS_NAMES)
}



# ============================================================
# 3. Verify experiment configuration
# ============================================================

print("=" * 70)
print("HYBRID EXPERIMENT SETUP")
print("=" * 70)
print("Random seed:", RANDOM_SEED)
print("Deterministic TensorFlow operations:", determinism_status)
print("Keras float type:", tf.keras.backend.floatx())
print("Input shape:", (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))
print("Number of classes:", NUMBER_OF_CLASSES)
print("Class mapping:", CLASS_TO_INDEX)





HYBRID EXPERIMENT SETUP
Random seed: 42
Deterministic TensorFlow operations: enabled
Keras float type: float32
Input shape: (224, 224, 3)
Number of classes: 4
Class mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}


In [ ]:
# ============================================================
# LOAD AND VALIDATE THE FIXED FIVE-FOLD ASSIGNMENTS
# ============================================================

import pandas as pd

# Load the fixed five-fold Training assignments
assignments = pd.read_csv(FOLDS_FILE)


# ============================================================
# 1. Validate the table structure
# ============================================================

# Define the columns that must exist in the fold-assignment file
required_columns = {"relative_path", "class", "fold"}


# Find any required columns that are missing
missing_columns = (required_columns - set(assignments.columns))


# Stop if any required column is missing
if missing_columns:
    raise ValueError("The fold file is missing columns: "f"{sorted(missing_columns)}")



# ============================================================
# 2. Create class indices and complete image paths
# ============================================================

# Assign the fixed numeric class index to every Training image
assignments["class_index"] = (assignments["class"].map(CLASS_TO_INDEX))


# Create the complete path to every Training image
assignments["image_path"] = (assignments["relative_path"].apply(
        lambda path:
        DATA_DIR
        / Path(path)))


# ============================================================
# 3. Validate paths and assignments
# ============================================================

# Find any Training images that do not exist
missing_image_paths = [
    image_path
    for image_path
    in assignments["image_path"]
    if not image_path.exists()
]


# Find any class names that do not belong to the four expected classes
unexpected_classes = sorted(set(assignments["class"]) - set(CLASS_NAMES))


# Find any fold values outside the fixed folds 1 to 5
unexpected_folds = sorted(set(assignments["fold"]) - {1, 2, 3, 4, 5})


# Count duplicate Training image paths
duplicate_paths = (assignments["relative_path"].duplicated().sum())


# Count missing values in the required columns
missing_values = (
    assignments[["relative_path", "class", "fold"]].isna().sum())


# Check whether any class failed to receive a numeric class index
missing_class_indices = (assignments["class_index"].isna().sum())


# ============================================================
# 4. Verify the fixed fold distribution
# ============================================================

# Count how many images from every class belong to each outer fold
fold_distribution = pd.crosstab(assignments["fold"], assignments["class"])


# Put the class columns into the fixed project class order
fold_distribution = (fold_distribution.reindex(
        columns=CLASS_NAMES,
        fill_value=0))


# Add the total number of images in each fold
fold_distribution["total"] = (fold_distribution.sum(axis=1))


# Each of the five fixed folds must contain 280 images from every class
expected_fold_distribution = pd.DataFrame(
    {
        class_name: [280] * 5
        for class_name in CLASS_NAMES
    },
    index=[1, 2, 3, 4, 5])


expected_fold_distribution["total"] = 1120


# ============================================================
# 5. Display the validation results
# ============================================================

print("=" * 70)
print("FIXED FIVE-FOLD ASSIGNMENT VALIDATION")
print("=" * 70)
print("Rows:", len(assignments))
print("Columns:", assignments.columns.tolist())
print("\n--- Assignment checks ---")
print("Duplicate relative paths:", duplicate_paths)
print("Missing image files:", len(missing_image_paths))
print("Unexpected classes:", unexpected_classes)
print("Unexpected fold values:", unexpected_folds)
print("Missing class indices:", missing_class_indices)
print("\nMissing values:")
print(missing_values)
print("\n--- Images per fold and class ---")
display(fold_distribution)
print("\n--- Sample records ---")
display(
    assignments[["relative_path", "class", "class_index", "fold", "image_path"]].head())


# ============================================================
# 6. Stop if any validation check failed
# ============================================================

# The fold file must contain exactly 5,600 Training images
if len(assignments) != 5600:
    raise ValueError("Expected 5,600 Training images, "f"but found {len(assignments)}.")


# Every Training image path must be unique
if duplicate_paths != 0:
    raise ValueError("Duplicate paths were found in the fold file.")


# Every Training image listed in the CSV must exist
if missing_image_paths:
    raise FileNotFoundError(f"{len(missing_image_paths)} Training image files are missing.")


# Only the expected four classes are allowed
if unexpected_classes:
    raise ValueError(f"Unexpected classes found: {unexpected_classes}")


# Only folds 1 to 5 are allowed
if unexpected_folds:
    raise ValueError(f"Unexpected fold values found: {unexpected_folds}")


# Required columns must not contain missing values
if missing_values.sum() != 0:
    raise ValueError("Missing values were found in the fold file.")


# Every class must map successfully to its fixed numeric index
if missing_class_indices != 0:
    raise ValueError(
        "One or more Training classes could not be mapped to a class index.")


# Verify the exact fixed five-fold class distribution
if not fold_distribution.equals(expected_fold_distribution):
    raise ValueError(
        "The fixed five-fold class distribution does not match the expected "
        "280 images per class and 1,120 images per fold.")

print("\nFixed five-fold assignment validation passed.")

FIXED FIVE-FOLD ASSIGNMENT VALIDATION
Rows: 5600
Columns: ['relative_path', 'class', 'fold', 'class_index', 'image_path']

--- Assignment checks ---
Duplicate relative paths: 0
Missing image files: 0
Unexpected classes: []
Unexpected fold values: []
Missing class indices: 0

Missing values:
relative_path    0
class            0
fold             0
dtype: int64

--- Images per fold and class ---


class,glioma,meningioma,notumor,pituitary,total
fold,,,,,
1,280,280,280,280,1120
2,280,280,280,280,1120
3,280,280,280,280,1120
4,280,280,280,280,1120
5,280,280,280,280,1120



--- Sample records ---


,relative_path,class,class_index,fold,image_path
0,Training/glioma/Tr-gl_100.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
1,Training/glioma/Tr-gl_1001.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
2,Training/glioma/Tr-gl_1003.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
3,Training/glioma/Tr-gl_1014.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...
4,Training/glioma/Tr-gl_1015.png,glioma,0,1,/content/brain-tumour-mri-classification/proce...



Fixed five-fold assignment validation passed.


In [ ]:
# ============================================================
# CREATING THE FIXED IMAGENET EFFICIENTNETB0 FEATURE EXTRACTOR
# ============================================================

# Create a new ImageNet pretrained EfficientNetB0 without its original classification layer
efficientnetb0_feature_extractor = tf.keras.applications.EfficientNetB0(
    include_top=False,
    weights="imagenet",
    pooling="avg",
    input_shape=(IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))

# Freeze every EfficientNetB0 layer
efficientnetb0_feature_extractor.trainable = False

# Record the feature dimension produced by the model
FEATURE_DIMENSION = int(efficientnetb0_feature_extractor.output.shape[1])



# ============================================================
# VERIFYing THE FEATURE EXTRACTOR
# ============================================================

print("=" * 70)
print("FIXED IMAGENET EFFICIENTNETB0 FEATURE EXTRACTOR")
print("=" * 70)
print("Model name:", efficientnetb0_feature_extractor.name)
print("Input shape:", efficientnetb0_feature_extractor.input_shape)
print("Output shape:", efficientnetb0_feature_extractor.output_shape)
print("Features per image:", FEATURE_DIMENSION)
print("Trainable:", efficientnetb0_feature_extractor.trainable)


# No EfficientNetB0 weights may be trainable
if efficientnetb0_feature_extractor.trainable_weights:
    raise RuntimeError("The EfficientNetB0 feature extractor contains trainable weights.")

print("\nFixed ImageNet EfficientNetB0 feature extractor verified successfully.")


16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
FIXED IMAGENET EFFICIENTNETB0 FEATURE EXTRACTOR
Model name: efficientnetb0
Input shape: (None, 224, 224, 3)
Output shape: (None, 1280)
Features per image: 1280
Trainable: False

Fixed ImageNet EfficientNetB0 feature extractor verified successfully.


In [ ]:
# ============================================================
# DEFINING EFFICIENTNETB0 IMAGE PREPROCESSING FOR FEATURE EXTRACTION
# ============================================================

def load_and_preprocess_efficientnetb0_image(image_path, class_index):
  """
  Load one grayscale MRI image, convert it to pseudo-RGB and keep pixel values as float32 in the range 0-255
  """

  # Reading the PNG image file
  image_bytes = tf.io.read_file(image_path)

  # Decoding as a single-channel grayscale image
  grayscale_image = tf.io.decode_png(image_bytes, channels = 1)

  # Confirming the expected grayscale image shape
  grayscale_image = tf.ensure_shape(grayscale_image, (IMAGE_HEIGHT, IMAGE_WIDTH, 1))

  # Repeat the grayscale channel three times
  pseudo_rgb_image = tf.image.grayscale_to_rgb(grayscale_image)

  # Convert pixel values to float32
  efficientnet_image = tf.cast(pseudo_rgb_image, tf.float32)

  # Confirm the final EfficientNetB0 input shape
  efficientnet_image = tf.ensure_shape(efficientnet_image, (IMAGE_HEIGHT, IMAGE_WIDTH, IMAGE_CHANNELS))

  return efficientnet_image, class_index


In [ ]:
# ============================================================
# CREATE THE EFFICIENTNETB0 FEATURE-EXTRACTION DATASET
# ============================================================

def create_efficientnetb0_feature_dataset(image_paths, class_indices, batch_size):
    """
    Create a deterministic TensorFlow dataset for fixed EfficientNetB0 feature extraction without shuffling.
    """

    # Creating the dataset from image paths and class indices
    dataset = tf.data.Dataset.from_tensor_slices((image_paths, class_indices))

    # Keeping dataset processing deterministic
    dataset_options = tf.data.Options()
    dataset_options.experimental_deterministic = True

    dataset = dataset.with_options(dataset_options)

    # Loading and preprocessing each MRI image
    dataset = dataset.map(load_and_preprocess_efficientnetb0_image, num_parallel_calls=tf.data.AUTOTUNE, deterministic=True)

    # Grouping images into batches without dropping the final batch
    dataset = dataset.batch(batch_size, drop_remainder=False)

    # Preparing upcoming batches in advance
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset

In [ ]:
# ============================================================
# DEFINE THE EFFICIENTNETB0 FEATURE-EXTRACTION FUNCTION
# ============================================================

def extract_efficientnetb0_features(dataset, feature_extractor):
  """
  Extracts fixed EfficientNetB0features and preseerve the corresponding class indices in dataset order.
  """

  feature_batches = []
  class_index_batches = []

  # Process the dataset one batch at a time
  for efficientnet_image_batch, class_index_batch in dataset:

    # Extract fixed ImageNet EfficientNetB0 features
    feature_batch = feature_extractor(efficientnet_image_batch, training=False)

    # Store the features and corresponding class indices
    feature_batches.append(feature_batch.numpy())
    class_index_batches.append(class_index_batch.numpy())


  # Combining all batches into complete NumPy arrays
  features = np.concatenate(feature_batches, axis = 0).astype(np.float32)

  class_indices = np.concatenate(class_index_batches, axis = 0).astype(np.int32)

  return features, class_indices


In [ ]:
from numpy._core.multiarray import dtype

# ============================================================
# PREPARE THE TRAINING DATA FOR FEATURE EXTRACTION
# ============================================================

# Storing the Training image paths in fold-assignment row order
training_image_paths = assignments["image_path"].astype(str).to_numpy()


# Storing the Training class indices in fold-assignment row order
training_class_indices = assignments["class_index"].to_numpy(dtype = np.int32)


# Storing the corresponding fixed outer-fold assignments
training_fold_assignments = assignments["fold"].to_numpy(dtype = np.int32)


# Storing the relative paths for later feature-cache verification
training_relative_paths = assignments["relative_path"].astype(str).to_numpy()


# Verifying that all Training arrays contain the expected 5,600 rows
if not(len(training_image_paths) == len(training_class_indices) == len(training_fold_assignments)
    == len(training_relative_paths) == 5600):

  raise RuntimeError("The prepared Training arrays do not all the expected 5,600 images")


print("Training images prepared.", len(training_image_paths) )


Training images prepared. 5600


In [ ]:
# ============================================================
# CREATING THE TRAINING FEATURE EXTRACTION DATASET
# ============================================================

FEATURE_EXTRACTION_BATCH_SIZE = 64

training_feature_dataset = create_efficientnetb0_feature_dataset(
    image_paths=training_image_paths,
    class_indices=training_class_indices,
    batch_size=FEATURE_EXTRACTION_BATCH_SIZE)

print("Training feature dataset created.")

Training feature dataset created.


In [ ]:
# ============================================================
# EXTRACTING EFFICIENTNETB0 FEATURES FROM THE TRAINING PARTITION
# ============================================================

training_feature_extraction_start = time.perf_counter()

# features and the class number
training_features, extracted_training_class_indices = extract_efficientnetb0_features(
    dataset = training_feature_dataset, feature_extractor = efficientnetb0_feature_extractor)


# Time taken for feature extraction
training_feature_extraction_seconds = (time.perf_counter() - training_feature_extraction_start)


if training_features.shape != (5600, FEATURE_DIMENSION):
    raise RuntimeError("Unexpected Training feature matrix shape: "f"{training_features.shape}")


if not np.array_equal(extracted_training_class_indices, training_class_indices):
    raise RuntimeError("Extracted Training class indices do not match the original Training class indices.")


if not np.isfinite(training_features).all():
    raise RuntimeError("Non-finite values were found in the Training feature matrix.")


print("Training images:", training_features.shape[0])
print("Features per image:", training_features.shape[1])
print("Feature matrix shape:", training_features.shape)
print("Class-index alignment verified:", True)
print("Training feature extraction time:", f"{training_feature_extraction_seconds:.2f}s")


Training images: 5600
Features per image: 1280
Feature matrix shape: (5600, 1280)
Class-index alignment verified: True
Training feature extraction time: 27.21s


In [ ]:
# ============================================================
# SAVING THE TRAINING EFFICIENTNETB0 FEATURES
# ============================================================

# Create the persistent hybrid-model_2 results directory
HYBRID_RESULTS_DIR = (
    Path("/content/drive/MyDrive/brain_tumour_colab")
    / "results"
    / "efficientnetb0_fixed_features")

HYBRID_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Create the path for the cached Training features
TRAINING_FEATURES_PATH = (HYBRID_RESULTS_DIR / "training_efficientnetb0_features.npz")


# Save the features together with their labels, folds and relative image paths
np.savez_compressed(
    TRAINING_FEATURES_PATH,
    features=training_features,
    class_indices=training_class_indices,
    fold_assignments=training_fold_assignments,
    relative_paths=training_relative_paths,
)


print("Training feature cache saved:", TRAINING_FEATURES_PATH)
print("Feature matrix shape:", training_features.shape)

Training feature cache saved: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/training_efficientnetb0_features.npz
Feature matrix shape: (5600, 1280)


In [ ]:
# ============================================================
# DEFINING THE SVM HYPERPARAMETER SEARCH
# ============================================================

from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


# Define the SVM pipeline
svm_pipeline = Pipeline(steps=[("scaler", StandardScaler()), ("classifier", SVC())])


# Define the fixed SVM search grid
SVM_PARAMETER_GRID = {
    "classifier__C": [0.01, 0.1, 1, 10, 100],
    "classifier__gamma": ["scale", 1e-5, 1e-4, 1e-3, 1e-2]}


# Define the Training-only inner cross-validation
svm_inner_cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)


# Create the SVM grid search
svm_grid_search = GridSearchCV(estimator=svm_pipeline, param_grid=SVM_PARAMETER_GRID, scoring="f1_macro", cv=svm_inner_cross_validation,
    n_jobs=-1, refit=True, return_train_score=False)


print("SVM hyperparameter combinations:", 25)
print("Inner cross-validation folds:", 3)
print("Selection metric: macro F1")

SVM hyperparameter combinations: 25
Inner cross-validation folds: 3
Selection metric: macro F1


In [ ]:
# ============================================================
# RUN NESTED FIVE-FOLD CROSS-VALIDATION FOR THE SVM
# ============================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score



# Store the outer-fold results
svm_fold_results = []

svm_oof_predictions = np.empty(len(training_class_indices), dtype=np.int32)

# Start timing the compete nested SVM evaluation
svm_nested_cv_start = time.perf_counter()

# Going through each predefined outer validation fold
for validation_fold in range(1, 6):

    print("=" * 70)
    print(f"SVM OUTER FOLD {validation_fold}")
    print("=" * 70)

    # Identifying the outer Training and validation rows
    outer_training_mask = training_fold_assignments != validation_fold
    outer_validation_mask = training_fold_assignments == validation_fold


    # Create the outer Training data
    outer_training_features = training_features[outer_training_mask]
    outer_training_class_indices = training_class_indices[outer_training_mask]


    # Create the outer validation data
    outer_validation_features = training_features[outer_validation_mask]
    outer_validation_class_indices = training_class_indices[outer_validation_mask]


    print("Outer Training images:", len(outer_training_class_indices))
    print("Outer validation images:", len(outer_validation_class_indices))


    # Run the 3-fold Training-only SVM hyperparameter search
    svm_grid_search.fit(outer_training_features, outer_training_class_indices)


    # Predict the held-out outer validation fold
    fold_predictions = svm_grid_search.predict(outer_validation_features)


    # Store the predictions in their original Training-row positions
    svm_oof_predictions[outer_validation_mask] = fold_predictions


    # Calculate outer-fold metrics
    fold_accuracy = accuracy_score(outer_validation_class_indices, fold_predictions)

    fold_macro_precision = precision_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    fold_macro_recall = recall_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    fold_macro_f1 = f1_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)


    # Save the result for this outer fold
    svm_fold_results.append(
        {
            "fold": validation_fold,
            "accuracy": fold_accuracy,
            "macro_precision": fold_macro_precision,
            "macro_recall": fold_macro_recall,
            "macro_f1": fold_macro_f1,
            "best_C": svm_grid_search.best_params_["classifier__C"],
            "best_gamma": svm_grid_search.best_params_["classifier__gamma"],
            "best_inner_macro_f1": (svm_grid_search.best_score_),
        }
    )


    print("Best parameters:", svm_grid_search.best_params_)
    print("Outer-fold macro F1:", round(fold_macro_f1, 6))

svm_nested_cv_seconds = (time.perf_counter() - svm_nested_cv_start)

print("SVM nested cross-validation time:", f"{svm_nested_cv_seconds:.2f}s")
print("\nSVM five-fold nested cross-validation completed.")

SVM OUTER FOLD 1
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 10, 'classifier__gamma': 'scale'}
Outer-fold macro F1: 0.956293
SVM OUTER FOLD 2
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 10, 'classifier__gamma': 'scale'}
Outer-fold macro F1: 0.958219
SVM OUTER FOLD 3
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 100, 'classifier__gamma': 'scale'}
Outer-fold macro F1: 0.969857
SVM OUTER FOLD 4
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 100, 'classifier__gamma': 'scale'}
Outer-fold macro F1: 0.969607
SVM OUTER FOLD 5
Outer Training images: 4480
Outer validation images: 1120
Best parameters: {'classifier__C': 10, 'classifier__gamma': 'scale'}
Outer-fold macro F1: 0.959929
SVM nested cross-validation time: 912.55s

SVM five-fold nested cross-validation completed.


In [ ]:
# ======================================================================
# SUMMARIZE THE SVM NESTED CROSS-VALIDATION RESULTS
# ======================================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, confusion_matrix

# Converting the five outer-fold results into a DataFrame
svm_fold_results_dataframe = pd.DataFrame(svm_fold_results)


# Calculating the mean outer-fold metrics
svm_mean_accuracy = svm_fold_results_dataframe["accuracy"].mean()
svm_mean_macro_precision = svm_fold_results_dataframe["macro_precision"].mean()
svm_mean_macro_recall = svm_fold_results_dataframe["macro_recall"].mean()
svm_mean_macro_f1 = svm_fold_results_dataframe["macro_f1"].mean()


# Calculating sample standard deviations across the five folds
svm_std_accuracy = svm_fold_results_dataframe["accuracy"].std(ddof=1)
svm_std_macro_precision = svm_fold_results_dataframe["macro_precision"].std(ddof=1)
svm_std_macro_recall = svm_fold_results_dataframe["macro_recall"].std(ddof=1)
svm_std_macro_f1 = svm_fold_results_dataframe["macro_f1"].std(ddof=1)


# Calculating pooled out-of-fold metrics
svm_oof_accuracy = accuracy_score(training_class_indices, svm_oof_predictions)
svm_oof_macro_precision = precision_score(training_class_indices, svm_oof_predictions, average="macro", zero_division=0)
svm_oof_macro_recall = recall_score(training_class_indices, svm_oof_predictions, average="macro", zero_division=0)
svm_oof_macro_f1 = f1_score(training_class_indices, svm_oof_predictions, average="macro", zero_division=0)


# Create the pooled OOF confusion matrix
svm_oof_confusion_matrix = confusion_matrix(
    training_class_indices,
    svm_oof_predictions,
    labels=list(range(NUMBER_OF_CLASSES)))



# DISPLAY THE RESULTS

display(svm_fold_results_dataframe)
print("\n--- Five-fold mean ± sample SD ---")
print("Accuracy:", f"{svm_mean_accuracy:.6f} ± {svm_std_accuracy:.6f}")

print("Macro precision:",
    f"{svm_mean_macro_precision:.6f} ± "
    f"{svm_std_macro_precision:.6f}")

print("Macro recall:",
    f"{svm_mean_macro_recall:.6f} ± "
    f"{svm_std_macro_recall:.6f}")

print("Macro F1:",
    f"{svm_mean_macro_f1:.6f} ± "
    f"{svm_std_macro_f1:.6f}")


print("\n--- Pooled OOF metrics ---")
print("Accuracy:", f"{svm_oof_accuracy:.6f}")
print("Macro precision:", f"{svm_oof_macro_precision:.6f}")
print("Macro recall:", f"{svm_oof_macro_recall:.6f}")
print("Macro F1:", f"{svm_oof_macro_f1:.6f}")
print("\nPooled OOF confusion matrix:")
print(svm_oof_confusion_matrix)


,fold,accuracy,macro_precision,macro_recall,macro_f1,best_C,best_gamma,best_inner_macro_f1
0,1,0.956250,0.958084,0.956250,0.956293,10,scale,0.944607
1,2,0.958036,0.958885,0.958036,0.958219,10,scale,0.944726
2,3,0.969643,0.971070,0.969643,0.969857,100,scale,0.947546
3,4,0.969643,0.969636,0.969643,0.969607,100,scale,0.943128
4,5,0.959821,0.960117,0.959821,0.959929,10,scale,0.945904



--- Five-fold mean ± sample SD ---
Accuracy: 0.962679 ± 0.006482
Macro precision: 0.963558 ± 0.006265
Macro recall: 0.962679 ± 0.006482
Macro F1: 0.962781 ± 0.006475

--- Pooled OOF metrics ---
Accuracy: 0.962679
Macro precision: 0.963356
Macro recall: 0.962679
Macro F1: 0.962789

Pooled OOF confusion matrix:
[[1306   90    2    2]
 [  23 1337   24   16]
 [   7   16 1373    4]
 [   3   19    3 1375]]


In [ ]:
# ======================================================================
# SELECT THE FINAL SVM HYPERPARAMETERS USING ALL TRAINING FEATURES
# ======================================================================

# Create the final SVM pipeline
final_svm_grid_search = GridSearchCV(estimator=svm_pipeline, param_grid=SVM_PARAMETER_GRID, scoring="f1_macro",
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED),
    n_jobs=-1, refit=True, return_train_score=False)

# Starting time for the final SVM search
final_svm_search_start = time.perf_counter()


# Running the final training-only hyperparameter search on all 5600 images
final_svm_grid_search.fit(training_features, training_class_indices)

# Stop timing
final_svm_search_seconds = (time.perf_counter() - final_svm_search_start)


# Recording the final refit time on all 5,600 images
final_svm_refit_seconds = final_svm_grid_search.refit_time_

print("Final SVM hyperparameters:", final_svm_grid_search.best_params_)
print("Best training CV macro F1", round(final_svm_grid_search.best_score_, 6))
print("Final SVM hyperparameter search time:", f"{final_svm_search_seconds:.2f}s")
print("Final SVM refit time:", f"{final_svm_refit_seconds:.2f}s")


Final SVM hyperparameters: {'classifier__C': 100, 'classifier__gamma': 'scale'}
Best training CV macro F1 0.95584
Final SVM hyperparameter search time: 252.26s
Final SVM refit time: 8.63s


In [ ]:
# ======================================================================
# SAVE THE FINAL TRAINED SVM
# ======================================================================

import joblib

# The final SVM already refitted on all 5600 images
final_svm_model = final_svm_grid_search.best_estimator_

# The path for the final trained SVM
FINAL_SVM_MODEL_PATH = (HYBRID_RESULTS_DIR / "final_efficientnetb0_svm.joblib")

# Saving the final SVM
joblib.dump(final_svm_model, FINAL_SVM_MODEL_PATH)

print("Final SVM parameters:", final_svm_grid_search.best_params_)
print("Final SVM model saved:", FINAL_SVM_MODEL_PATH)

Final SVM parameters: {'classifier__C': 100, 'classifier__gamma': 'scale'}
Final SVM model saved: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/final_efficientnetb0_svm.joblib


In [ ]:
# ======================================================================
# DEFINE THE RANDOM FOREST SUCCESSIVE-HALVING SEARCH
# ======================================================================

from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingRandomSearchCV, StratifiedKFold

# Original Random Forest search space
RF_PARAMETER_DISTRIBUTIONS = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 20,  40,  60,  80],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", 0.02, 0.05, 0.1, 0.2],
    "bootstrap": [True, False]
}

# Succesive halving configuration
HALVING_INITIAL_CANDIDATES = 100
HALVING_FACTOR = 3
HALVING_MINIMUM_TREES = 25
HALVING_MAXIMUM_TREES = 675

# Random Forest classifier
random_forest_classifier = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1)


# Inner cross-validation
rf_inner_cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)


# Create the RF grid search
rf_halving_search = HalvingRandomSearchCV(
    estimator=random_forest_classifier,
    param_distributions=RF_PARAMETER_DISTRIBUTIONS,
    n_candidates=HALVING_INITIAL_CANDIDATES,
    factor=HALVING_FACTOR,
    resource="n_estimators",
    min_resources=HALVING_MINIMUM_TREES,
    max_resources=HALVING_MAXIMUM_TREES,
    scoring="f1_macro",
    cv=rf_inner_cross_validation,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)


print("Initial RF candidates:", HALVING_INITIAL_CANDIDATES)
print("Halving factor:", HALVING_FACTOR)
print("Tree resources: 25 -> 75 -> 225 -> 675")
print("Inner CV folds:", 3)
print("Selection metric: macro F1")
print("Feature standardisation:", False)



Initial RF candidates: 100
Halving factor: 3
Tree resources: 25 -> 75 -> 225 -> 675
Inner CV folds: 3
Selection metric: macro F1
Feature standardisation: False


In [ ]:
# ======================================================================
# RUN NESTED FIVE-FOLD CROSS-VALIDATION FOR THE RANDOM FOREST
# ======================================================================

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import time
from pathlib import Path

# Create a directory for saving each completed RF fold
RF_CV_RESULTS_DIR = (HYBRID_RESULTS_DIR / "random_forest" / "cross_validation")

RF_CV_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# Store the outer-fold results
rf_fold_results = []

rf_oof_predictions = np.empty(len(training_class_indices), dtype=np.int32)

# Start time for the RF evaluation
rf_nested_cv_start = time.perf_counter()

# Going through each predefined outer validation fold
for validation_fold in range(1, 6):

    print("=" * 70)
    print(f"RF OUTER FOLD {validation_fold}")
    print("=" * 70)

    # Start timing the current outer fold
    outer_fold_start = time.perf_counter()


    # Ientifying the outer training and validation rows
    outer_training_mask = training_fold_assignments != validation_fold
    outer_validation_mask = training_fold_assignments == validation_fold


    # Create the outer Training data
    outer_training_features = training_features[outer_training_mask]
    outer_training_class_indices = training_class_indices[outer_training_mask]


    # Create the outer validation data
    outer_validation_features = training_features[outer_validation_mask]
    outer_validation_class_indices = training_class_indices[outer_validation_mask]

    print("Outer Training images:", len(outer_training_class_indices))
    print("Outer validation images:", len(outer_validation_class_indices))

    # Running a 3-fold training only succesive halving search
    rf_halving_search.fit(outer_training_features, outer_training_class_indices)

    # Predict the untouched outer validation fold
    fold_predictions = rf_halving_search.predict(outer_validation_features)

    # Store the predictions in their original Training-row positions
    rf_oof_predictions[outer_validation_mask] = fold_predictions


    # Calculate outer-fold metrics
    fold_accuracy = accuracy_score(outer_validation_class_indices, fold_predictions)

    fold_macro_precision = precision_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    fold_macro_recall = recall_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    fold_macro_f1 = f1_score(outer_validation_class_indices, fold_predictions, average="macro", zero_division=0)

    # Record the selected hyperparameters
    best_parameters = rf_halving_search.best_params_


    # Calculate the time taken for this outer fold
    outer_fold_seconds = (time.perf_counter() - outer_fold_start)
    print("Outer fold time:", f"{outer_fold_seconds:.2f}s")


    # Storing the selected RF configuration
    fold_results = {
            "fold": validation_fold,
            "accuracy": fold_accuracy,
            "macro_precision": fold_macro_precision,
            "macro_recall": fold_macro_recall,
            "macro_f1": fold_macro_f1,
            "best_n_estimators": (rf_halving_search.best_estimator_.n_estimators),
            "best_criterion": best_parameters["criterion"],
            "best_max_depth": best_parameters["max_depth"],
            "best_min_samples_split": (best_parameters["min_samples_split"]),
            "best_min_samples_leaf": (best_parameters["min_samples_leaf"]),
            "best_max_features": (best_parameters["max_features"]),
            "best_bootstrap": (best_parameters["bootstrap"]),
            "best_inner_macro_f1": (rf_halving_search.best_score_),
            "outer_fold_time": outer_fold_seconds,
            }

    rf_fold_results.append(fold_results)


    # ================================================================
    # SAVE THIS COMPLETED FOLD IMMEDIATELY
    # ================================================================

    fold_result_path = (RF_CV_RESULTS_DIR / f"fold_{validation_fold}_results.csv")
    fold_predictions_path = (RF_CV_RESULTS_DIR / f"fold_{validation_fold}_predictions.npz")


    # Save the fold metrics and selected hyperparameters
    pd.DataFrame([fold_results]).to_csv(fold_result_path, index=False)


    # Save the true labels and predictions for this outer fold
    np.savez_compressed(
        fold_predictions_path,
        true_class_indices=outer_validation_class_indices,
        predicted_class_indices=fold_predictions,
        relative_paths=training_relative_paths[outer_validation_mask])


    print("Best parameters:", rf_halving_search.best_params_)
    print("Selected trees:", rf_halving_search.best_estimator_.n_estimators)
    print("Outer-fold macro F1:", round(fold_macro_f1, 6))
    print("Fold time:", f"{outer_fold_seconds:.2f}s")
    print("Fold saved:", fold_result_path)


# Stop timing
rf_nested_cv_seconds = (time.perf_counter() - rf_nested_cv_start)

print("RF nested cross-validation time:", f"{rf_nested_cv_seconds:.2f}s")
print("\nRF five-fold nested cross-validation completed.")



RF OUTER FOLD 1
Outer Training images: 4480
Outer validation images: 1120
Outer fold time: 3620.71s
Best parameters: {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.1, 'max_depth': 60, 'criterion': 'gini', 'bootstrap': False, 'n_estimators': 675}
Selected trees: 675
Outer-fold macro F1: 0.913648
Fold time: 3620.71s
Fold saved: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/random_forest/cross_validation/fold_1_results.csv
RF OUTER FOLD 2
Outer Training images: 4480
Outer validation images: 1120
Outer fold time: 2900.16s
Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'sqrt', 'max_depth': 20, 'criterion': 'entropy', 'bootstrap': False, 'n_estimators': 675}
Selected trees: 675
Outer-fold macro F1: 0.911581
Fold time: 2900.16s
Fold saved: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/random_forest/cross_validation/fold_2_results.csv
RF OUTER FOLD 3
Outer Training images: 4

In [ ]:
RF_CV_RESULTS_DIR = (
    HYBRID_RESULTS_DIR
    / "random_forest"
    / "cross_validation"
)

print("RF CV results directory:", RF_CV_RESULTS_DIR)

RF CV results directory: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/random_forest/cross_validation


In [ ]:
# ================================================================
# RECONSTRUCT AND SUMMARIZE ALL FIVE SAVED RF OUTER FOLDS
# ================================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Store the five saved fold-result tables
saved_rf_fold_results = []

# Reconstruct pooled OOF predictions
reconstructed_rf_oof_predictions = np.empty(len(training_class_indices), dtype=np.int32)

# Track which Training rows receive on OOF prediction
oof_prediction_assigned = np.zeros(len(training_class_indices), dtype=bool)


# ================================================================
# LOAD ALL FIVE SAVED RF OUTER FOLDS
# ================================================================

for validation_fold in range(1,6):

    fold_result_path = RF_CV_RESULTS_DIR / f"fold_{validation_fold}_results.csv"

    fold_predictions_path = (RF_CV_RESULTS_DIR / f"fold_{validation_fold}_predictions.npz")


    # Load fold metrics and selected hyperparameters
    fold_result_dataframe = pd.read_csv(fold_result_path)

    saved_rf_fold_results.append(fold_result_dataframe)


    # Load saved validation labels and predictions
    with np.load(fold_predictions_path, allow_pickle=False) as fold_prediction_data:

        fold_true_class_indices = (fold_prediction_data["true_class_indices"])

        fold_predicted_class_indices = (fold_prediction_data["predicted_class_indices"])


    # Identify the original rows belonging to this validation fold
    outer_validation_mask = (training_fold_assignments == validation_fold)


    # Verify saved labels against the fixed fold assignments
    expected_true_class_indices = training_class_indices[outer_validation_mask]


    if not np.array_equal(fold_true_class_indices, expected_true_class_indices):
        raise RuntimeError(f"Saved RF labels mismatch in fold {validation_fold}.")


    # Restore predictions to their original Training-row positions
    reconstructed_rf_oof_predictions[outer_validation_mask] = fold_predicted_class_indices


    # Mark these rows as reconstructed
    oof_prediction_assigned[outer_validation_mask] = True


    print(f"Loaded RF fold {validation_fold}:", len(fold_predicted_class_indices), "predictions")



# ======================================================================
# COMBINE THE FIVE SAVED RESULT TABLES
# ======================================================================

rf_fold_results_dataframe = pd.concat(saved_rf_fold_results, ignore_index=True).sort_values(
    "fold").reset_index(drop=True)


if len(rf_fold_results_dataframe) != 5:
    raise RuntimeError("Expected exactly five saved RF folds.")


if not oof_prediction_assigned.all():
    raise RuntimeError("Not all 5,600 Training images received an RF OOF prediction.")


# ======================================================================
# FIVE-FOLD MEAN ± SAMPLE STANDARD DEVIATION
# ======================================================================

rf_mean_accuracy = (rf_fold_results_dataframe["accuracy"].mean())

rf_std_accuracy = (rf_fold_results_dataframe["accuracy"].std(ddof=1))


rf_mean_macro_precision = (rf_fold_results_dataframe["macro_precision"].mean())

rf_std_macro_precision = (rf_fold_results_dataframe["macro_precision"].std(ddof=1))


rf_mean_macro_recall = (rf_fold_results_dataframe["macro_recall"].mean())

rf_std_macro_recall = (rf_fold_results_dataframe["macro_recall"].std(ddof=1))


rf_mean_macro_f1 = (rf_fold_results_dataframe["macro_f1"].mean())

rf_std_macro_f1 = (rf_fold_results_dataframe["macro_f1"].std(ddof=1))


# ======================================================================
# POOLED OOF METRICS
# ======================================================================

rf_oof_accuracy = accuracy_score(training_class_indices, reconstructed_rf_oof_predictions)

rf_oof_macro_precision = precision_score(training_class_indices, reconstructed_rf_oof_predictions, average="macro", zero_division=0)

rf_oof_macro_recall = recall_score(training_class_indices, reconstructed_rf_oof_predictions, average="macro", zero_division=0)

rf_oof_macro_f1 = f1_score(training_class_indices, reconstructed_rf_oof_predictions, average="macro", zero_division=0)


rf_oof_confusion_matrix = confusion_matrix(training_class_indices, reconstructed_rf_oof_predictions, labels=list(range(NUMBER_OF_CLASSES)))


# Total time across the five saved outer folds
rf_total_outer_fold_seconds = rf_fold_results_dataframe["outer_fold_time"].sum()



# ======================================================================
# DISPLAY RESULTS
# ======================================================================

display(rf_fold_results_dataframe)
print("\n--- Five-fold mean ± sample SD ---")
print("Accuracy:", f"{rf_mean_accuracy:.6f} ± {rf_std_accuracy:.6f}")
print("Macro precision:", f"{rf_mean_macro_precision:.6f} ± {rf_std_macro_precision:.6f}")
print("Macro recall:", f"{rf_mean_macro_recall:.6f} ± {rf_std_macro_recall:.6f}")
print("Macro F1:", f"{rf_mean_macro_f1:.6f} ± {rf_std_macro_f1:.6f}")
print("\n--- Pooled OOF metrics ---")
print("Accuracy:", f"{rf_oof_accuracy:.6f}")
print("Macro precision:", f"{rf_oof_macro_precision:.6f}")
print("Macro recall:", f"{rf_oof_macro_recall:.6f}")
print("Macro F1:", f"{rf_oof_macro_f1:.6f}")
print("\nPooled OOF confusion matrix:")
print(rf_oof_confusion_matrix)
print("\nTotal time across all five outer folds:", f"{rf_total_outer_fold_seconds:.2f}s")
print("All 5,600 OOF predictions reconstructed:", oof_prediction_assigned.all())



Loaded RF fold 1: 1120 predictions
Loaded RF fold 2: 1120 predictions
Loaded RF fold 3: 1120 predictions
Loaded RF fold 4: 1120 predictions
Loaded RF fold 5: 1120 predictions


,fold,accuracy,macro_precision,macro_recall,macro_f1,best_n_estimators,best_criterion,best_max_depth,best_min_samples_split,best_min_samples_leaf,best_max_features,best_bootstrap,best_inner_macro_f1,outer_fold_time
0,1,0.913393,0.917899,0.913393,0.913648,675,gini,60.0,5,1,0.1,False,0.901313,3620.706876
1,2,0.911607,0.913332,0.911607,0.911581,675,entropy,20.0,2,1,sqrt,False,0.900538,2900.163705
2,3,0.916071,0.919546,0.916071,0.916617,675,gini,NaN,2,1,0.2,False,0.902175,4887.885460
3,4,0.923214,0.924494,0.923214,0.923121,675,entropy,60.0,2,2,0.1,False,0.900624,3482.179079
4,5,0.917857,0.918487,0.917857,0.917837,675,gini,80.0,5,1,0.05,False,0.899105,2874.082900



--- Five-fold mean ± sample SD ---
Accuracy: 0.916429 ± 0.004491
Macro precision: 0.918752 ± 0.003993
Macro recall: 0.916429 ± 0.004491
Macro F1: 0.916561 ± 0.004414

--- Pooled OOF metrics ---
Accuracy: 0.916429
Macro precision: 0.918491
Macro recall: 0.916429
Macro F1: 0.916592

Pooled OOF confusion matrix:
[[1206  173    8   13]
 [  59 1235   31   75]
 [   6   49 1323   22]
 [   6   25    1 1368]]

Total time across all five outer folds: 17765.02s
All 5,600 OOF predictions reconstructed: True


In [ ]:
from typing import final
# ================================================================
# SELECT THE FINAL RANDOM FOREST HYPERPARAMETERS USING ALL TRAINING FEATURES
# ================================================================


# Create a fresh Random Forest classifier
fInal_random_forest_classifier = RandomForestClassifier(random_state=RANDOM_SEED, n_jobs=1)


# Create the final 3-fold Training only cross-validation
final_rf_inner_cross_validation = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_SEED)


# Create a fresh succesive-halving search
final_rf_halving_search = HalvingRandomSearchCV(
    estimator=fInal_random_forest_classifier,
    param_distributions=RF_PARAMETER_DISTRIBUTIONS,
    n_candidates=HALVING_INITIAL_CANDIDATES,
    factor=HALVING_FACTOR,
    resource="n_estimators",
    min_resources=HALVING_MINIMUM_TREES,
    max_resources=HALVING_MAXIMUM_TREES,
    scoring="f1_macro",
    cv=final_rf_inner_cross_validation,
    random_state=RANDOM_SEED,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
)

# Start time for the final RF search
final_rf_search_time = time.perf_counter()

# Run the final trainining only seaarch on all 5,600 feature vectors
final_rf_halving_search.fit(training_features, training_class_indices)

# Stop timing
final_rf_search_seconds = time.perf_counter() - final_rf_search_time

# Record the final refit time
final_rf_refit_time = final_rf_halving_search.refit_time_


print("Final RF hyperparameters:", final_rf_halving_search.best_params_)
print("Selected trees:", final_rf_halving_search.best_estimator_.n_estimators)
print("Best Training CV macro F1:", round(final_rf_halving_search.best_score_, 6))
print("Final RF hyperparameter search time:", f"{final_rf_search_seconds:.2f}s")
print("Final RF refit time:", f"{final_rf_refit_time:.2f}s")




Final RF hyperparameters: {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.1, 'max_depth': 40, 'criterion': 'entropy', 'bootstrap': False, 'n_estimators': 675}
Selected trees: 675
Best Training CV macro F1: 0.910706
Final RF hyperparameter search time: 4530.81s
Final RF refit time: 1104.99s


In [ ]:
# ======================================================================
# SAVE THE FINAL TRAINED RANDOM FOREST
# ======================================================================

import joblib


# The final RF is already refitted on all 5,600 Training features
final_rf_model = final_rf_halving_search.best_estimator_


# Path for the final trained EfficientNetB0 + RF model
FINAL_RF_MODEL_PATH = HYBRID_RESULTS_DIR / "final_efficientnetb0_random_forest.joblib"


# Save the final RF model
joblib.dump(final_rf_model, FINAL_RF_MODEL_PATH)


print("Final RF parameters:", final_rf_halving_search.best_params_)
print("Final RF model saved:", FINAL_RF_MODEL_PATH)

Final RF parameters: {'min_samples_split': 5, 'min_samples_leaf': 1, 'max_features': 0.1, 'max_depth': 40, 'criterion': 'entropy', 'bootstrap': False, 'n_estimators': 675}
Final RF model saved: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/final_efficientnetb0_random_forest.joblib


In [ ]:
# ======================================================================
# PREPARING THE HELD-OUT TESTING PARTITION
# ======================================================================

# Testing directory
TESTING_DIR = DATA_DIR / "Testing"

# Collect Testing image paths in the fixed class order
testing_image_paths = []
testing_class_indices = []
testing_relative_paths = []

for class_index, class_name in enumerate(CLASS_NAMES):
    class_directory = TESTING_DIR / class_name

    class_image_paths = sorted(class_directory.glob("*.png"))

    print(f"{class_name} Testing images:", len(class_image_paths))

    for image_path in class_image_paths:
        testing_image_paths.append(image_path)
        testing_class_indices.append(class_index)
        testing_relative_paths.append(str(image_path.relative_to(DATA_DIR)))


# Converting to NumPy arrays
testing_image_paths = np.array(testing_image_paths, dtype = str)
testing_class_indices = np.array(testing_class_indices, dtype = np.int32)
testing_relative_paths = np.array(testing_relative_paths, dtype = str)


# ======================================================================
# VERIFY TESTING PARTITION
# ======================================================================

if len(testing_image_paths) != 1598:
  raise RuntimeError("Wrong number of testing images.")

expected_testing_class_counts = {
    0: 400, # glioma
    1: 400, # meningioma
    2: 398, # notumor
    3: 400, # pituitary

}

for class_index, expected_count in expected_testing_class_counts.items():
    actual_count = np.sum(testing_class_indices == class_index)

    if actual_count != expected_count:
        raise RuntimeError(f"Unexpected Testing count for class {class_index}.")


print("\nTesting images:", len(testing_image_paths))
print("Testing partition verification passed")


glioma Testing images: 400
meningioma Testing images: 400
notumor Testing images: 398
pituitary Testing images: 400

Testing images: 1598
Testing partition verification passed


In [ ]:
# ======================================================================
# CREATE THE EFFICIENTNETB0 FEATURE DATASET FOR THE HELD-OUT TESTING PARTITION
# ======================================================================

FEATURE_EXTRACTION_BATCH_SIZE = 64

testing_feature_dataset = create_efficientnetb0_feature_dataset(
    image_paths = testing_image_paths,
    class_indices = testing_class_indices,
    batch_size = FEATURE_EXTRACTION_BATCH_SIZE)

print("Testing images:", len(testing_image_paths))
print("Feature extraction batch size", FEATURE_EXTRACTION_BATCH_SIZE)
print("Testing feature dataset:", testing_feature_dataset)
print("Testing feature dataset verified")

Testing images: 1598
Feature extraction batch size 64
Testing feature dataset: <_PrefetchDataset element_spec=(TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int32, name=None))>
Testing feature dataset verified


In [ ]:
# ======================================================================
# EXTRACT EFFICIENTNETB0 FEATURES FROM THE HELD-OUT TESTING PARTITION
# ======================================================================

# Start timing the feature extraction
testing_feature_extraction_start = time.perf_counter()

# Extract fixed EfficientNetB0 features
testing_features, extracted_testing_class_indices = extract_efficientnetb0_features(
    dataset = testing_feature_dataset,
    feature_extractor = efficientnetb0_feature_extractor)

# Stop the timing
testing_feature_extraction_seconds = time.perf_counter() - testing_feature_extraction_start


# ======================================================================
# VERIFY THE EXTRACTED TESTING FEATURES
# ======================================================================

if testing_features.shape != (len(testing_class_indices), FEATURE_DIMENSION):
    raise RuntimeError("Unexpected testing feature matrix")

if not np.array_equal(extracted_testing_class_indices, testing_class_indices):
    raise RuntimeError("Unexpected testing class indices")

if not np.isfinite(testing_features).all():
    raise RuntimeError("Non-finite testing features")

print("Testing images:", testing_features.shape[0])
print("Features per image:", testing_features.shape[1])
print("Testing feature matrix shape:", testing_features.shape)
print("Testing feature extraction time:", f"{testing_feature_extraction_seconds:.2f}s")
print("Class-index alignment verified:", True)

Testing images: 1598
Features per image: 1280
Testing feature matrix shape: (1598, 1280)
Testing feature extraction time: 8.83s
Class-index alignment verified: True


In [ ]:
# ======================================================================
# FINAL HELD-OUT TESTING EVALUATION: EFFICIENTNETB0 + SVM
# ======================================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
import joblib

from pathlib import Path

HYBRID_RESULTS_DIR = (
    Path("/content/drive/MyDrive/brain_tumour_colab")
    / "results"
    / "efficientnetb0_fixed_features"
)

final_svm_model = joblib.load(HYBRID_RESULTS_DIR / "final_efficientnetb0_svm.joblib")

# Start timing classifier prediction only
svm_testing_prediction_start = time.perf_counter()

# Predict the held-out Testing partition
svm_testing_predictions = final_svm_model.predict(testing_features)

# Stop the timing
svm_testing_prediction_seconds = time.perf_counter() - svm_testing_prediction_start

# End to end testing time: EfficientNetB0 feature extraction + SVM prediction
svm_testing_end_to_end_seconds = testing_feature_extraction_seconds + svm_testing_prediction_seconds


# ======================================================================
# CALCULATE FINAL TESTING METRICS
# ======================================================================

svm_testing_accuracy = accuracy_score(testing_class_indices, svm_testing_predictions)

svm_testing_macro_precision = precision_score(testing_class_indices, svm_testing_predictions, average="macro", zero_division=0)

svm_testing_macro_recall = recall_score(testing_class_indices, svm_testing_predictions, average="macro", zero_division=0)

svm_testing_macro_f1 = f1_score(testing_class_indices, svm_testing_predictions, average="macro", zero_division=0)

svm_testing_confusion_matrix = confusion_matrix(testing_class_indices, svm_testing_predictions, labels=list(range(NUMBER_OF_CLASSES)))

svm_testing_classification_report = classification_report(testing_class_indices, svm_testing_predictions, labels = list(range(NUMBER_OF_CLASSES)), target_names=CLASS_NAMES, zero_division=0, digits = 6)


# ======================================================================
# DISPLAY THE FINAL TESTING RESULTS
# ======================================================================

print("--- EfficientNetB0 + SVM: Held-out Testing results ---")
print("Accuracy:", f"{svm_testing_accuracy:.6f}")
print("Macro precision:", f"{svm_testing_macro_precision:.6f}")
print("Macro recall:", f"{svm_testing_macro_recall:.6f}")
print("Macro F1:", f"{svm_testing_macro_f1:.6f}")
print("\nClassification report:")
print(svm_testing_classification_report)
print("Confusion matrix:")
print(svm_testing_confusion_matrix)
print("\nSVM prediction time:", f"{svm_testing_prediction_seconds:.4f}s")
print("EfficientNetB0 Testing feature extraction time:", f"{testing_feature_extraction_seconds:.2f}s")
print("End-to-end Testing time:", f"{svm_testing_end_to_end_seconds:.2f}s")


--- EfficientNetB0 + SVM: Held-out Testing results ---
Accuracy: 0.941802
Macro precision: 0.946322
Macro recall: 0.941875
Macro F1: 0.940539

Classification report:
              precision    recall  f1-score   support

      glioma   0.975385  0.792500  0.874483       400
  meningioma   0.858388  0.985000  0.917346       400
     notumor   0.959036  1.000000  0.979090       398
   pituitary   0.992481  0.990000  0.991239       400

    accuracy                       0.941802      1598
   macro avg   0.946322  0.941875  0.940539      1598
weighted avg   0.946307  0.941802  0.940491      1598

Confusion matrix:
[[317  65  16   2]
 [  4 394   1   1]
 [  0   0 398   0]
 [  4   0   0 396]]

SVM prediction time: 4.8144s
EfficientNetB0 Testing feature extraction time: 8.83s
End-to-end Testing time: 13.65s


In [ ]:
# ======================================================================
# FINAL HELD-OUT TESTING EVALUATION: EFFICIENTNETB0 + RANDOM FOREST
# ======================================================================

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

import joblib

final_rf_model = joblib.load(
    HYBRID_RESULTS_DIR / "final_efficientnetb0_random_forest.joblib"
)

# Start timing classifier prediction only
rf_testing_prediction_start = time.perf_counter()

# Predict the held-out Testing partition
rf_testing_predictions = final_rf_model.predict(testing_features)

# Stop the timing
rf_testing_prediction_seconds = time.perf_counter() - rf_testing_prediction_start

# End to end testing time: EfficientNetB0 feature extraction + RF prediction
rf_testing_end_to_end_seconds = testing_feature_extraction_seconds + rf_testing_prediction_seconds


# ======================================================================
# CALCULATE FINAL TESTING METRICS
# ======================================================================

rf_testing_accuracy = accuracy_score(testing_class_indices, rf_testing_predictions)

rf_testing_macro_precision = precision_score(testing_class_indices, rf_testing_predictions, average="macro", zero_division=0)

rf_testing_macro_recall = recall_score(testing_class_indices, rf_testing_predictions, average="macro", zero_division=0)

rf_testing_macro_f1 = f1_score(testing_class_indices, rf_testing_predictions, average="macro", zero_division=0)

rf_testing_confusion_matrix = confusion_matrix(testing_class_indices, rf_testing_predictions, labels=list(range(NUMBER_OF_CLASSES)))

rf_testing_classification_report = classification_report(testing_class_indices, rf_testing_predictions, labels = list(range(NUMBER_OF_CLASSES)), target_names=CLASS_NAMES, zero_division=0, digits = 6)


# ======================================================================
# DISPLAY THE FINAL TESTING RESULTS
# ======================================================================

print("--- EfficientNetB0 + Random Forest: Held-out Testing results ---")
print("Accuracy:", f"{rf_testing_accuracy:.6f}")
print("Macro precision:", f"{rf_testing_macro_precision:.6f}")
print("Macro recall:", f"{rf_testing_macro_recall:.6f}")
print("Macro F1:", f"{rf_testing_macro_f1:.6f}")
print("\nClassification report:")
print(rf_testing_classification_report)
print("Confusion matrix:")
print(rf_testing_confusion_matrix)
print("\nRF prediction time:", f"{rf_testing_prediction_seconds:.4f}s")
print("EfficientNetB0 Testing feature extraction time:", f"{testing_feature_extraction_seconds:.2f}s")
print("End-to-end Testing time:", f"{rf_testing_end_to_end_seconds:.2f}s")


--- EfficientNetB0 + Random Forest: Held-out Testing results ---
Accuracy: 0.911139
Macro precision: 0.915832
Macro recall: 0.911250
Macro F1: 0.909481

Classification report:
              precision    recall  f1-score   support

      glioma   0.940063  0.745000  0.831241       400
  meningioma   0.812636  0.932500  0.868452       400
     notumor   0.943128  1.000000  0.970732       398
   pituitary   0.967500  0.967500  0.967500       400

    accuracy                       0.911139      1598
   macro avg   0.915832  0.911250  0.909481      1598
weighted avg   0.915798  0.911139  0.909405      1598

Confusion matrix:
[[298  77  20   5]
 [ 16 373   3   8]
 [  0   0 398   0]
 [  3   9   1 387]]

RF prediction time: 0.2310s
EfficientNetB0 Testing feature extraction time: 8.83s
End-to-end Testing time: 9.07s


In [ ]:
# ======================================================================
# SAVE THE FINAL EFFICIENTNETB0 HYBRID TESTING RESULTS
# ======================================================================

EFFICIENTNETB0_TESTING_RESULTS_DIR = HYBRID_RESULTS_DIR / "testing"


EFFICIENTNETB0_TESTING_RESULTS_DIR.mkdir(parents=True, exist_ok=True)


# ======================================================================
# SAVE SUMMARY METRICS
# ======================================================================

efficientnetb0_testing_summary = pd.DataFrame(
    [
        {
            "classifier": "SVM",
            "accuracy": svm_testing_accuracy,
            "macro_precision": svm_testing_macro_precision,
            "macro_recall": svm_testing_macro_recall,
            "macro_f1": svm_testing_macro_f1,
            "prediction_time_seconds": svm_testing_prediction_seconds,
            "feature_extraction_time_seconds": testing_feature_extraction_seconds,
            "end_to_end_time_seconds": svm_testing_end_to_end_seconds,
        },
        {
            "classifier": "Random Forest",
            "accuracy": rf_testing_accuracy,
            "macro_precision": rf_testing_macro_precision,
            "macro_recall": rf_testing_macro_recall,
            "macro_f1": rf_testing_macro_f1,
            "prediction_time_seconds": rf_testing_prediction_seconds,
            "feature_extraction_time_seconds": testing_feature_extraction_seconds,
            "end_to_end_time_seconds": rf_testing_end_to_end_seconds,
        },
    ]
)


TESTING_SUMMARY_PATH = EFFICIENTNETB0_TESTING_RESULTS_DIR / "efficientnetb0_hybrid_testing_summary.csv"

efficientnetb0_testing_summary.to_csv(TESTING_SUMMARY_PATH, index=False)


# ======================================================================
# SAVE TESTING PREDICTIONS
# ======================================================================

TESTING_PREDICTIONS_PATH = EFFICIENTNETB0_TESTING_RESULTS_DIR / "efficientnetb0_hybrid_testing_predictions.npz"

np.savez_compressed(
    TESTING_PREDICTIONS_PATH,
    true_class_indices=testing_class_indices,
    svm_predictions=svm_testing_predictions,
    rf_predictions=rf_testing_predictions,
    relative_paths=testing_relative_paths,
)


# ======================================================================
# SAVE CONFUSION MATRICES
# ======================================================================

np.save(EFFICIENTNETB0_TESTING_RESULTS_DIR / "svm_testing_confusion_matrix.npy", svm_testing_confusion_matrix)

np.save(EFFICIENTNETB0_TESTING_RESULTS_DIR / "rf_testing_confusion_matrix.npy", rf_testing_confusion_matrix)


print("Testing summary saved:", TESTING_SUMMARY_PATH)
print("Testing predictions saved:", TESTING_PREDICTIONS_PATH)
print("EfficientNetB0 hybrid Testing results saved successfully.")

Testing summary saved: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/testing/efficientnetb0_hybrid_testing_summary.csv
Testing predictions saved: /content/drive/MyDrive/brain_tumour_colab/results/efficientnetb0_fixed_features/testing/efficientnetb0_hybrid_testing_predictions.npz
EfficientNetB0 hybrid Testing results saved successfully.
